# 0.Config

ส่วนนี้ตรวจสอบเวอร์ชัน Python และแพ็กเกจที่เกี่ยวข้อง 
- เตรียม API key
- กำหนด model ID ที่จะใช้ตลอดการทดลอง เพื่อให้การรัน benchmark ใช้สภาพแวดล้อมที่ระบุได้ชัดเจน


In [1]:
import sys
import os
import time
import json
from getpass import getpass
from pathlib import Path
from importlib.metadata import version

from dotenv import load_dotenv
from deepagents import create_deep_agent
from langchain_openai import ChatOpenAI
from langchain_core.messages import AIMessage

In [2]:
print("Python:", sys.version.split()[0])
print("Kernel executable:", sys.executable)

assert sys.version_info >= (3, 11), (
    "กรุณาใช้ Python 3.11 ขึ้นไปสำหรับ Notebook นี้"
)

Python: 3.12.15
Kernel executable: /usr/local/bin/python3.12


In [3]:
PACKAGE_VERSIONS = {
    package: version(package)
    for package in [
        "deepagents",
        "langchain",
        "langgraph",
        "langchain-openai",
        "langchain-core",
        "openai",
    ]
}

print("Working directory:", Path.cwd())
print(json.dumps(PACKAGE_VERSIONS, indent=2))

Working directory: /workspace/deep_agent
{
  "deepagents": "0.7.21",
  "langchain": "1.4.3",
  "langgraph": "1.2.12",
  "langchain-openai": "1.6.7",
  "langchain-core": "1.6.6",
  "openai": "3.24.0"
}


In [4]:
load_dotenv(Path.cwd() / ".env", override=False)

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass(
        "Enter OPENAI_API_KEY: "
    ).strip()

assert os.getenv("OPENAI_API_KEY"), "ยังไม่มี OPENAI_API_KEY"

MODEL = os.getenv("BENCHMARK_MODEL", "").strip()

if not MODEL:
    MODEL = input(
        "ใส่ OpenAI model ID ที่บัญชีคุณใช้งานได้: "
    ).strip()

assert MODEL, "กรุณาระบุ model ID"

# เก็บไว้สำหรับรัน cell ซ้ำใน kernel นี้
os.environ["BENCHMARK_MODEL"] = MODEL

print("API key loaded:", bool(os.getenv("OPENAI_API_KEY")))
print("Model:", MODEL)

API key loaded: True
Model: gpt-6-luna


In [5]:
OPEN_AI_API_KEY = os.getenv("OPENAI_API_KEY")[-5:-1]
print(f"OpenAI_API_KEY : {OPEN_AI_API_KEY}")
print(f"OpenAI_Model   : {MODEL}")

OpenAI_API_KEY : FGLc
OpenAI_Model   : gpt-6-luna


<hr>

# 1.Create Deep Agent

สร้างโมเดลและ Deep Agent สำหรับทดสอบการเชื่อมต่อเบื้องต้น โดยกำหนดให้ตอบข้อความตามรูปแบบที่ระบุและไม่ใช้เครื่องมือ จากนั้นตรวจคำตอบ เวลาใช้งาน และข้อมูล token ที่โมเดลรายงาน

### ตรวจผล Smoke Test

เซลล์ในหัวข้อนี้เรียก agent ตรวจว่าคำตอบตรงตามที่กำหนด ไม่มีการเรียก tool และจบด้วยข้อความจาก agent ตามที่คาดไว้


In [8]:
# Set model objective
model = ChatOpenAI(
    model=MODEL,
    timeout=60,
    max_retries=0,
    use_responses_api=True,
)

# Create Deep agent
# Create_deep_agent : สร้างและคืนค่า agent obj ในชื่อ smoke_agent
# System_prompt : เป็นตัวที่กำหนดว่า agent ควรจะทำงานยังไง เรียกใช้อะไรไม่เรียกใช้อะไร
smoke_agent = create_deep_agent(
    model=model,
    system_prompt=(
        "This is a connection smoke test. "
        "Do not call tools or delegate work. "
        "Reply with exactly: DEEP_AGENTS_SDK_OK"
    ),
)

print("Deep Agent created.")

Deep Agent created.


In [7]:
smoke_result = None
smoke_latency_s = None

started_at = time.perf_counter()
# ainvoke เป็นคำสั่งแบบ asyncronouse เลยจะต้องมีการใช้งานร่วมกับ await
# ainvoke เป็นคำสั่ง Run agent
try:
    smoke_result = await smoke_agent.ainvoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": "Run the smoke test.",
                }
            ]
        },
        config={"recursion_limit": 10},
    )
finally:
    smoke_latency_s = time.perf_counter() - started_at

print("Run completed.")
print("Latency:", round(smoke_latency_s, 2), "seconds")

Run completed.
Latency: 3.81 seconds


In [15]:
smoke_result

{'messages': [HumanMessage(content='Run the smoke test.', additional_kwargs={}, response_metadata={}, id='1ccb3b49-dee3-455c-ae41-dd6be5ee7e07'),
  AIMessage(content=[{'type': 'text', 'text': 'DEEP_AGENTS_SDK_OK', 'annotations': [], 'id': 'msg_0708a21514d109d9006ac47788894087d0946bbfb50b6b25e9', 'phase': 'final_answer'}], additional_kwargs={}, response_metadata={'id': 'resp_0708a21514d109d9006ac47786d79487d0a554af1a7a92f8b9', 'created_at': 1791260550.0, 'metadata': {}, 'model': 'gpt-6-luna', 'object': 'response', 'service_tier': 'default', 'status': 'completed', 'model_provider': 'openai', 'model_name': 'gpt-6-luna'}, id='resp_0708a21514d109d9006ac47786d79487d0a554af1a7a92f8b9', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 2000, 'output_tokens': 11, 'total_tokens': 2011, 'input_token_details': {'cache_creation': 1997, 'cache_read': 0}, 'output_token_details': {'reasoning': 0}})],
 'files': {}}

In [17]:
assert smoke_result is not None, "กรุณารัน Cell 6 ให้สำเร็จก่อน"

messages = smoke_result["messages"]
last_message = messages[-1]

assert isinstance(last_message, AIMessage), (
    "Run ไม่ได้จบด้วยข้อความจาก agent"
)

smoke_answer = last_message.text.strip()

ai_messages = [
    message
    for message in messages
    if isinstance(message, AIMessage)
]

tool_calls = [
    call
    for message in ai_messages
    for call in message.tool_calls
]

smoke_passed = (
    smoke_answer == "DEEP_AGENTS_SDK_OK"
    and len(tool_calls) == 0
)

print("Response         :", smoke_answer)
print("Smoke test passed:", smoke_passed)
print("Latency          :", round(smoke_latency_s, 2), "seconds")
print("AI messages      :", len(ai_messages))
print("Tool calls       :", len(tool_calls))

# รายงาน usage ของแต่ละข้อความที่มองเห็นในผลลัพธ์
for index, message in enumerate(ai_messages, start=1):
    print(f"\nAI message {index} usage:")
    print(json.dumps(message.usage_metadata, indent=2))

Response         : DEEP_AGENTS_SDK_OK
Smoke test passed: True
Latency          : 3.81 seconds
AI messages      : 1
Tool calls       : 0

AI message 1 usage:
{
  "input_tokens": 2000,
  "output_tokens": 11,
  "total_tokens": 2011,
  "input_token_details": {
    "cache_creation": 1997,
    "cache_read": 0
  },
  "output_token_details": {
    "reasoning": 0
  }
}


<hr>

# 2.Mathematic calculation

ทดสอบโจทย์คำนวณตัวชี้วัดทางการเงินจากข้อมูลที่กำหนด พร้อมเครื่องมือ calculator สำหรับการคำนวณ และตรวจทั้งรูปแบบ JSON ค่าผลลัพธ์ และการใช้เครื่องมือให้ตรงตามข้อกำหนด

### Math01 (Basic) กรณีข้อมูลบริษัทเดียว

โจทย์แรกให้คำนวณการเติบโตของรายได้ อัตรากำไรสุทธิ และหนี้สินต่อสินทรัพย์ แล้วตรวจว่าคำตอบมี key และค่าตัวเลขครบถ้วน

In [22]:
from typing import Literal
from langchain.tools import tool # tool เป็น ฟังก์ชั่น Decorator ที่เอาไว้ให้ agent เห็นว่าเป็น Tool แล้วเรียกใช้งาน
from langchain_core.messages import ToolMessage
import asyncio
import math
from pathlib import Path
from importlib.metadata import version
from datetime import datetime, timezone
from uuid import uuid4
import json


In [23]:
# ประกาศเครื่องมือการทำงาน ทางคณิตศาสตร์
# @tool เป็นตัว Decorator ที่เอาไว้ประกาศเพื่อครอบฟังก์ชั่นเอาไว้ใช้งาน
@tool
def calculator(
    operation: Literal["add", "subtract", "multiply", "divide"],
    a: float,
    b: float,
) -> float:
    """Calculate a + b, a - b, a * b, or a / b.

    Choose the operation and provide the two operands.
    For subtraction and division, operand order matters.
    """
    if operation == "add":
        return a + b
    if operation == "subtract":
        return a - b
    if operation == "multiply":
        return a * b
    if operation == "divide":
        if b == 0:
            raise ValueError("Cannot divide by zero.")
        return a / b

    raise ValueError(f"Unsupported operation: {operation}")


print("Tool ready:", calculator.name)

Tool ready: calculator


In [24]:
MATH_CASE_ID = "math_001"

math_prompt = """
บริษัทสมมติ Alpha มีข้อมูลจากงบรวมดังนี้
ตัวเลขทั้งหมดมีหน่วยเป็นล้านบาท:

- รายได้ปี 2024: 1,000
- รายได้ปี 2025: 1,250
- กำไรสุทธิปี 2025: 150
- หนี้สินรวม ณ สิ้นปี 2025: 600
- สินทรัพย์รวม ณ สิ้นปี 2025: 2,000

จงคำนวณ:
1. การเติบโตของรายได้ปี 2025 เทียบกับปี 2024 เป็นเปอร์เซ็นต์
2. อัตรากำไรสุทธิปี 2025 เป็นเปอร์เซ็นต์
3. หนี้สินรวมต่อสินทรัพย์รวม ณ สิ้นปี 2025 เป็นเปอร์เซ็นต์

ข้อกำหนด:
- ใช้ calculator สำหรับการคำนวณ
- ใช้เฉพาะข้อมูลที่ให้
- ตอบเป็น JSON object เท่านั้น ไม่ใส่ Markdown code fence
- มีสาม keys ตามนี้ และค่าต้องเป็นตัวเลข:
  revenue_growth_pct
  net_margin_pct
  liabilities_to_assets_pct
- ค่าเปอร์เซ็นต์ให้ใช้รูปแบบ เช่น 15 สำหรับ 15%
- ปัดผลลัพธ์สุดท้ายเป็นทศนิยมไม่เกินสองตำแหน่ง
"""

print("Case:", MATH_CASE_ID)
print(math_prompt)

Case: math_001

บริษัทสมมติ Alpha มีข้อมูลจากงบรวมดังนี้
ตัวเลขทั้งหมดมีหน่วยเป็นล้านบาท:

- รายได้ปี 2024: 1,000
- รายได้ปี 2025: 1,250
- กำไรสุทธิปี 2025: 150
- หนี้สินรวม ณ สิ้นปี 2025: 600
- สินทรัพย์รวม ณ สิ้นปี 2025: 2,000

จงคำนวณ:
1. การเติบโตของรายได้ปี 2025 เทียบกับปี 2024 เป็นเปอร์เซ็นต์
2. อัตรากำไรสุทธิปี 2025 เป็นเปอร์เซ็นต์
3. หนี้สินรวมต่อสินทรัพย์รวม ณ สิ้นปี 2025 เป็นเปอร์เซ็นต์

ข้อกำหนด:
- ใช้ calculator สำหรับการคำนวณ
- ใช้เฉพาะข้อมูลที่ให้
- ตอบเป็น JSON object เท่านั้น ไม่ใส่ Markdown code fence
- มีสาม keys ตามนี้ และค่าต้องเป็นตัวเลข:
  revenue_growth_pct
  net_margin_pct
  liabilities_to_assets_pct
- ค่าเปอร์เซ็นต์ให้ใช้รูปแบบ เช่น 15 สำหรับ 15%
- ปัดผลลัพธ์สุดท้ายเป็นทศนิยมไม่เกินสองตำแหน่ง



In [25]:
math_agent = create_deep_agent(
    model=model,
    tools=[calculator],
    system_prompt=(
        "You solve numerical tasks using the provided data. "
        "Choose the appropriate formulas yourself. "
        "Use calculator for arithmetic. "
        "Do not delegate work or use other tools for this task. "
        "Follow the user's output format exactly."
    ),
)

print("Math agent ready.")
print("Model:", model.model_name)

Math agent ready.
Model: gpt-6-luna


In [26]:
math_result = None
math_latency_s = None
math_runtime_status = "running"
math_error = None

started_at = time.perf_counter()

try:
    math_result = await asyncio.wait_for(
        math_agent.ainvoke(
            {
                "messages": [
                    {"role": "user", 
                     "content": math_prompt}
                ]
            },
            config={"recursion_limit": 40}, # maximum ของขั้นตอนคือ 40 ครั้งห้ามเกินกว่านี้แล้ว
        ),
        timeout=120, # ตั้ง Limit เวลาเอาไว้ว่าให้ทำงานได้สูงสุดแค่นี้ ถ้าหากว่า >120 -> timeout -> fail
    )
    math_runtime_status = "completed"

except Exception as exc:
    math_runtime_status = "failed"
    math_error = f"{type(exc).__name__}: {exc}"

finally:
    math_latency_s = time.perf_counter() - started_at

print("Runtime status:", math_runtime_status)
print("Latency:", round(math_latency_s, 2), "seconds")

if math_error:
    print("Error:", math_error)

Runtime status: completed
Latency: 15.92 seconds


In [28]:
math_result["messages"]

[HumanMessage(content='\nบริษัทสมมติ Alpha มีข้อมูลจากงบรวมดังนี้\nตัวเลขทั้งหมดมีหน่วยเป็นล้านบาท:\n\n- รายได้ปี 2024: 1,000\n- รายได้ปี 2025: 1,250\n- กำไรสุทธิปี 2025: 150\n- หนี้สินรวม ณ สิ้นปี 2025: 600\n- สินทรัพย์รวม ณ สิ้นปี 2025: 2,000\n\nจงคำนวณ:\n1. การเติบโตของรายได้ปี 2025 เทียบกับปี 2024 เป็นเปอร์เซ็นต์\n2. อัตรากำไรสุทธิปี 2025 เป็นเปอร์เซ็นต์\n3. หนี้สินรวมต่อสินทรัพย์รวม ณ สิ้นปี 2025 เป็นเปอร์เซ็นต์\n\nข้อกำหนด:\n- ใช้ calculator สำหรับการคำนวณ\n- ใช้เฉพาะข้อมูลที่ให้\n- ตอบเป็น JSON object เท่านั้น ไม่ใส่ Markdown code fence\n- มีสาม keys ตามนี้ และค่าต้องเป็นตัวเลข:\n  revenue_growth_pct\n  net_margin_pct\n  liabilities_to_assets_pct\n- ค่าเปอร์เซ็นต์ให้ใช้รูปแบบ เช่น 15 สำหรับ 15%\n- ปัดผลลัพธ์สุดท้ายเป็นทศนิยมไม่เกินสองตำแหน่ง\n', additional_kwargs={}, response_metadata={}, id='71ac1261-d305-4550-8bf2-6e3bddd8d34b'),
 AIMessage(content=[{'arguments': '{"operation":"subtract","a":1250,"b":1000}', 'call_id': 'call_6eqPzy4wB24Ftv8mBE9fT5lM', 'name': 'calculator', 'ty

In [27]:
assert math_result is not None, (
    "Cell 11 ยังไม่สำเร็จ ให้ตรวจ error ก่อน"
)

math_messages = math_result["messages"]
math_tool_calls = []

for message in math_messages:
    if isinstance(message, AIMessage):
        for call in message.tool_calls:
            math_tool_calls.append(call)
            print("CALL:", call["name"])
            print("ARGS:", json.dumps(call["args"], ensure_ascii=False))

    elif isinstance(message, ToolMessage):
        print("RESULT:", message.content)
        print()

math_last_message = math_messages[-1]

assert isinstance(math_last_message, AIMessage), (
    "Run ไม่ได้จบด้วยข้อความจาก agent"
)

math_answer = math_last_message.text.strip()

print("FINAL ANSWER:")
print(math_answer)
print("\nTool calls:", len(math_tool_calls))

CALL: calculator
ARGS: {"operation": "subtract", "a": 1250, "b": 1000}
CALL: calculator
ARGS: {"operation": "divide", "a": 150, "b": 1250}
CALL: calculator
ARGS: {"operation": "divide", "a": 600, "b": 2000}
RESULT: 250.0

RESULT: 0.12

RESULT: 0.3

CALL: calculator
ARGS: {"operation": "divide", "a": 250, "b": 1000}
RESULT: 0.25

CALL: calculator
ARGS: {"operation": "multiply", "a": 0.25, "b": 100}
RESULT: 25.0

CALL: calculator
ARGS: {"operation": "multiply", "a": 0.12, "b": 100}
RESULT: 12.0

CALL: calculator
ARGS: {"operation": "multiply", "a": 0.3, "b": 100}
RESULT: 30.0

FINAL ANSWER:
{"revenue_growth_pct":25,"net_margin_pct":12,"liabilities_to_assets_pct":30}

Tool calls: 7


In [29]:
expected = {
    "revenue_growth_pct": 25.0,
    "net_margin_pct": 12.0,
    "liabilities_to_assets_pct": 30.0,
}

try:
    parsed_answer = json.loads(math_answer)
except json.JSONDecodeError:
    parsed_answer = None

format_passed = (
    isinstance(parsed_answer, dict)
    and set(parsed_answer) == set(expected)
    and all(
        type(value) in (int, float) and math.isfinite(value)
        for value in parsed_answer.values()
    )
)

metric_checks = {}

for key, expected_value in expected.items():
    actual = (
        parsed_answer.get(key)
        if isinstance(parsed_answer, dict)
        else None
    )

    metric_checks[key] = (
        type(actual) in (int, float)
        and math.isfinite(actual)
        and math.isclose(
            actual,
            expected_value,
            rel_tol=0,
            abs_tol=0.01,
        )
    )

calculator_used = any(
    call["name"] == "calculator"
    for call in math_tool_calls
)

only_calculator_used = (
    calculator_used
    and all(
        call["name"] == "calculator"
        for call in math_tool_calls
    )
)

math_passed = (
    format_passed
    and all(metric_checks.values())
    and only_calculator_used
    and not math_last_message.tool_calls
)

math_report = {
    "case_id": MATH_CASE_ID,
    "sdk": "deepagents",
    "model": model.model_name,
    "runtime_status": math_runtime_status,
    "format_passed": format_passed,
    "metric_checks": metric_checks,
    "calculator_used": calculator_used,
    "only_calculator_used": only_calculator_used,
    "task_passed": math_passed,
    "latency_s": round(math_latency_s, 2),
    "tool_call_count": len(math_tool_calls),
    "answer": parsed_answer,
}

print(json.dumps(math_report, indent=2, ensure_ascii=False))

{
  "case_id": "math_001",
  "sdk": "deepagents",
  "model": "gpt-6-luna",
  "runtime_status": "completed",
  "format_passed": true,
  "metric_checks": {
    "revenue_growth_pct": true,
    "net_margin_pct": true,
    "liabilities_to_assets_pct": true
  },
  "calculator_used": true,
  "only_calculator_used": true,
  "task_passed": true,
  "latency_s": 15.92,
  "tool_call_count": 7,
  "answer": {
    "revenue_growth_pct": 25,
    "net_margin_pct": 12,
    "liabilities_to_assets_pct": 30
  }
}


<hr>

### Math02 (Complex) กรณีขอบเขตและการตรวจความถูกต้อง

เพิ่มความซับซ้อนจากกรณีแรก โดยเปรียบเทียบตัวชี้วัดของหลายบริษัท จัดอันดับผลลัพธ์ และคัดเลือกบริษัทตามเงื่อนไขที่กำหนด นอกจากตรวจคำตอบแล้ว ยังเก็บรายละเอียดการเรียก calculator และบันทึกสาเหตุของความคลาดเคลื่อนที่พบในผลทดลอง

กลุ่มเซลล์นี้ประกอบด้วยโจทย์หลายรูปแบบและการตรวจผลตามค่าคาดหมาย เพื่อให้เห็นว่า agent จัดการข้อมูล ตัวหาร และรูปแบบผลลัพธ์ได้ถูกต้องเพียงใด


In [32]:
MATH_002_CASE_ID = "math_002"

math_002_prompt = """
บริษัทสมมติสามแห่งมีข้อมูลจากงบรวมดังนี้
ตัวเลขทั้งหมดมีหน่วยเป็นล้านบาท และใช้หลักบัญชีเดียวกัน

Alpha:
- รายได้ปี 2024: 1000
- รายได้ปี 2025: 1250
- กำไรสุทธิปี 2025: 150
- หนี้สินรวม ณ สิ้นปี 2025: 600
- สินทรัพย์รวม ณ สิ้นปี 2025: 2000

Beta:
- รายได้ปี 2024: 800
- รายได้ปี 2025: 880
- กำไรสุทธิปี 2025: 176
- หนี้สินรวม ณ สิ้นปี 2025: 900
- สินทรัพย์รวม ณ สิ้นปี 2025: 1500

Gamma:
- รายได้ปี 2024: 1500
- รายได้ปี 2025: 1800
- กำไรสุทธิปี 2025: 180
- หนี้สินรวม ณ สิ้นปี 2025: 400
- สินทรัพย์รวม ณ สิ้นปี 2025: 2000

งานที่ต้องทำ:
1. คำนวณสามตัวชี้วัดของแต่ละบริษัทเป็นเปอร์เซ็นต์:
   - การเติบโตของรายได้ปี 2025 เทียบกับปี 2024
   - อัตรากำไรสุทธิปี 2025
   - หนี้สินรวมต่อสินทรัพย์รวม ณ สิ้นปี 2025
2. จัดอันดับบริษัท:
   - การเติบโตของรายได้: มากไปน้อย
   - อัตรากำไรสุทธิ: มากไปน้อย
   - หนี้สินรวมต่อสินทรัพย์รวม: น้อยไปมาก
3. คัดบริษัทที่ผ่านเงื่อนไขทั้งสองข้อ:
   - การเติบโตของรายได้ตั้งแต่ 20% ขึ้นไป
   - อัตรากำไรสุทธิตั้งแต่ 12% ขึ้นไป
   เรียงรายชื่อบริษัทที่ผ่านตามตัวอักษร

ข้อกำหนด:
- ใช้ calculator สำหรับการคำนวณ
- ใช้เฉพาะข้อมูลที่ให้
- เปอร์เซ็นต์ใช้ตัวเลข เช่น 15 สำหรับ 15%
- ปัดผลลัพธ์สุดท้ายเป็นทศนิยมไม่เกินสองตำแหน่ง
- ตอบ JSON object เท่านั้น ไม่ใส่ Markdown code fence
- มี top-level keys เท่านั้นดังนี้:

  companies:
    object ที่มี keys Alpha, Beta, Gamma
    แต่ละบริษัทมีสาม numeric keys:
      revenue_growth_pct
      net_margin_pct
      liabilities_to_assets_pct

  rankings:
    object ที่มีสาม keys แต่ละค่าเป็น list ของชื่อบริษัท:
      revenue_growth_desc
      net_margin_desc
      liabilities_to_assets_asc

  eligible_companies:
    list ของชื่อบริษัทที่ผ่านเงื่อนไข
"""

print("Case:", MATH_002_CASE_ID)
print(math_002_prompt)

Case: math_002

บริษัทสมมติสามแห่งมีข้อมูลจากงบรวมดังนี้
ตัวเลขทั้งหมดมีหน่วยเป็นล้านบาท และใช้หลักบัญชีเดียวกัน

Alpha:
- รายได้ปี 2024: 1000
- รายได้ปี 2025: 1250
- กำไรสุทธิปี 2025: 150
- หนี้สินรวม ณ สิ้นปี 2025: 600
- สินทรัพย์รวม ณ สิ้นปี 2025: 2000

Beta:
- รายได้ปี 2024: 800
- รายได้ปี 2025: 880
- กำไรสุทธิปี 2025: 176
- หนี้สินรวม ณ สิ้นปี 2025: 900
- สินทรัพย์รวม ณ สิ้นปี 2025: 1500

Gamma:
- รายได้ปี 2024: 1500
- รายได้ปี 2025: 1800
- กำไรสุทธิปี 2025: 180
- หนี้สินรวม ณ สิ้นปี 2025: 400
- สินทรัพย์รวม ณ สิ้นปี 2025: 2000

งานที่ต้องทำ:
1. คำนวณสามตัวชี้วัดของแต่ละบริษัทเป็นเปอร์เซ็นต์:
   - การเติบโตของรายได้ปี 2025 เทียบกับปี 2024
   - อัตรากำไรสุทธิปี 2025
   - หนี้สินรวมต่อสินทรัพย์รวม ณ สิ้นปี 2025
2. จัดอันดับบริษัท:
   - การเติบโตของรายได้: มากไปน้อย
   - อัตรากำไรสุทธิ: มากไปน้อย
   - หนี้สินรวมต่อสินทรัพย์รวม: น้อยไปมาก
3. คัดบริษัทที่ผ่านเงื่อนไขทั้งสองข้อ:
   - การเติบโตของรายได้ตั้งแต่ 20% ขึ้นไป
   - อัตรากำไรสุทธิตั้งแต่ 12% ขึ้นไป
   เรียงรายชื่อบริษัทที่ผ่านต

In [33]:
math_002_result = None
math_002_error  = None
math_002_status = "running"

started_at = time.perf_counter()

try:
    math_002_result = await asyncio.wait_for(
        math_agent.ainvoke(
            {
                "messages": [
                    {"role": "user", "content": math_002_prompt}
                ]
            },
            config={"recursion_limit": 80},
        ),
        timeout=180,
    )
    math_002_status = "completed"

except Exception as exc:
    math_002_status = "failed"
    math_002_error = f"{type(exc).__name__}: {exc}"

finally:
    math_002_latency_s = time.perf_counter() - started_at

print("Runtime status:", math_002_status)
print("Latency:", round(math_002_latency_s, 2), "seconds")

if math_002_error:
    print("Error:", math_002_error)

Runtime status: completed
Latency: 13.5 seconds


In [34]:
assert math_002_result is not None, (
    "Cell 15 ยังไม่สำเร็จ ให้ตรวจ error ก่อน"
)

math_002_messages = math_002_result["messages"]
math_002_calls = []

for message in math_002_messages:
    if isinstance(message, AIMessage):
        for call in message.tool_calls:
            math_002_calls.append(call)
            print("CALL:", call["name"])
            print("ARGS:", json.dumps(call["args"], ensure_ascii=False))

    elif isinstance(message, ToolMessage):
        print("RESULT:", message.content)
        print()

math_002_last = math_002_messages[-1]

assert isinstance(math_002_last, AIMessage), (
    "Run ไม่ได้จบด้วยข้อความจาก agent"
)

math_002_answer = math_002_last.text.strip()

print("FINAL ANSWER:")
print(math_002_answer)
print("\nTool calls:", len(math_002_calls))

CALL: calculator
ARGS: {"operation": "subtract", "a": 1250, "b": 1000}
CALL: calculator
ARGS: {"operation": "divide", "a": 150, "b": 1250}
CALL: calculator
ARGS: {"operation": "divide", "a": 600, "b": 2000}
CALL: calculator
ARGS: {"operation": "subtract", "a": 880, "b": 800}
CALL: calculator
ARGS: {"operation": "divide", "a": 176, "b": 880}
CALL: calculator
ARGS: {"operation": "divide", "a": 900, "b": 1500}
CALL: calculator
ARGS: {"operation": "subtract", "a": 1800, "b": 1500}
CALL: calculator
ARGS: {"operation": "divide", "a": 180, "b": 1800}
CALL: calculator
ARGS: {"operation": "divide", "a": 400, "b": 2000}
RESULT: 250.0

RESULT: 0.12

RESULT: 0.3

RESULT: 80.0

RESULT: 0.2

RESULT: 0.6

RESULT: 300.0

RESULT: 0.1

RESULT: 0.2

CALL: calculator
ARGS: {"operation": "divide", "a": 250, "b": 1000}
CALL: calculator
ARGS: {"operation": "divide", "a": 80, "b": 800}
CALL: calculator
ARGS: {"operation": "divide", "a": 300, "b": 1500}
CALL: calculator
ARGS: {"operation": "multiply", "a": 0.1

In [35]:
math_002_expected = {
    "companies": {
        "Alpha": {
            "revenue_growth_pct": 25.0,
            "net_margin_pct": 12.0,
            "liabilities_to_assets_pct": 30.0,
        },
        "Beta": {
            "revenue_growth_pct": 10.0,
            "net_margin_pct": 20.0,
            "liabilities_to_assets_pct": 60.0,
        },
        "Gamma": {
            "revenue_growth_pct": 20.0,
            "net_margin_pct": 10.0,
            "liabilities_to_assets_pct": 20.0,
        },
    },
    "rankings": {
        "revenue_growth_desc": ["Alpha", "Gamma", "Beta"],
        "net_margin_desc": ["Beta", "Alpha", "Gamma"],
        "liabilities_to_assets_asc": ["Gamma", "Alpha", "Beta"],
    },
    "eligible_companies": ["Alpha"],
}


def matches_expected(actual, expected):
    """Compare structure exactly; allow 0.01 tolerance for numbers."""
    if isinstance(expected, dict):
        return (
            isinstance(actual, dict)
            and set(actual) == set(expected)
            and all(
                matches_expected(actual[key], value)
                for key, value in expected.items()
            )
        )

    if isinstance(expected, list):
        return isinstance(actual, list) and actual == expected

    if type(expected) in (int, float):
        return (
            type(actual) in (int, float)
            and math.isfinite(actual)
            and math.isclose(actual, expected, rel_tol=0, abs_tol=0.01)
        )

    return type(actual) is type(expected) and actual == expected


try:
    math_002_parsed = json.loads(math_002_answer)
except json.JSONDecodeError:
    math_002_parsed = None

actual_root = (
    math_002_parsed if isinstance(math_002_parsed, dict) else {}
)

math_002_checks = {
    "top_level_structure": (
        isinstance(math_002_parsed, dict)
        and set(actual_root) == set(math_002_expected)
    ),
    "all_company_metrics": matches_expected(
        actual_root.get("companies"),
        math_002_expected["companies"],
    ),
    "all_rankings": matches_expected(
        actual_root.get("rankings"),
        math_002_expected["rankings"],
    ),
    "eligible_companies": matches_expected(
        actual_root.get("eligible_companies"),
        math_002_expected["eligible_companies"],
    ),
    "only_calculator_used": (
        len(math_002_calls) > 0
        and all(call["name"] == "calculator" for call in math_002_calls)
    ),
    "final_answer_complete": not math_002_last.tool_calls,
}

math_002_report = {
    "case_id": MATH_002_CASE_ID,
    "sdk": "deepagents",
    "model": model.model_name,
    "runtime_status": math_002_status,
    "checks": math_002_checks,
    "task_passed": (
        math_002_status == "completed"
        and all(math_002_checks.values())
    ),
    "latency_s": round(math_002_latency_s, 2),
    "tool_call_count": len(math_002_calls),
    "answer": math_002_parsed,
}

print(json.dumps(math_002_report, indent=2, ensure_ascii=False))

{
  "case_id": "math_002",
  "sdk": "deepagents",
  "model": "gpt-6-luna",
  "runtime_status": "completed",
  "checks": {
    "top_level_structure": true,
    "all_company_metrics": true,
    "all_rankings": true,
    "eligible_companies": true,
    "only_calculator_used": true,
    "final_answer_complete": true
  },
  "task_passed": true,
  "latency_s": 13.5,
  "tool_call_count": 17,
  "answer": {
    "companies": {
      "Alpha": {
        "revenue_growth_pct": 25,
        "net_margin_pct": 12,
        "liabilities_to_assets_pct": 30
      },
      "Beta": {
        "revenue_growth_pct": 10,
        "net_margin_pct": 20,
        "liabilities_to_assets_pct": 60
      },
      "Gamma": {
        "revenue_growth_pct": 20,
        "net_margin_pct": 10,
        "liabilities_to_assets_pct": 20
      }
    },
    "rankings": {
      "revenue_growth_desc": [
        "Alpha",
        "Gamma",
        "Beta"
      ],
      "net_margin_desc": [
        "Beta",
        "Alpha",
        "Gamma"
 

In [36]:
# โค้ดนี้เป็นการสร้าง Tool call แบบมี ID ว่าเราทำอะไรก่อนและหลังกันแน่
tool_results_by_id = {
    message.tool_call_id: message
    for message in math_002_messages
    if isinstance(message, ToolMessage)
}

for index, call in enumerate(math_002_calls, start=1):
    result_message = tool_results_by_id.get(call["id"])

    print(f"{index}. {call['name']}")
    print("   Arguments:", json.dumps(call["args"], ensure_ascii=False))
    print(
        "   Result:",
        result_message.content
        if result_message is not None
        else "(ไม่มีผลลัพธ์ที่จับคู่ได้)",
    )
    print()

1. calculator
   Arguments: {"operation": "subtract", "a": 1250, "b": 1000}
   Result: 250.0

2. calculator
   Arguments: {"operation": "divide", "a": 150, "b": 1250}
   Result: 0.12

3. calculator
   Arguments: {"operation": "divide", "a": 600, "b": 2000}
   Result: 0.3

4. calculator
   Arguments: {"operation": "subtract", "a": 880, "b": 800}
   Result: 80.0

5. calculator
   Arguments: {"operation": "divide", "a": 176, "b": 880}
   Result: 0.2

6. calculator
   Arguments: {"operation": "divide", "a": 900, "b": 1500}
   Result: 0.6

7. calculator
   Arguments: {"operation": "subtract", "a": 1800, "b": 1500}
   Result: 300.0

8. calculator
   Arguments: {"operation": "divide", "a": 180, "b": 1800}
   Result: 0.1

9. calculator
   Arguments: {"operation": "divide", "a": 400, "b": 2000}
   Result: 0.2

10. calculator
   Arguments: {"operation": "divide", "a": 250, "b": 1000}
   Result: 0.25

11. calculator
   Arguments: {"operation": "divide", "a": 80, "b": 800}
   Result: 0.1

12. calc

In [37]:
math_002_report["failure_analysis"] = {
    "category": "wrong_operand_selection",
    "metric": "Alpha.net_margin_pct",
    "observed_arguments": {
        "operation": "divide",
        "a": 150,
        "b": 1000,
    },
    "expected_arguments": {
        "operation": "divide",
        "a": 150,
        "b": 1250,
    },
    "calculator_correct_for_given_arguments": True,
}

print("Recorded:", math_002_report["failure_analysis"])

Recorded: {'category': 'wrong_operand_selection', 'metric': 'Alpha.net_margin_pct', 'observed_arguments': {'operation': 'divide', 'a': 150, 'b': 1000}, 'expected_arguments': {'operation': 'divide', 'a': 150, 'b': 1250}, 'calculator_correct_for_given_arguments': True}


### Math03 (Error handerling)

In [39]:
math_003_prompt = """
บริษัทสมมติ Delta มีข้อมูลจากงบรวม:

- รายได้ปี 2024: 0 ล้านบาท
- รายได้ปี 2025: 1.5 พันล้านบาท
- กำไรสุทธิปี 2025: ไม่มีข้อมูล
- หนี้สินรวม ณ สิ้นปี 2025: 600 ล้านบาท
- สินทรัพย์รวม ณ สิ้นปี 2025: 2 พันล้านบาท

จง:
1. แปลงรายได้ปี 2025 เป็นล้านบาท
2. คำนวณการเติบโตของรายได้ปี 2025 เทียบปี 2024 เป็นเปอร์เซ็นต์
3. คำนวณอัตรากำไรสุทธิปี 2025 เป็นเปอร์เซ็นต์
4. คำนวณหนี้สินรวมต่อสินทรัพย์รวมเป็นเปอร์เซ็นต์

ข้อกำหนด:
- ใช้ calculator สำหรับการคำนวณและแปลงหน่วย
- ข้อมูลขาด ห้ามสมมติตัวเลขขึ้นมา
- ค่าที่คำนวณไม่ได้ให้เป็น null ไม่ใช้ 0 หรือ Infinity แทน
- ระบุ reason_code ของแต่ละอัตราส่วน:
  OK = คำนวณได้
  ZERO_DENOMINATOR = ตัวหารเป็นศูนย์
  MISSING_DATA = ข้อมูลที่จำเป็นขาด
- เปอร์เซ็นต์ใช้ตัวเลข เช่น 15 สำหรับ 15%
- ตอบ JSON เท่านั้น ไม่ใส่ Markdown code fence
- ใช้โครงสร้างดังนี้ โดยแทนที่ข้อความตัวอย่างด้วยค่าจริง:

{
  "revenue_2025_million": "ตัวเลข",
  "revenue_growth": {
    "value_pct": "ตัวเลขหรือ null",
    "reason_code": "รหัส"
  },
  "net_margin": {
    "value_pct": "ตัวเลขหรือ null",
    "reason_code": "รหัส"
  },
  "liabilities_to_assets": {
    "value_pct": "ตัวเลขหรือ null",
    "reason_code": "รหัส"
  }
}
"""

print(math_003_prompt)


บริษัทสมมติ Delta มีข้อมูลจากงบรวม:

- รายได้ปี 2024: 0 ล้านบาท
- รายได้ปี 2025: 1.5 พันล้านบาท
- กำไรสุทธิปี 2025: ไม่มีข้อมูล
- หนี้สินรวม ณ สิ้นปี 2025: 600 ล้านบาท
- สินทรัพย์รวม ณ สิ้นปี 2025: 2 พันล้านบาท

จง:
1. แปลงรายได้ปี 2025 เป็นล้านบาท
2. คำนวณการเติบโตของรายได้ปี 2025 เทียบปี 2024 เป็นเปอร์เซ็นต์
3. คำนวณอัตรากำไรสุทธิปี 2025 เป็นเปอร์เซ็นต์
4. คำนวณหนี้สินรวมต่อสินทรัพย์รวมเป็นเปอร์เซ็นต์

ข้อกำหนด:
- ใช้ calculator สำหรับการคำนวณและแปลงหน่วย
- ข้อมูลขาด ห้ามสมมติตัวเลขขึ้นมา
- ค่าที่คำนวณไม่ได้ให้เป็น null ไม่ใช้ 0 หรือ Infinity แทน
- ระบุ reason_code ของแต่ละอัตราส่วน:
  OK = คำนวณได้
  ZERO_DENOMINATOR = ตัวหารเป็นศูนย์
  MISSING_DATA = ข้อมูลที่จำเป็นขาด
- เปอร์เซ็นต์ใช้ตัวเลข เช่น 15 สำหรับ 15%
- ตอบ JSON เท่านั้น ไม่ใส่ Markdown code fence
- ใช้โครงสร้างดังนี้ โดยแทนที่ข้อความตัวอย่างด้วยค่าจริง:

{
  "revenue_2025_million": "ตัวเลข",
  "revenue_growth": {
    "value_pct": "ตัวเลขหรือ null",
    "reason_code": "รหัส"
  },
  "net_margin": {
    "value_pct": "ตัวเลข

In [40]:
math_003_result = None
started_at = time.perf_counter()

try:
    math_003_result = await asyncio.wait_for(
        math_agent.ainvoke(
            {
                "messages": [
                    {"role": "user", "content": math_003_prompt}
                ]
            },
            config={"recursion_limit": 40},
        ),
        timeout=120,
    )
finally:
    math_003_latency_s = time.perf_counter() - started_at

math_003_messages = math_003_result["messages"]
math_003_calls = []

for message in math_003_messages:
    if isinstance(message, AIMessage):
        for call in message.tool_calls:
            math_003_calls.append(call)
            print("CALL:", call["name"], call["args"])
    elif isinstance(message, ToolMessage):
        print("RESULT:", message.content)

math_003_last = math_003_messages[-1]
assert isinstance(math_003_last, AIMessage)

math_003_answer = math_003_last.text.strip()

print("\nFINAL ANSWER:")
print(math_003_answer)
print("\nLatency:", round(math_003_latency_s, 2), "seconds")

CALL: calculator {'operation': 'multiply', 'a': 1.5, 'b': 1000}
RESULT: 1500.0
CALL: calculator {'operation': 'divide', 'a': 600, 'b': 2000}
RESULT: 0.3
CALL: calculator {'operation': 'multiply', 'a': 0.3, 'b': 100}
RESULT: 30.0

FINAL ANSWER:
{
  "revenue_2025_million": 1500,
  "revenue_growth": {
    "value_pct": null,
    "reason_code": "ZERO_DENOMINATOR"
  },
  "net_margin": {
    "value_pct": null,
    "reason_code": "MISSING_DATA"
  },
  "liabilities_to_assets": {
    "value_pct": 30,
    "reason_code": "OK"
  }
}

Latency: 10.73 seconds


In [41]:
math_003_expected = {
    "revenue_2025_million": 1500.0,
    "revenue_growth": {
        "value_pct": None,
        "reason_code": "ZERO_DENOMINATOR",
    },
    "net_margin": {
        "value_pct": None,
        "reason_code": "MISSING_DATA",
    },
    "liabilities_to_assets": {
        "value_pct": 30.0,
        "reason_code": "OK",
    },
}

try:
    math_003_parsed = json.loads(math_003_answer)
except json.JSONDecodeError:
    math_003_parsed = None

actual = (
    math_003_parsed if isinstance(math_003_parsed, dict) else {}
)

math_003_checks = {
    key: matches_expected(actual.get(key), expected)
    for key, expected in math_003_expected.items()
}

math_003_checks["exact_structure_and_values"] = matches_expected(
    math_003_parsed, math_003_expected
)

math_003_checks["only_calculator_used"] = (
    len(math_003_calls) > 0
    and all(call["name"] == "calculator" for call in math_003_calls)
)

math_003_checks["final_answer_complete"] = not math_003_last.tool_calls

math_003_report = {
    "case_id": "math_003",
    "sdk": "deepagents",
    "model": model.model_name,
    "runtime_status": "completed",
    "checks": math_003_checks,
    "task_passed": all(math_003_checks.values()),
    "latency_s": round(math_003_latency_s, 2),
    "tool_call_count": len(math_003_calls),
    "answer": math_003_parsed,
}

print(json.dumps(math_003_report, indent=2, ensure_ascii=False))

{
  "case_id": "math_003",
  "sdk": "deepagents",
  "model": "gpt-6-luna",
  "runtime_status": "completed",
  "checks": {
    "revenue_2025_million": true,
    "revenue_growth": true,
    "net_margin": true,
    "liabilities_to_assets": true,
    "exact_structure_and_values": true,
    "only_calculator_used": true,
    "final_answer_complete": true
  },
  "task_passed": true,
  "latency_s": 10.73,
  "tool_call_count": 3,
  "answer": {
    "revenue_2025_million": 1500,
    "revenue_growth": {
      "value_pct": null,
      "reason_code": "ZERO_DENOMINATOR"
    },
    "net_margin": {
      "value_pct": null,
      "reason_code": "MISSING_DATA"
    },
    "liabilities_to_assets": {
      "value_pct": 30,
      "reason_code": "OK"
    }
  }
}


### บันทึกผลการทดสอบ Math

In [42]:
shared_dir = Path("/workspace/shared")

assert shared_dir.is_dir(), (
    "ไม่พบ /workspace/shared "
    "กรุณาตรวจ volume mapping ใน docker-compose.yml"
)

output_dir = shared_dir / "results" / "deepagents"
output_dir.mkdir(parents=True, exist_ok=True)

saved_at = datetime.now(timezone.utc)

math_checkpoint = {
    "saved_at_utc": saved_at.isoformat(),
    "versions": {
        name: version(name)
        for name in ["deepagents", "langchain", "langchain-openai"]
    },
    "results": [
        math_report,
        math_002_report,
        math_003_report,
    ],
}

output_path = output_dir / (
    f"math_{saved_at:%Y%m%dT%H%M%SZ}_{uuid4().hex[:8]}.json"
)

with output_path.open("x", encoding="utf-8") as file:
    json.dump(
        math_checkpoint,
        file,
        indent=2,
        ensure_ascii=False,
        allow_nan=False,
    )

print("Saved:", output_path)
print("Experiments:", len(math_checkpoint["results"]))

Saved: /workspace/shared/results/deepagents/math_20261006T053337Z_502bf588.json
Experiments: 3


<hr>

# 3.MCP

ทดสอบการให้ Deep Agent เรียกเครื่องมือผ่าน Model Context Protocol (MCP) โดยใช้เซิร์ฟเวอร์จำลองและบันทึก trace ของคำขอและผลตอบกลับ เพื่อประเมินการเลือกใช้เครื่องมือ การจัดการข้อผิดพลาด และการลองใหม่

In [43]:
import asyncio
import json
import time

from importlib.metadata import version
from fastmcp import FastMCP                     # สร้าง MCP server แบบจำลอง
from langchain.mcp import MCPAdapter            # เชื่อม MCP client เข้ากับ LangChain และแปลงเครื่องมือ MCP ให้อยู่ในรูปที่ agent ใช้ได้
from deepagents import create_deep_agent        # สร้าง Deepagent
from langchain_core.messages import AIMessage   # ดู Tool call เวลาที่ agent ตอบกลับ

/tmp/ipykernel_85/3825417229.py:7: LangChainBetaWarning: `langchain.mcp` is in beta. It is actively being worked on, so the API may change.
  from langchain.mcp import MCPAdapter


In [44]:
MCP_VERSIONS = {
    name: version(name)
    for name in [
        "deepagents",
        "langchain",
        "langchain-openai",
        "fastmcp",
    ]
}

print(json.dumps(MCP_VERSIONS, indent=2))
print("Model:", model.model_name)

{
  "deepagents": "0.7.21",
  "langchain": "1.4.3",
  "langchain-openai": "1.6.7",
  "fastmcp": "4.0.11"
}
Model: gpt-6-luna


### Create Virtual MCP

In [45]:
finance_mcp = FastMCP("Finance Benchmark")

mcp_server_log = []

COMPANY = {
    "company_id": "cmp_a17",
    "company_name": "Alpha",
}

STATEMENT = {
    "company_id": "cmp_a17",
    "year": 2025,
    "revenue_million": 1375,
    "net_profit_million": 165,
    "source_id": "fixture_alpha_2025_v1",
}


@finance_mcp.tool()
def search_company(query: str) -> dict:
    """Search company names and return matching company IDs and names."""
    normalized = query.strip().casefold()

    matches = (
        [COMPANY.copy()]
        if normalized and normalized in COMPANY["company_name"].casefold()
        else []
    )

    output = {"companies": matches}

    mcp_server_log.append({
        "tool": "search_company",
        "arguments": {"query": query},
        "output": output,
    })

    return output


@finance_mcp.tool()
def get_financial_statement(company_id: str, year: int) -> dict:
    """Retrieve a financial statement using a company ID and year."""
    found = (
        company_id == STATEMENT["company_id"]
        and year == STATEMENT["year"]
    )

    output = (
        STATEMENT.copy()
        if found
        else {"error": "STATEMENT_NOT_FOUND"}
    )

    mcp_server_log.append({
        "tool": "get_financial_statement",
        "arguments": {"company_id": company_id, "year": year},
        "output": output,
    })

    return output


print("MCP server ready.")

MCP server ready.


### MCP01 (ดูงบบริษัท)

กรณีพื้นฐานสำหรับเรียกเครื่องมือ MCP เพื่อขอข้อมูลงบ ตรวจรูปแบบคำตอบและ trace ที่เซิร์ฟเวอร์ได้รับ


In [47]:
mcp_prompt = """
หาข้อมูลงบปี 2025 ของบริษัท Alpha จากเครื่องมือที่ให้

ตอบ JSON object เท่านั้น มี keys:
- company_id
- year
- revenue_million
- net_profit_million
- source_id

ใช้ข้อมูลจากผลเครื่องมือจริง ห้ามเดารหัสบริษัทหรือตัวเลข
ไม่ใส่ Markdown code fence
"""

async def run_mcp_case():
    setup_started = time.perf_counter()
    async with MCPAdapter(finance_mcp) as adapter:
        tools = await adapter.list_tools()
        discovered = [tool.name for tool in tools]

        print("Discovered tools:", discovered)
        agent = create_deep_agent(
            model=model,
            tools=tools,
            system_prompt=(
                "Answer using the provided financial data tools. "
                "Resolve the company before retrieving its statement. "
                "Do not guess identifiers or financial figures. "
                "Do not delegate or use other tools. "
                "Follow the requested JSON format."
            ),
        )
        setup_s = time.perf_counter() - setup_started
        agent_started = time.perf_counter()
        result = await agent.ainvoke(
            {"messages": [{"role": "user", "content": mcp_prompt}]},
            config={"recursion_limit": 40},
        )
        agent_s = time.perf_counter() - agent_started
    return {
        "result": result,
        "discovered_tools": discovered,
        "setup_s": setup_s,
        "agent_s": agent_s,
    }

In [48]:
mcp_run = None
mcp_error = None
mcp_status = "running"

mcp_server_log.clear()
started_at = time.perf_counter()
try:
    mcp_run = await asyncio.wait_for(
        run_mcp_case(),
        timeout=120,
    )
    mcp_status = "completed"

except Exception as exc:
    mcp_status = "failed"
    mcp_error = f"{type(exc).__name__}: {exc}"

finally:
    mcp_total_s = time.perf_counter() - started_at

print("Runtime status:", mcp_status)
print("Total latency:", round(mcp_total_s, 2), "seconds")

if mcp_error:
    print("Error:", mcp_error)

Discovered tools: ['search_company', 'get_financial_statement']
Runtime status: completed
Total latency: 9.26 seconds


In [50]:
assert mcp_run is not None, "ตรวจ error จาก MCP Cell 5 ก่อน"

print("SERVER LOG:")
print(json.dumps(mcp_server_log, indent=2, ensure_ascii=False))

mcp_messages = mcp_run["result"]["messages"]
mcp_last = mcp_messages[-1]

assert isinstance(mcp_last, AIMessage), ("Run ไม่ได้จบด้วยข้อความจาก agent")

mcp_answer = mcp_last.text.strip()
try:
    mcp_parsed = json.loads(mcp_answer)
except json.JSONDecodeError:
    mcp_parsed = None

# ตรวจ dependency: ผลค้นหาบริษัทต้องมาก่อนการดึงงบ
chain_passed = False
for index, event in enumerate(mcp_server_log):
    if event["tool"] != "search_company":
        continue
    found_ids = {company["company_id"] for company in event["output"]["companies"]}
    for later_event in mcp_server_log[index + 1:]:
        if later_event["tool"] != "get_financial_statement":
            continue
        arguments = later_event["arguments"]
        if (
            arguments["company_id"] in found_ids
            and arguments["year"] == 2025
            and "error" not in later_event["output"]
        ):
            chain_passed = True

expected_tools = {"search_company", "get_financial_statement"}
agent_calls = [
    call
    for message in mcp_messages
    if isinstance(message, AIMessage)
    for call in message.tool_calls
]

mcp_checks = {
    "tools_discovered": (
        set(mcp_run["discovered_tools"]) == expected_tools
    ),
    "company_id_forwarded": chain_passed,
    "answer_correct": (
        isinstance(mcp_parsed, dict)
        and mcp_parsed == STATEMENT
    ),
    "only_expected_tools": (
        len(agent_calls) > 0
        and all(call["name"] in expected_tools for call in agent_calls)
    ),
    "final_answer_complete": not mcp_last.tool_calls,
}

mcp_report = {
    "case_id": "mcp_001",
    "sdk": "deepagents",
    "model": model.model_name,
    "versions": MCP_VERSIONS,
    "transport": "in_memory",
    "runtime_status": mcp_status,
    "checks": mcp_checks,
    "task_passed": (
        mcp_status == "completed"
        and all(mcp_checks.values())
    ),
    "setup_s": round(mcp_run["setup_s"], 2),
    "agent_s": round(mcp_run["agent_s"], 2),
    "latency_s": round(mcp_total_s, 2),
    "server_tool_calls": len(mcp_server_log),
    "answer": mcp_parsed,
}

print("\nREPORT:")
print(json.dumps(mcp_report, indent=2, ensure_ascii=False))

SERVER LOG:
[
  {
    "tool": "search_company",
    "arguments": {
      "query": "Alpha"
    },
    "output": {
      "companies": [
        {
          "company_id": "cmp_a17",
          "company_name": "Alpha"
        }
      ]
    }
  },
  {
    "tool": "get_financial_statement",
    "arguments": {
      "company_id": "cmp_a17",
      "year": 2025
    },
    "output": {
      "company_id": "cmp_a17",
      "year": 2025,
      "revenue_million": 1375,
      "net_profit_million": 165,
      "source_id": "fixture_alpha_2025_v1"
    }
  }
]

REPORT:
{
  "case_id": "mcp_001",
  "sdk": "deepagents",
  "model": "gpt-6-luna",
  "versions": {
    "deepagents": "0.7.21",
    "langchain": "1.4.3",
    "langchain-openai": "1.6.7",
    "fastmcp": "4.0.11"
  },
  "transport": "in_memory",
  "runtime_status": "completed",
  "checks": {
    "tools_discovered": true,
    "company_id_forwarded": true,
    "answer_correct": true,
    "only_expected_tools": true,
    "final_answer_complete": true
  },

### MCP02 (หาบริษัทที่หา ไม่ได้มีตัวตน)

กรณีทดสอบถัดไปตรวจการทำงานกับคำขออีกแบบ โดยเก็บ trace ของกรณีก่อนหน้าไว้ก่อนเริ่มการทดลองใหม่ เพื่อแยกผลของแต่ละเคสได้ชัดเจน


In [51]:
mcp_server_log

[{'tool': 'search_company',
  'arguments': {'query': 'Alpha'},
  'output': {'companies': [{'company_id': 'cmp_a17',
     'company_name': 'Alpha'}]}},
 {'tool': 'get_financial_statement',
  'arguments': {'company_id': 'cmp_a17', 'year': 2025},
  'output': {'company_id': 'cmp_a17',
   'year': 2025,
   'revenue_million': 1375,
   'net_profit_million': 165,
   'source_id': 'fixture_alpha_2025_v1'}}]

In [52]:
from copy import deepcopy

# เก็บ trace ของ mcp_001 ก่อนล้าง log เพื่อรันโจทย์ใหม่
mcp_001_trace = deepcopy(mcp_server_log)

mcp_002_prompt = """
ค้นหาบริษัท Omega จากเครื่องมือที่ให้ และดึงงบปี 2025 หากพบบริษัท

ข้อกำหนด:
- ห้ามเดา company ID หรือใช้ข้อมูลของบริษัทอื่นแทน
- หากไม่พบบริษัท ให้รายงานว่าไม่พบและไม่เรียกดึงงบ
- ตอบ JSON object เท่านั้น ไม่ใส่ Markdown code fence

โครงสร้างคำตอบ:
{
  "status": "found หรือ not_found",
  "company_name": "Omega",
  "statement": "ข้อมูลงบจากเครื่องมือ หรือ null หากไม่พบ"
}
"""

print(mcp_002_prompt)


ค้นหาบริษัท Omega จากเครื่องมือที่ให้ และดึงงบปี 2025 หากพบบริษัท

ข้อกำหนด:
- ห้ามเดา company ID หรือใช้ข้อมูลของบริษัทอื่นแทน
- หากไม่พบบริษัท ให้รายงานว่าไม่พบและไม่เรียกดึงงบ
- ตอบ JSON object เท่านั้น ไม่ใส่ Markdown code fence

โครงสร้างคำตอบ:
{
  "status": "found หรือ not_found",
  "company_name": "Omega",
  "statement": "ข้อมูลงบจากเครื่องมือ หรือ null หากไม่พบ"
}



In [53]:
async def run_mcp_case(prompt):
    setup_started = time.perf_counter()

    async with MCPAdapter(finance_mcp) as adapter:
        tools = await adapter.list_tools()
        discovered = [tool.name for tool in tools]

        agent = create_deep_agent(
            model=model,
            tools=tools,
            system_prompt=(
                "Answer using the provided financial data tools. "
                "Resolve the company before retrieving its statement. "
                "Do not guess identifiers or financial figures. "
                "Do not delegate or use other tools. "
                "Follow the requested JSON format."
            ),
        )

        setup_s = time.perf_counter() - setup_started
        agent_started = time.perf_counter()

        result = await agent.ainvoke(
            {"messages": [{"role": "user", "content": prompt}]},
            config={"recursion_limit": 40},
        )

        agent_s = time.perf_counter() - agent_started

    return {
        "result": result,
        "discovered_tools": discovered,
        "setup_s": setup_s,
        "agent_s": agent_s,
    }

In [54]:
mcp_002_run = None
mcp_002_error = None
mcp_002_status = "running"

mcp_server_log.clear()
started_at = time.perf_counter()

try:
    mcp_002_run = await asyncio.wait_for(
        run_mcp_case(mcp_002_prompt),
        timeout=120,
    )
    mcp_002_status = "completed"

except Exception as exc:
    mcp_002_status = "failed"
    mcp_002_error = f"{type(exc).__name__}: {exc}"

finally:
    mcp_002_total_s = time.perf_counter() - started_at
    mcp_002_trace = deepcopy(mcp_server_log)

print("Runtime status:", mcp_002_status)
print("Latency:", round(mcp_002_total_s, 2), "seconds")

if mcp_002_error:
    print("Error:", mcp_002_error)

Runtime status: completed
Latency: 6.68 seconds


In [55]:
assert mcp_002_run is not None, "ตรวจ error จาก Cell 3 ก่อน"

print("SERVER LOG:")
print(json.dumps(mcp_002_trace, indent=2, ensure_ascii=False))

messages_002 = mcp_002_run["result"]["messages"]
last_002 = messages_002[-1]

assert isinstance(last_002, AIMessage)

answer_002 = last_002.text.strip()

try:
    parsed_002 = json.loads(answer_002)
except json.JSONDecodeError:
    parsed_002 = None

calls_002 = [
    call
    for message in messages_002
    if isinstance(message, AIMessage)
    for call in message.tool_calls
]

search_events = [
    event
    for event in mcp_002_trace
    if event["tool"] == "search_company"
]

expected_002 = {
    "status": "not_found",
    "company_name": "Omega",
    "statement": None,
}

checks_002 = {
    "tools_discovered": (
        set(mcp_002_run["discovered_tools"])
        == {"search_company", "get_financial_statement"}
    ),
    "received_empty_search_result": any(
        event["output"]["companies"] == []
        for event in search_events
    ),
    "did_not_fetch_statement": not any(
        event["tool"] == "get_financial_statement"
        for event in mcp_002_trace
    ),
    "only_search_tool_used": (
        len(calls_002) > 0
        and all(call["name"] == "search_company" for call in calls_002)
    ),
    "answer_correct": parsed_002 == expected_002,
    "final_answer_complete": not last_002.tool_calls,
}

mcp_002_report = {
    "case_id": "mcp_002",
    "sdk": "deepagents",
    "model": model.model_name,
    "versions": MCP_VERSIONS,
    "transport": "in_memory",
    "runtime_status": mcp_002_status,
    "checks": checks_002,
    "task_passed": (
        mcp_002_status == "completed"
        and all(checks_002.values())
    ),
    "setup_s": round(mcp_002_run["setup_s"], 2),
    "agent_s": round(mcp_002_run["agent_s"], 2),
    "latency_s": round(mcp_002_total_s, 2),
    "server_tool_calls": len(mcp_002_trace),
    "answer": parsed_002,
}

print("\nREPORT:")
print(json.dumps(mcp_002_report, indent=2, ensure_ascii=False))

SERVER LOG:
[
  {
    "tool": "search_company",
    "arguments": {
      "query": "Omega"
    },
    "output": {
      "companies": []
    }
  }
]

REPORT:
{
  "case_id": "mcp_002",
  "sdk": "deepagents",
  "model": "gpt-6-luna",
  "versions": {
    "deepagents": "0.7.21",
    "langchain": "1.4.3",
    "langchain-openai": "1.6.7",
    "fastmcp": "4.0.11"
  },
  "transport": "in_memory",
  "runtime_status": "completed",
  "checks": {
    "tools_discovered": true,
    "received_empty_search_result": true,
    "did_not_fetch_statement": true,
    "only_search_tool_used": true,
    "answer_correct": true,
    "final_answer_complete": true
  },
  "task_passed": true,
  "setup_s": 0.04,
  "agent_s": 6.64,
  "latency_s": 6.68,
  "server_tool_calls": 1,
  "answer": {
    "status": "not_found",
    "company_name": "Omega",
    "statement": null
  }
}


<hr>

### MCP03 (ถ้าหากว่ามี Error ให้ทำการ Retry)

โค้ดชุดนี้ทดสอบว่า agent จะ ลองเรียกเครื่องมือซ้ำหนึ่งครั้งเมื่อได้รับ error ที่ระบุว่า retry ได้ แล้วนำข้อมูลที่ได้สำเร็จมาตอบหรือไม่

In [56]:
def build_retry_mcp():
    server = FastMCP("Finance Retry Benchmark")
    trace = []
    state = {"statement_attempts": 0}

    @server.tool()
    def search_company(query: str) -> dict:
        """Search company names and return matching company IDs and names."""
        normalized = query.strip().casefold()

        output = {
            "companies": (
                [COMPANY.copy()]
                if normalized
                and normalized in COMPANY["company_name"].casefold()
                else []
            )
        }

        trace.append({
            "tool": "search_company",
            "arguments": {"query": query},
            "output": output,
        })

        return output

    @server.tool()
    def get_financial_statement(company_id: str, year: int) -> dict:
        """Retrieve a financial statement by company ID and year.

        On failure, the response includes an error code and retryable flag.
        """
        valid_request = (
            company_id == STATEMENT["company_id"]
            and year == STATEMENT["year"]
        )

        if not valid_request:
            output = {
                "error": {
                    "code": "STATEMENT_NOT_FOUND",
                    "retryable": False,
                }
            }
        else:
            state["statement_attempts"] += 1

            if state["statement_attempts"] == 1:
                output = {
                    "error": {
                        "code": "TEMPORARY_UNAVAILABLE",
                        "retryable": True,
                    }
                }
            else:
                output = STATEMENT.copy()

        trace.append({
            "tool": "get_financial_statement",
            "arguments": {"company_id": company_id, "year": year},
            "output": output,
        })

        return output

    return server, trace


print("Retry server factory ready.")

Retry server factory ready.


In [57]:
async def run_mcp_on_server(server, prompt):
    setup_started = time.perf_counter()

    async with MCPAdapter(server) as adapter:
        tools = await adapter.list_tools()

        agent = create_deep_agent(
            model=model,
            tools=tools,
            system_prompt=(
                "Answer using the provided financial data tools. "
                "Resolve the company before retrieving its statement. "
                "Do not guess identifiers or financial figures. "
                "Do not delegate or use other tools. "
                "Follow the requested JSON format."
            ),
        )

        setup_s = time.perf_counter() - setup_started
        agent_started = time.perf_counter()

        result = await agent.ainvoke(
            {"messages": [{"role": "user", "content": prompt}]},
            config={"recursion_limit": 40},
        )

        agent_s = time.perf_counter() - agent_started

    return {
        "result": result,
        "discovered_tools": [tool.name for tool in tools],
        "setup_s": setup_s,
        "agent_s": agent_s,
    }

In [58]:
mcp_003_prompt = """
หาข้อมูลงบปี 2025 ของบริษัท Alpha จากเครื่องมือที่ให้

นโยบายจัดการข้อผิดพลาด:
- หาก tool คืน error ที่ retryable เป็น true ให้ลองคำขอนั้นใหม่
  ได้ไม่เกินหนึ่งครั้ง
- หากยังไม่สำเร็จ ให้รายงานข้อผิดพลาดตามจริง
- ห้ามเดารหัสบริษัทหรือตัวเลข

เมื่อสำเร็จ ตอบ JSON object เท่านั้น มี keys:
- company_id
- year
- revenue_million
- net_profit_million
- source_id

ไม่ใส่ Markdown code fence
"""

# สร้างใหม่ทุกครั้ง เพื่อ reset การจำลอง error
retry_server, mcp_003_trace = build_retry_mcp()

mcp_003_run = None
mcp_003_error = None
mcp_003_status = "running"

started_at = time.perf_counter()

try:
    mcp_003_run = await asyncio.wait_for(
        run_mcp_on_server(retry_server, mcp_003_prompt),
        timeout=120,
    )
    mcp_003_status = "completed"

except Exception as exc:
    mcp_003_status = "failed"
    mcp_003_error = f"{type(exc).__name__}: {exc}"

finally:
    mcp_003_total_s = time.perf_counter() - started_at

print("Runtime status:", mcp_003_status)
print("Latency:", round(mcp_003_total_s, 2), "seconds")

if mcp_003_error:
    print("Error:", mcp_003_error)

Runtime status: completed
Latency: 12.42 seconds


In [59]:
assert mcp_003_run is not None, "ตรวจ error จาก Cell 3 ก่อน"

print("SERVER LOG:")
print(json.dumps(mcp_003_trace, indent=2, ensure_ascii=False))

messages_003 = mcp_003_run["result"]["messages"]
last_003 = messages_003[-1]

assert isinstance(last_003, AIMessage)

try:
    parsed_003 = json.loads(last_003.text.strip())
except json.JSONDecodeError:
    parsed_003 = None

statement_events = [
    event
    for event in mcp_003_trace
    if event["tool"] == "get_financial_statement"
]

expected_arguments = {
    "company_id": COMPANY["company_id"],
    "year": 2025,
}

retry_sequence_correct = (
    len(statement_events) == 2
    and all(
        event["arguments"] == expected_arguments
        for event in statement_events
    )
    and statement_events[0]["output"].get("error") == {
        "code": "TEMPORARY_UNAVAILABLE",
        "retryable": True,
    }
    and statement_events[1]["output"] == STATEMENT
)

# ยืนยันว่าค้นพบ ID ก่อนดึงงบ
resolved_ids = set()
company_id_forwarded = True

for event in mcp_003_trace:
    if event["tool"] == "search_company":
        resolved_ids.update(
            company["company_id"]
            for company in event["output"]["companies"]
        )
    elif event["tool"] == "get_financial_statement":
        if event["arguments"]["company_id"] not in resolved_ids:
            company_id_forwarded = False

calls_003 = [
    call
    for message in messages_003
    if isinstance(message, AIMessage)
    for call in message.tool_calls
]

expected_tools = {"search_company", "get_financial_statement"}

checks_003 = {
    "tools_discovered": (
        set(mcp_003_run["discovered_tools"]) == expected_tools
    ),
    "company_id_forwarded": (
        bool(statement_events) and company_id_forwarded
    ),
    "retried_once_and_recovered": retry_sequence_correct,
    "answer_correct": parsed_003 == STATEMENT,
    "only_expected_tools": (
        bool(calls_003)
        and all(call["name"] in expected_tools for call in calls_003)
    ),
    "final_answer_complete": not last_003.tool_calls,
}

mcp_003_report = {
    "case_id": "mcp_003",
    "sdk": "deepagents",
    "model": model.model_name,
    "versions": MCP_VERSIONS,
    "transport": "in_memory",
    "failure_mode": "retryable_error_in_tool_result",
    "runtime_status": mcp_003_status,
    "checks": checks_003,
    "task_passed": (
        mcp_003_status == "completed"
        and all(checks_003.values())
    ),
    "setup_s": round(mcp_003_run["setup_s"], 2),
    "agent_s": round(mcp_003_run["agent_s"], 2),
    "latency_s": round(mcp_003_total_s, 2),
    "server_tool_calls": len(mcp_003_trace),
    "answer": parsed_003,
}

print("\nREPORT:")
print(json.dumps(mcp_003_report, indent=2, ensure_ascii=False))

SERVER LOG:
[
  {
    "tool": "search_company",
    "arguments": {
      "query": "Alpha"
    },
    "output": {
      "companies": [
        {
          "company_id": "cmp_a17",
          "company_name": "Alpha"
        }
      ]
    }
  },
  {
    "tool": "get_financial_statement",
    "arguments": {
      "company_id": "cmp_a17",
      "year": 2025
    },
    "output": {
      "error": {
        "code": "TEMPORARY_UNAVAILABLE",
        "retryable": true
      }
    }
  },
  {
    "tool": "get_financial_statement",
    "arguments": {
      "company_id": "cmp_a17",
      "year": 2025
    },
    "output": {
      "company_id": "cmp_a17",
      "year": 2025,
      "revenue_million": 1375,
      "net_profit_million": 165,
      "source_id": "fixture_alpha_2025_v1"
    }
  }
]

REPORT:
{
  "case_id": "mcp_003",
  "sdk": "deepagents",
  "model": "gpt-6-luna",
  "versions": {
    "deepagents": "0.7.21",
    "langchain": "1.4.3",
    "langchain-openai": "1.6.7",
    "fastmcp": "4.0.11"
  },


### บันทึกผลการทดสอบ MCP

รวบรวมผลและรายละเอียดการเรียกเครื่องมือเป็นระเบียน แล้วจัดเก็บเป็นไฟล์เพื่อใช้ตรวจสอบหรือเปรียบเทียบภายหลัง

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from uuid import uuid4
from copy import deepcopy
import json

shared_dir = Path("/workspace/shared")
assert shared_dir.is_dir(), "ไม่พบ /workspace/shared ให้ตรวจ Docker mount"

output_dir = shared_dir / "results" / "deepagents"
output_dir.mkdir(parents=True, exist_ok=True)

cases = [
    (mcp_report, mcp_prompt, mcp_001_trace),
    (mcp_002_report, mcp_002_prompt, mcp_002_trace),
    (mcp_003_report, mcp_003_prompt, mcp_003_trace),
]

results = []

for report, prompt, trace in cases:
    # เก็บ ID เดิมหากกดบันทึกผลชุดเดิมซ้ำ
    report.setdefault("run_id", uuid4().hex)

    record = deepcopy(report)
    record["prompt"] = prompt
    record["server_trace"] = deepcopy(trace)

    results.append(record)

saved_at = datetime.now(timezone.utc)

mcp_checkpoint = {
    "schema_version": "1.0",
    "saved_at_utc": saved_at.isoformat(),
    "sdk": "deepagents",
    "category": "mcp",
    "results": results,
}

output_path = output_dir / (
    f"mcp_{saved_at:%Y%m%dT%H%M%SZ}_{uuid4().hex[:8]}.json"
)

# แปลงให้สำเร็จก่อนสร้างไฟล์
payload = json.dumps(
    mcp_checkpoint,
    indent=2,
    ensure_ascii=False,
    allow_nan=False,
)

with output_path.open("x", encoding="utf-8") as file:
    file.write(payload)

print("Saved:", output_path)
print("Experiments:", len(results))
print("Passed:", sum(item["task_passed"] for item in results))

for item in results:
    print(
        f"- {item['case_id']}: "
        f"passed={item['task_passed']}, "
        f"calls={item['server_tool_calls']}, "
        f"latency={item['latency_s']}s"
    )

<hr>

# 4.Web search

ทดสอบความสามารถในการค้นคว้าข้อมูลบนเว็บ โดยตรวจทั้งการใช้เครื่องมือค้นหาและการเปิดอ่านแหล่งข้อมูล รวมถึงความครบถ้วนของคำตอบและแหล่งอ้างอิง

In [80]:
import asyncio
import json
import time

from collections import Counter
from copy import deepcopy
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path
from uuid import uuid4

from deepagents import create_deep_agent
from langchain_openai import ChatOpenAI
from langchain_core.messages import AIMessage
from IPython.display import display, Markdown

In [81]:
web_versions = {
    name: version(name)
    for name in [
        "deepagents",
        "langchain",
        "langchain-openai",
        "openai",
    ]
}

In [82]:
native_web_system_prompt = """
You are a financial research assistant.

Search the web for official sources, then open relevant URLs
found during search using the web tool's page-opening capability.
Read the opened sources before analyzing the figures.
Do not rely only on search snippets.

For comparisons between companies, open supporting sources
for each company.

Prefer official company publications or filings submitted to the SEC.
Distinguish fiscal years from calendar years and quarters.
Cite sources supporting your claims.
If you cannot open a source or evidence is insufficient,
state the limitation. Do not invent data or claim a page was read.

Do not delegate or use other tools for this task.
Answer in Thai.
""".strip()

native_web_model = ChatOpenAI(
    model=MODEL,
    use_responses_api=True, 
    include=["web_search_call.action.sources"],
    timeout=90,
    max_retries=0,
)

In [83]:
native_web_agent = create_deep_agent(
    model=native_web_model,
    tools=[{"type": "web_search"}], # Websearch + Webfetch
    system_prompt=native_web_system_prompt,
)

print("Model:", native_web_model.model_name)
print("Web search + page opening: OpenAI hosted tool through Deep Agents")

Model: gpt-6-luna
Web search + page opening: OpenAI hosted tool through Deep Agents


### Websearch01 (ค้นหาและเปรียบเทียบ)

เปรียบเทียบรายได้ Microsoft FY2024 กับ FY2023 โดยค้นเว็บและเปิดอ่านแหล่งข้อมูลทางการก่อนตอบ

In [84]:
native_web_prompt = """
ค้นเว็บเพื่อเปรียบเทียบรายได้รวมของ Microsoft
สำหรับปีงบประมาณ 2024 และ 2023

ขั้นตอนที่ต้องทำ:
- ค้นหาแหล่งข้อมูลทางการของ Microsoft
- เปิดอ่าน URL ที่พบจากการค้นหาอย่างน้อยหนึ่งหน้า
- ตรวจตัวเลขจากเนื้อหาที่เปิดอ่าน ไม่ใช้เพียงข้อความย่อผลค้นหา

ข้อกำหนด:
1. ใช้รายได้รวมทั้งปี ไม่ใช้รายไตรมาสหรือราย segment
2. แสดงตัวเลขเป็นล้านดอลลาร์สหรัฐ
3. ระบุวันสิ้นสุดปีงบประมาณของแต่ละปี
4. คำนวณอัตราการเติบโต พร้อมแสดงสูตรและการแทนค่า
5. แสดงอัตราการเติบโตทศนิยมสองตำแหน่ง
6. สรุปภาษาไทยสั้น ๆ ว่ารายได้เปลี่ยนแปลงอย่างไร
7. อ้างอิงแหล่งข้อมูลที่รองรับตัวเลข
8. หากเปิดแหล่งข้อมูลไม่ได้หรือข้อมูลไม่พอ ให้ระบุข้อจำกัด
9. ห้ามเดาตัวเลข

ตอบเป็นตาราง ตามด้วยสูตร การคำนวณ และข้อสรุป
ต้องค้นเว็บและเปิดอ่านแหล่งข้อมูลจริงก่อนตอบ
""".strip()

print(native_web_prompt)

ค้นเว็บเพื่อเปรียบเทียบรายได้รวมของ Microsoft
สำหรับปีงบประมาณ 2024 และ 2023

ขั้นตอนที่ต้องทำ:
- ค้นหาแหล่งข้อมูลทางการของ Microsoft
- เปิดอ่าน URL ที่พบจากการค้นหาอย่างน้อยหนึ่งหน้า
- ตรวจตัวเลขจากเนื้อหาที่เปิดอ่าน ไม่ใช้เพียงข้อความย่อผลค้นหา

ข้อกำหนด:
1. ใช้รายได้รวมทั้งปี ไม่ใช้รายไตรมาสหรือราย segment
2. แสดงตัวเลขเป็นล้านดอลลาร์สหรัฐ
3. ระบุวันสิ้นสุดปีงบประมาณของแต่ละปี
4. คำนวณอัตราการเติบโต พร้อมแสดงสูตรและการแทนค่า
5. แสดงอัตราการเติบโตทศนิยมสองตำแหน่ง
6. สรุปภาษาไทยสั้น ๆ ว่ารายได้เปลี่ยนแปลงอย่างไร
7. อ้างอิงแหล่งข้อมูลที่รองรับตัวเลข
8. หากเปิดแหล่งข้อมูลไม่ได้หรือข้อมูลไม่พอ ให้ระบุข้อจำกัด
9. ห้ามเดาตัวเลข

ตอบเป็นตาราง ตามด้วยสูตร การคำนวณ และข้อสรุป
ต้องค้นเว็บและเปิดอ่านแหล่งข้อมูลจริงก่อนตอบ


In [85]:
native_web_run_id = uuid4().hex
native_web_started_at = datetime.now(timezone.utc).isoformat()

native_web_result = None
native_web_error = None
native_web_status = "running"

started_at = time.perf_counter()

try:
    native_web_result = await asyncio.wait_for(
        native_web_agent.ainvoke(
            {
                "messages": [
                    {"role": "user", "content": native_web_prompt}
                ]
            },
            config={"recursion_limit": 30},
        ),
        timeout=180,
    )
    native_web_status = "completed"

except Exception as exc:
    native_web_status = "failed"
    native_web_error = f"{type(exc).__name__}: {exc}"

finally:
    native_web_latency_s = time.perf_counter() - started_at

print("Runtime status:", native_web_status)
print("Latency:", round(native_web_latency_s, 2), "seconds")

if native_web_error:
    print("Error:", native_web_error)

Runtime status: completed
Latency: 10.53 seconds


/usr/local/lib/python3.12/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `literal['url']` - serialized value may not be as expected [field_name='type', input_value='api', input_type=str])
  PydanticSerializationUnexpectedValue(Expected `ActionOpenPage` - serialized value may not be as expected [field_name='action', input_value=ActionSearch(type='search...name='oai-calculator')]), input_type=ActionSearch])
  PydanticSerializationUnexpectedValue(Expected `ActionFind` - serialized value may not be as expected [field_name='action', input_value=ActionSearch(type='search...name='oai-calculator')]), input_type=ActionSearch])
  return self.__pydantic_serializer__.to_python(


In [86]:
native_messages = (
    native_web_result.get("messages", [])
    if native_web_result is not None
    else []
)

native_last = native_messages[-1] if native_messages else None

native_web_answer = (
    native_last.text.strip()
    if isinstance(native_last, AIMessage)
    else ""
)

native_search_calls = []
native_citations = []
native_usage = []
native_other_calls = []
native_invalid_calls = []

for message in native_messages:
    if not isinstance(message, AIMessage):
        continue

    native_usage.append(message.usage_metadata)
    native_other_calls.extend(message.tool_calls)
    native_invalid_calls.extend(message.invalid_tool_calls)

    for block in message.content_blocks:
        if (
            block.get("type") == "server_tool_call"
            and block.get("name") == "web_search"
        ):
            native_search_calls.append(block)

        if block.get("type") == "text":
            native_citations.extend(
                annotation
                for annotation in block.get("annotations", [])
                if annotation.get("url")
            )

native_actions = Counter(
    call.get("args", {}).get("type", "unknown")
    for call in native_search_calls
)

native_opened_urls = [
    call.get("args", {}).get("url")
    for call in native_search_calls
    if call.get("args", {}).get("type") == "open_page"
]

native_checks = {
    "runtime_completed": native_web_status == "completed",
    "web_search_observed": native_actions["search"] > 0,
    "web_fetch_observed": native_actions["open_page"] > 0,
    "citations_observed": bool(native_citations),
    "no_other_tool_calls": not native_other_calls,
    "no_invalid_tool_calls": not native_invalid_calls,
    "final_answer_complete": (
        isinstance(native_last, AIMessage)
        and bool(native_web_answer)
        and not native_last.tool_calls
    ),
}

native_workflow_passed = all(native_checks.values())

native_web_report = {
    "case_id": "web_native_001",
    "case_version": "2.0",
    "run_id": native_web_run_id,
    "started_at_utc": native_web_started_at,
    "sdk": "deepagents",
    "model": native_web_model.model_name,
    "versions": deepcopy(web_versions),
    "search_provider": "openai",
    "search_mode": "provider_native",
    "fetch_mode": "provider_native_open_page",
    "runtime_status": native_web_status,
    "error": native_web_error,
    "latency_s": round(native_web_latency_s, 2),
    "checks": native_checks,
    "workflow_passed": native_workflow_passed,
    "web_search_observed": native_checks["web_search_observed"],
    "web_fetch_observed": native_checks["web_fetch_observed"],
    "action_counts": dict(native_actions),
    "opened_urls": native_opened_urls,
    "system_prompt": native_web_system_prompt,
    "prompt": native_web_prompt,
    "answer": native_web_answer,
    "tool_trace": deepcopy(native_search_calls),
    "citations": deepcopy(native_citations),
    "usage_per_visible_ai_message": deepcopy(native_usage),
    "review": None,
    "task_passed": None if native_workflow_passed else False,
}

display(Markdown(native_web_answer or "*ไม่มีคำตอบสุดท้าย*"))

display(Markdown(f"""
| รายการ | ผล |
|---|---|
| Case | web_native_001 |
| เวลารวม | {native_web_latency_s:.2f} วินาที |
| Search actions | {native_actions["search"]} |
| Open-page actions | {native_actions["open_page"]} |
| ผ่านเงื่อนไขการทำงาน | {native_workflow_passed} |
| ผลรวม | {"รอตรวจคำตอบและหลักฐาน" if native_workflow_passed else "ไม่ผ่านเงื่อนไขการทำงาน"} |
"""))

print("Opened URLs:")
for url in native_opened_urls:
    print("-", url)

print("\nCHECKS:")
print(json.dumps(native_checks, indent=2, ensure_ascii=False))

| ปีงบประมาณ | วันสิ้นสุดปีงบประมาณ | รายได้รวม (ล้านดอลลาร์สหรัฐ) |
|---|---:|---:|
| 2023 | 30 มิถุนายน 2023 | 211,915 |
| 2024 | 30 มิถุนายน 2024 | 245,122 |

ตัวเลขรายได้เป็นรายได้รวมทั้งปี ไม่ใช่รายได้รายไตรมาสหรือราย segment โดยรายงานประจำปีของ Microsoft ระบุหน่วยเป็นล้านดอลลาร์ และแสดงรายได้ของปีงบประมาณ 2024 และ 2023 ตามลำดับ ([microsoft.com](https://www.microsoft.com/investor/reports/ar24/index.html))

**สูตรอัตราการเติบโต**

\[
\text{อัตราการเติบโต}
=\frac{\text{รายได้ปี 2024}-\text{รายได้ปี 2023}}{\text{รายได้ปี 2023}}\times 100
\]

**แทนค่า**

\[
\frac{245{,}122-211{,}915}{211{,}915}\times 100
=15.67\%
\]

**สรุป:** รายได้รวมของ Microsoft เพิ่มขึ้น **33,207 ล้านดอลลาร์สหรัฐ** หรือ **15.67%** ในปีงบประมาณ 2024 เมื่อเทียบกับปีงบประมาณ 2023. ([microsoft.com](https://www.microsoft.com/investor/reports/ar24/index.html))


| รายการ | ผล |
|---|---|
| Case | web_native_001 |
| เวลารวม | 10.53 วินาที |
| Search actions | 2 |
| Open-page actions | 1 |
| ผ่านเงื่อนไขการทำงาน | True |
| ผลรวม | รอตรวจคำตอบและหลักฐาน |


Opened URLs:
- https://www.microsoft.com/investor/reports/ar24/index.html

CHECKS:
{
  "runtime_completed": true,
  "web_search_observed": true,
  "web_fetch_observed": true,
  "citations_observed": true,
  "no_other_tool_calls": true,
  "no_invalid_tool_calls": true,
  "final_answer_complete": true
}


### Websearch02 (เปรียบเทียบอ้างอิง)

เปรียบเทียบ Microsoft กับ Alphabet FY2024
โดยค้นเว็บและเปิดอ่านแหล่งข้อมูลของทั้งสองบริษัทก่อนตอบ

In [92]:
web_002_prompt = """
ค้นเว็บเพื่อเปรียบเทียบ Microsoft กับ Alphabet
โดยใช้ผลประกอบการรวมทั้งปีงบประมาณ 2024 ของแต่ละบริษัท

ขั้นตอนที่ต้องทำ:
- ค้นหาแหล่งข้อมูลทางการของบริษัท หรือเอกสารที่ยื่นต่อ SEC
- เปิดอ่านแหล่งข้อมูลของแต่ละบริษัท อย่างน้อยบริษัทละหนึ่ง URL
  รวมอย่างน้อยสอง URL ที่แตกต่างกัน
- ตรวจตัวเลขจากเนื้อหาที่เปิดอ่าน ไม่ใช้เพียงข้อความย่อผลค้นหา

สิ่งที่ต้องหา:
1. วันเริ่มต้นและวันสิ้นสุดปีงบประมาณ
2. รายได้รวม
3. กำไรจากการดำเนินงาน (Operating income)
4. Operating margin ที่คำนวณจากตัวเลขข้างต้น

ข้อกำหนด:
- ใช้ข้อมูล GAAP จากงบรวมทั้งปี ไม่ใช้รายไตรมาสหรือราย segment
- ใช้แหล่งข้อมูลทางการของบริษัท หรือเอกสารที่บริษัทยื่นต่อ SEC
- แสดงจำนวนเงินเป็นล้านดอลลาร์สหรัฐ
- คำนวณ Operating margin เป็นเปอร์เซ็นต์ ทศนิยมสองตำแหน่ง
- แสดงสูตรและการแทนค่าของแต่ละบริษัท
- อ้างอิงให้ชัดว่าแต่ละแหล่งรองรับบริษัทใด
- ตรวจว่าช่วงปีงบประมาณตรงกันหรือไม่ และอธิบายผลต่อการเปรียบเทียบ
- สรุปว่าบริษัทใดมี Operating margin สูงกว่าในงบที่นำมาเทียบ
- ไม่สรุปว่าบริษัทนั้นดีกว่าทุกด้านจากตัวชี้วัดนี้เพียงตัวเดียว
- หากเปิดแหล่งข้อมูลไม่ได้หรือข้อมูลไม่พอ ให้ระบุข้อจำกัด
- ห้ามเดาตัวเลข

ตอบภาษาไทย ประกอบด้วย:
1. ตารางเปรียบเทียบ
2. สูตรและการคำนวณ
3. ข้อสรุปและข้อจำกัดสั้น ๆ

ต้องค้นเว็บและเปิดอ่านแหล่งข้อมูลจริงก่อนตอบ
""".strip()

print(web_002_prompt)

ค้นเว็บเพื่อเปรียบเทียบ Microsoft กับ Alphabet
โดยใช้ผลประกอบการรวมทั้งปีงบประมาณ 2024 ของแต่ละบริษัท

ขั้นตอนที่ต้องทำ:
- ค้นหาแหล่งข้อมูลทางการของบริษัท หรือเอกสารที่ยื่นต่อ SEC
- เปิดอ่านแหล่งข้อมูลของแต่ละบริษัท อย่างน้อยบริษัทละหนึ่ง URL
  รวมอย่างน้อยสอง URL ที่แตกต่างกัน
- ตรวจตัวเลขจากเนื้อหาที่เปิดอ่าน ไม่ใช้เพียงข้อความย่อผลค้นหา

สิ่งที่ต้องหา:
1. วันเริ่มต้นและวันสิ้นสุดปีงบประมาณ
2. รายได้รวม
3. กำไรจากการดำเนินงาน (Operating income)
4. Operating margin ที่คำนวณจากตัวเลขข้างต้น

ข้อกำหนด:
- ใช้ข้อมูล GAAP จากงบรวมทั้งปี ไม่ใช้รายไตรมาสหรือราย segment
- ใช้แหล่งข้อมูลทางการของบริษัท หรือเอกสารที่บริษัทยื่นต่อ SEC
- แสดงจำนวนเงินเป็นล้านดอลลาร์สหรัฐ
- คำนวณ Operating margin เป็นเปอร์เซ็นต์ ทศนิยมสองตำแหน่ง
- แสดงสูตรและการแทนค่าของแต่ละบริษัท
- อ้างอิงให้ชัดว่าแต่ละแหล่งรองรับบริษัทใด
- ตรวจว่าช่วงปีงบประมาณตรงกันหรือไม่ และอธิบายผลต่อการเปรียบเทียบ
- สรุปว่าบริษัทใดมี Operating margin สูงกว่าในงบที่นำมาเทียบ
- ไม่สรุปว่าบริษัทนั้นดีกว่าทุกด้านจากตัวชี้วัดนี้เพียงตัวเดียว
- 

In [95]:
web_002_run_id = uuid4().hex
web_002_started_at = datetime.now(timezone.utc).isoformat()

web_002_result = None
web_002_error = None
web_002_status = "running"

started_at = time.perf_counter()

try:
    web_002_result = await asyncio.wait_for(
        native_web_agent.ainvoke(
            {
                "messages": [
                    {"role": "user", "content": web_002_prompt}
                ]
            },
            config={"recursion_limit": 30},
        ),
        timeout=180,
    )
    web_002_status = "completed"

except Exception as exc:
    web_002_status = "failed"
    web_002_error = f"{type(exc).__name__}: {exc}"

finally:
    web_002_latency_s = time.perf_counter() - started_at

print("Runtime status:", web_002_status)
print("Latency:", round(web_002_latency_s, 2), "seconds")

if web_002_error:
    print("Error:", web_002_error)

Runtime status: completed
Latency: 16.84 seconds


/usr/local/lib/python3.12/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `literal['url']` - serialized value may not be as expected [field_name='type', input_value='api', input_type=str])
  PydanticSerializationUnexpectedValue(Expected `ActionOpenPage` - serialized value may not be as expected [field_name='action', input_value=ActionSearch(type='search...name='oai-calculator')]), input_type=ActionSearch])
  PydanticSerializationUnexpectedValue(Expected `ActionFind` - serialized value may not be as expected [field_name='action', input_value=ActionSearch(type='search...name='oai-calculator')]), input_type=ActionSearch])
  return self.__pydantic_serializer__.to_python(


In [96]:
from urllib.parse import urlsplit

web_002_messages = (
    web_002_result.get("messages", [])
    if web_002_result is not None
    else []
)

web_002_last = web_002_messages[-1] if web_002_messages else None

web_002_answer = (
    web_002_last.text.strip()
    if isinstance(web_002_last, AIMessage)
    else ""
)

web_002_calls = []
web_002_citations = []
web_002_usage = []
web_002_other_calls = []
web_002_invalid_calls = []

for message in web_002_messages:
    if not isinstance(message, AIMessage):
        continue

    web_002_usage.append(message.usage_metadata)
    web_002_other_calls.extend(message.tool_calls)
    web_002_invalid_calls.extend(message.invalid_tool_calls)

    for block in message.content_blocks:
        if (
            block.get("type") == "server_tool_call"
            and block.get("name") == "web_search"
        ):
            web_002_calls.append(block)

        if block.get("type") == "text":
            web_002_citations.extend(
                annotation
                for annotation in block.get("annotations", [])
                if annotation.get("url")
            )

web_002_actions = Counter(
    call.get("args", {}).get("type", "unknown")
    for call in web_002_calls
)

web_002_opened_urls = [
    call.get("args", {}).get("url")
    for call in web_002_calls
    if call.get("args", {}).get("type") == "open_page"
]

# ไม่นับหน้าเดิมเป็นหลายหน้าเพียงเพราะ tracking query ต่างกัน
web_002_unique_pages = set()

for url in web_002_opened_urls:
    if not isinstance(url, str):
        continue

    try:
        parsed = urlsplit(url)
        if parsed.scheme in {"http", "https"} and parsed.hostname:
            web_002_unique_pages.add(
                (
                    parsed.hostname.lower(),
                    parsed.path.rstrip("/") or "/",
                )
            )
    except ValueError:
        continue

web_002_sources = {
    citation["url"]: citation.get("title") or citation["url"]
    for citation in web_002_citations
}

web_002_checks = {
    "runtime_completed": web_002_status == "completed",
    "web_search_observed": web_002_actions["search"] > 0,
    "web_fetch_observed": web_002_actions["open_page"] > 0,
    "at_least_two_distinct_pages_opened": (
        len(web_002_unique_pages) >= 2
    ),
    "citations_observed": bool(web_002_citations),
    "no_other_tool_calls": not web_002_other_calls,
    "no_invalid_tool_calls": not web_002_invalid_calls,
    "final_answer_complete": (
        isinstance(web_002_last, AIMessage)
        and bool(web_002_answer)
        and not web_002_last.tool_calls
    ),
}

web_002_workflow_passed = all(web_002_checks.values())

web_002_report = {
    "case_id": "web_native_002",
    "case_version": "2.0",
    "run_id": web_002_run_id,
    "started_at_utc": web_002_started_at,
    "sdk": "deepagents",
    "model": native_web_model.model_name,
    "versions": deepcopy(web_versions),
    "search_provider": "openai",
    "search_mode": "provider_native",
    "fetch_mode": "provider_native_open_page",
    "runtime_status": web_002_status,
    "error": web_002_error,
    "latency_s": round(web_002_latency_s, 2),
    "checks": web_002_checks,
    "workflow_passed": web_002_workflow_passed,
    "web_search_observed": web_002_checks["web_search_observed"],
    "web_fetch_observed": web_002_checks["web_fetch_observed"],
    "action_counts": dict(web_002_actions),
    "opened_urls": web_002_opened_urls,
    "unique_citation_urls": len(web_002_sources),
    "system_prompt": native_web_system_prompt,
    "prompt": web_002_prompt,
    "answer": web_002_answer,
    "tool_trace": deepcopy(web_002_calls),
    "citations": deepcopy(web_002_citations),
    "usage_per_visible_ai_message": deepcopy(web_002_usage),
    "review": None,
    "task_passed": None if web_002_workflow_passed else False,
}

display(Markdown(web_002_answer or "*ไม่มีคำตอบสุดท้าย*"))

display(Markdown(f"""
| รายการ | ผล |
|---|---|
| Case | web_native_002 |
| เวลารวม | {web_002_latency_s:.2f} วินาที |
| Search actions | {web_002_actions["search"]} |
| Open-page actions | {web_002_actions["open_page"]} |
| หน้าเว็บที่เปิดไม่ซ้ำ | {len(web_002_unique_pages)} |
| ผ่านเงื่อนไขการทำงาน | {web_002_workflow_passed} |
| ผลรวม | {"รอตรวจคำตอบและหลักฐาน" if web_002_workflow_passed else "ไม่ผ่านเงื่อนไขการทำงาน"} |
"""))

print("Opened URLs:")
for url in web_002_opened_urls:
    print("-", url)

print("\nCHECKS:")
print(json.dumps(web_002_checks, indent=2, ensure_ascii=False))

display(Markdown("**แหล่งอ้างอิง**"))

for url, title in web_002_sources.items():
    display(Markdown(f"- [{title}]({url})"))

## ตารางเปรียบเทียบผลประกอบการปี 2024

| บริษัท | ช่วงปีงบประมาณ | รายได้รวม (ล้านดอลลาร์สหรัฐ) | กำไรจากการดำเนินงาน (ล้านดอลลาร์สหรัฐ) | Operating margin |
|---|---|---:|---:|---:|
| Microsoft | 1 กรกฎาคม 2023–30 มิถุนายน 2024 | 245,122 | 109,433 | **44.64%** |
| Alphabet | 1 มกราคม–31 ธันวาคม 2024 | 350,018 | 112,390 | **32.11%** |

ตัวเลขของ Microsoft มาจากผลทั้ง 12 เดือนสิ้นสุดวันที่ 30 มิถุนายน 2024 ซึ่งเอกสารของบริษัทระบุว่าเป็น GAAP; ตัวเลข Alphabet มาจากงบรวมทั้งปีสิ้นสุดวันที่ 31 ธันวาคม 2024 ซึ่งแสดงรายได้รวมและกำไรจากการดำเนินงานในงบกำไรขาดทุนรวม ([microsoft.gcs-web.com](https://microsoft.gcs-web.com/static-files/5876595f-cd7a-47ac-a2c3-b3b6f9099452))

## สูตรและการคำนวณ

**Operating margin = (กำไรจากการดำเนินงาน ÷ รายได้รวม) × 100**

- **Microsoft:** (109,433 ÷ 245,122) × 100 = **44.64%** ([microsoft.gcs-web.com](https://microsoft.gcs-web.com/static-files/5876595f-cd7a-47ac-a2c3-b3b6f9099452))
- **Alphabet:** (112,390 ÷ 350,018) × 100 = **32.11%** ([sec.gov](https://www.sec.gov/Archives/edgar/data/1652044/000165204425000014/goog-20241231.htm))

## ข้อสรุปและข้อจำกัด

Microsoft มี Operating margin สูงกว่า Alphabet ในงบปี 2024 ที่นำมาเทียบประมาณ **12.53 จุดเปอร์เซ็นต์** อย่างไรก็ตาม ช่วงปีงบประมาณ **ไม่ตรงกัน**: ปีของ Microsoft สิ้นสุดวันที่ 30 มิถุนายน ส่วน Alphabet สิ้นสุดวันที่ 31 ธันวาคม 2024 จึงครอบคลุมช่วงเวลาต่างกัน แม้ทั้งคู่เป็นผลรวมทั้งปีงบประมาณ 2024 การเปรียบเทียบนี้บอกได้เฉพาะอัตรากำไรจากการดำเนินงานในงบที่นำมาเทียบ ไม่ได้หมายความว่าบริษัทหนึ่งดีกว่าอีกบริษัทในทุกด้าน.


| รายการ | ผล |
|---|---|
| Case | web_native_002 |
| เวลารวม | 16.84 วินาที |
| Search actions | 3 |
| Open-page actions | 3 |
| หน้าเว็บที่เปิดไม่ซ้ำ | 2 |
| ผ่านเงื่อนไขการทำงาน | True |
| ผลรวม | รอตรวจคำตอบและหลักฐาน |


Opened URLs:
- https://www.sec.gov/Archives/edgar/data/789019/000095017024087843/msft-20240630.htm
- https://www.sec.gov/Archives/edgar/data/789019/000095017024087843/msft-20240630.htm
- https://www.microsoft.com/investor/reports/ar24/?msockid=0bd6ea2b337269252ec0fceb326b6870

CHECKS:
{
  "runtime_completed": true,
  "web_search_observed": true,
  "web_fetch_observed": true,
  "at_least_two_distinct_pages_opened": true,
  "citations_observed": true,
  "no_other_tool_calls": true,
  "no_invalid_tool_calls": true,
  "final_answer_complete": true
}


**แหล่งอ้างอิง**

- [Form 8-K for Microsoft Corp filed 07/30/2024](https://microsoft.gcs-web.com/static-files/5876595f-cd7a-47ac-a2c3-b3b6f9099452)

- [goog-20241231](https://www.sec.gov/Archives/edgar/data/1652044/000165204425000014/goog-20241231.htm)

In [97]:
# ผลตรวจเฉพาะรอบที่ส่งมา:
# เปิด Microsoft แล้ว แต่ไม่พบการเปิดแหล่งข้อมูล Alphabet

web_002_report["checks"]["opened_sources_for_each_company"] = False
web_002_report["workflow_passed"] = False
web_002_report["task_passed"] = False

web_002_report["review"] = {
    "method": "manual_review_of_shared_trace",
    "run_id": web_002_report["run_id"],
    "notes": [
        "พบการเปิดแหล่งข้อมูล Microsoft",
        "ไม่พบ open_page ของ Alphabet ใน trace ที่ส่งมาตรวจ",
    ],
}

print("Task passed:", web_002_report["task_passed"])

Task passed: False


### บันทึกผลการทดสอบ Websearch

In [98]:
results_dir = Path("/workspace/shared/results/deepagents")

assert Path("/workspace/shared").is_dir(), (
    "Container ยังมองไม่เห็น /workspace/shared"
)

results_dir.mkdir(parents=True, exist_ok=True)

# ใช้ผลและคะแนนของแต่ละ report ตามจริง
# ไม่ใส่คะแนน manual review ของการรันเมื่อวานทับรอบใหม่
web_001_saved = deepcopy(native_web_report)
web_002_saved = deepcopy(web_002_report)

saved_at = datetime.now(timezone.utc)

web_checkpoint = {
    "schema_version": "1.0",
    "saved_at_utc": saved_at.isoformat(),
    "sdk": "deepagents",
    "category": "web_native",
    "case_version": "2.0",
    "results": [
        web_001_saved,
        web_002_saved,
    ],
}

web_output_path = results_dir / (
    f"web_native_v2_{saved_at:%Y%m%dT%H%M%SZ}_{uuid4().hex[:8]}.json"
)

payload = json.dumps(
    web_checkpoint,
    indent=2,
    ensure_ascii=False,
    allow_nan=False,
)

with web_output_path.open("x", encoding="utf-8") as file:
    file.write(payload)

verified = json.loads(
    web_output_path.read_text(encoding="utf-8")
)

assert verified == web_checkpoint, "ข้อมูลที่อ่านกลับไม่ตรงกับที่บันทึก"

print("Saved:", web_output_path)
print("Experiments:", len(verified["results"]))
print("JSON file verified.")

for report in verified["results"]:
    status = (
        "passed" if report["task_passed"] is True
        else "failed" if report["task_passed"] is False
        else "pending_review"
    )

    print(
        f"- {report['case_id']}: "
        f"search={report['web_search_observed']}, "
        f"fetch={report['web_fetch_observed']}, "
        f"status={status}"
    )

Saved: /workspace/shared/results/deepagents/web_native_v2_20261006T084731Z_e5920ef4.json
Experiments: 2
JSON file verified.
- web_native_001: search=True, fetch=True, status=pending_review
- web_native_002: search=True, fetch=True, status=failed


<hr>

# 5 Planning

ทดสอบงานที่ต้องวางแผนและทำหลายขั้นตอน โดยกำหนดเครื่องมือและข้อกำหนดใน prompt จากนั้นบันทึกผลการทำงาน ตรวจลำดับการเรียกเครื่องมือ และประเมินว่าทำงานครบตามเป้าหมายหรือไม่

### เปรียบเทียบกรณีการวางแผน

กลุ่มเซลล์นี้มีกรณีทดลองมากกว่าหนึ่งแบบ เพื่อดูการวางแผน การลงมือทำ และการตรวจผลลัพธ์ภายใต้ข้อกำหนดที่ต่างกัน


In [100]:
import asyncio
import json
import time

from copy import deepcopy
from typing import Literal
from uuid import uuid4
from importlib.metadata import version

from deepagents import create_deep_agent
from langchain.tools import tool
from langchain.agents.middleware import TodoListMiddleware
from langchain_core.messages import AIMessage, ToolMessage
from IPython.display import display, Markdown

In [101]:
PLANNING_DATA = {
    "co-d81": {
        "company_id": "co-d81",
        "company_name": "Delta",
        "year": 2025,
        "previous_year": 2024,
        "unit": "million_thb",
        "revenue_previous": 1000,
        "revenue_current": 1300,
        "net_profit_current": 156,
        "liabilities": 540,
        "assets": 1800,
        "source_id": "fixture_delta_v1",
    },
    "co-e42": {
        "company_id": "co-e42",
        "company_name": "Echo",
        "year": 2025,
        "previous_year": 2024,
        "unit": "million_thb",
        "revenue_previous": 800,
        "revenue_current": 960,
        "net_profit_current": 192,
        "liabilities": 900,
        "assets": 1500,
        "source_id": "fixture_echo_v1",
    },
    "co-f63": {
        "company_id": "co-f63",
        "company_name": "Foxtrot",
        "year": 2025,
        "previous_year": 2024,
        "unit": "million_thb",
        "revenue_previous": 1500,
        "revenue_current": 1800,
        "net_profit_current": None,
        "liabilities": 400,
        "assets": 2000,
        "source_id": "fixture_foxtrot_v1",
    },
}

print("Dataset ready:", len(PLANNING_DATA), "companies")

Dataset ready: 3 companies


### Planning01

In [102]:
@tool
def list_companies() -> list[dict]:
    """List available companies with their IDs and names."""
    return [
        {
            "company_id": row["company_id"],
            "company_name": row["company_name"],
        }
        for row in PLANNING_DATA.values()
    ]


@tool
def read_company_financials(company_id: str) -> dict:
    """Read a company's annual financial data.

    A null value means the data is unavailable.
    """
    row = PLANNING_DATA.get(company_id)

    if row is None:
        return {"error": "COMPANY_NOT_FOUND"}

    return deepcopy(row)


@tool
def calculate_percentage(
    numerator: float,
    denominator: float,
    mode: Literal["ratio", "growth"],
) -> dict:
    """Calculate a percentage.

    ratio: numerator / denominator * 100.
    growth: (numerator - denominator) / denominator * 100;
            numerator is the current value, denominator the previous value.
    """
    if denominator == 0:
        return {"value_pct": None, "error": "ZERO_DENOMINATOR"}

    if mode == "ratio":
        value = numerator / denominator * 100
    elif mode == "growth":
        value = (numerator - denominator) / denominator * 100
    else:
        raise ValueError("Unsupported mode")

    return {"value_pct": round(value, 2)}


print("Business tools ready.")

Business tools ready.


In [103]:
planning_prompt = """
เปรียบเทียบบริษัททั้งหมดที่มีในระบบ โดยใช้ข้อมูลปี 2025
ในสามด้าน:
- การเติบโตของรายได้เทียบปีก่อน
- อัตรากำไรสุทธิ
- หนี้สินรวมต่อสินทรัพย์รวม

ระบุว่าบริษัทใดมีข้อมูลครบสำหรับคำนวณทั้งสามตัวชี้วัด
และบริษัทใดยังขาดข้อมูลอะไร

ข้อกำหนด:
- สร้างแผนงานของคุณผ่าน write_todos ก่อนเริ่มเรียก tools ดึงข้อมูล
- เลือกงานย่อยและลำดับการทำงานเอง
- อัปเดตสถานะ To-do ระหว่างทำงานให้ตรงกับสิ่งที่ทำจริง
- ใช้ calculate_percentage สำหรับการคำนวณ
- ข้อมูลขาดให้แสดง N/A และอธิบาย ห้ามสมมติตัวเลข
- ข้อมูลครบ หมายถึงมีข้อมูลสำหรับคำนวณทั้งสามตัวชี้วัด
- อ้างอิง source_id สำหรับข้อมูลของแต่ละบริษัท
- ตอบภาษาไทยเป็นตาราง พร้อมข้อสรุปสั้น ๆ
"""

planning_agent = create_deep_agent(
    model=model,
    tools=[
        list_companies,
        read_company_financials,
        calculate_percentage,
    ],
    middleware=[TodoListMiddleware()],
    system_prompt=(
        "Complete the user's analysis using the available tools. "
        "Maintain an actionable task list reflecting actual progress. "
        "Use only the supplied financial data. "
        "Treat missing values as unavailable, not as zero. "
        "For this experiment, perform the work yourself without delegation."
    ),
)

print("Planning agent ready.")

Planning agent ready.


In [104]:
planning_run_id = uuid4().hex

planning_versions = {
    name: version(name)
    for name in ["deepagents", "langchain", "langchain-openai"]
}

planning_result = None
planning_error = None
planning_status = "running"

started_at = time.perf_counter()

try:
    planning_result = await asyncio.wait_for(
        planning_agent.ainvoke(
            {
                "messages": [
                    {"role": "user", "content": planning_prompt}
                ]
            },
            config={"recursion_limit": 80},
        ),
        timeout=180,
    )
    planning_status = "completed"

except Exception as exc:
    planning_status = "failed"
    planning_error = f"{type(exc).__name__}: {exc}"

finally:
    planning_latency_s = time.perf_counter() - started_at

print("Runtime status:", planning_status)
print("Latency:", round(planning_latency_s, 2), "seconds")

if planning_error:
    print("Error:", planning_error)

Runtime status: completed
Latency: 31.92 seconds


In [105]:
assert planning_result is not None, "ตรวจ error จาก Cell 4 ก่อน"

planning_messages = planning_result["messages"]
planning_last = planning_messages[-1]

assert isinstance(planning_last, AIMessage)

planning_answer = planning_last.text.strip()
planning_calls = []
planning_trace = []
planning_usage = []

for index, message in enumerate(planning_messages):
    event = {
        "message_index": index,
        "type": message.type,
        "content": deepcopy(message.content),
    }

    if isinstance(message, AIMessage):
        event["tool_calls"] = deepcopy(message.tool_calls)
        planning_usage.append(message.usage_metadata)

        for call in message.tool_calls:
            planning_calls.append({
                "message_index": index,
                **deepcopy(call),
            })

    elif isinstance(message, ToolMessage):
        event["tool_call_id"] = message.tool_call_id
        event["name"] = message.name
        event["status"] = message.status

    planning_trace.append(event)

planning_updates = [
    call for call in planning_calls
    if call["name"] == "write_todos"
]

display(Markdown("**คำตอบสุดท้าย**"))
display(Markdown(planning_answer))

display(Markdown("**แผนและการอัปเดตระหว่างทำงาน**"))

for index, call in enumerate(planning_updates, start=1):
    print(f"\nPlan update {index} — message {call['message_index']}")

    for item in call["args"].get("todos", []):
        print(f"- [{item['status']}] {item['content']}")

display(Markdown("**ลำดับการเรียกเครื่องมือ**"))

for call in planning_calls:
    print(f"Message {call['message_index']}: {call['name']}")

**คำตอบสุดท้าย**

## เปรียบเทียบบริษัทในระบบ — ข้อมูลปี 2025

| บริษัท | รายได้เติบโตเทียบปี 2024 | อัตรากำไรสุทธิ | หนี้สินรวมต่อสินทรัพย์รวม | ข้อมูลครบทั้ง 3 ตัวชี้วัด? | ข้อมูลที่ขาด | source_id |
|---|---:|---:|---:|---|---|---|
| Delta | 30% | 12% | 30% | ครบ | — | `fixture_delta_v1` |
| Echo | 20% | 20% | 60% | ครบ | — | `fixture_echo_v1` |
| Foxtrot | 20% | N/A | 20% | ไม่ครบ | กำไรสุทธิปี 2025 ไม่มีข้อมูล จึงคำนวณอัตรากำไรสุทธิไม่ได้ | `fixture_foxtrot_v1` |

**สรุป:** Delta และ Echo มีข้อมูลครบสำหรับคำนวณทั้งสามตัวชี้วัด ส่วน Foxtrot ขาดข้อมูลกำไรสุทธิปี 2025 จึงแสดงอัตรากำไรสุทธิเป็น N/A โดยไม่สมมติตัวเลข ค่าทั้งหมดแสดงเป็นร้อยละ

**แผนและการอัปเดตระหว่างทำงาน**


Plan update 1 — message 1
- [in_progress] ดึงรายชื่อบริษัททั้งหมดในระบบ
- [pending] อ่านข้อมูลการเงินปี 2025 และปีก่อนที่จำเป็นของทุกบริษัท
- [pending] คำนวณตัวชี้วัดที่คำนวณได้ด้วย calculate_percentage และตรวจข้อมูลที่ขาด
- [pending] สรุปผลเป็นตารางภาษาไทยพร้อม source_id และข้อสรุป

Plan update 2 — message 5
- [completed] ดึงรายชื่อบริษัททั้งหมดในระบบ
- [in_progress] อ่านข้อมูลการเงินปี 2025 และปีก่อนที่จำเป็นของทุกบริษัท
- [pending] คำนวณตัวชี้วัดที่คำนวณได้ด้วย calculate_percentage และตรวจข้อมูลที่ขาด
- [pending] สรุปผลเป็นตารางภาษาไทยพร้อม source_id และข้อสรุป

Plan update 3 — message 11
- [completed] ดึงรายชื่อบริษัททั้งหมดในระบบ
- [completed] อ่านข้อมูลการเงินปี 2025 และปีก่อนที่จำเป็นของทุกบริษัท
- [in_progress] คำนวณตัวชี้วัดที่คำนวณได้ด้วย calculate_percentage และตรวจข้อมูลที่ขาด
- [pending] สรุปผลเป็นตารางภาษาไทยพร้อม source_id และข้อสรุป

Plan update 4 — message 22
- [completed] ดึงรายชื่อบริษัททั้งหมดในระบบ
- [completed] อ่านข้อมูลการเงินปี 2025 และปีก่อนที่จำเป็นของทุกบริ

**ลำดับการเรียกเครื่องมือ**

Message 1: write_todos
Message 3: list_companies
Message 5: write_todos
Message 7: read_company_financials
Message 7: read_company_financials
Message 7: read_company_financials
Message 11: write_todos
Message 13: calculate_percentage
Message 13: calculate_percentage
Message 13: calculate_percentage
Message 13: calculate_percentage
Message 13: calculate_percentage
Message 13: calculate_percentage
Message 13: calculate_percentage
Message 13: calculate_percentage
Message 22: write_todos
Message 24: write_todos


In [106]:
data_calls = [
    call for call in planning_calls
    if call["name"] in {"list_companies", "read_company_financials"}
]

plan_before_data = bool(planning_updates and data_calls) and (
    planning_updates[0]["message_index"]
    < data_calls[0]["message_index"]
)

different_plans = {
    json.dumps(call["args"].get("todos", []), sort_keys=True)
    for call in planning_updates
}

planning_observations = {
    "plan_created": bool(planning_updates),
    "plan_requested_before_data_calls": plan_before_data,
    "plan_changed_during_run": len(different_plans) > 1,
    "final_answer_present": (
        bool(planning_answer) and not planning_last.tool_calls
    ),
}

planning_report = {
    "case_id": "planning_001",
    "run_id": planning_run_id,
    "sdk": "deepagents",
    "model": model.model_name,
    "versions": planning_versions,
    "configuration": {
        "planning_tool": "TodoListMiddleware",
        "planning_explicitly_requested": True,
        "delegation_allowed_by_prompt": False,
    },
    "runtime_status": planning_status,
    "latency_s": round(planning_latency_s, 2),
    "observations": planning_observations,
    "tool_call_count": len(planning_calls),
    "plan_update_count": len(planning_updates),
    "final_todos": deepcopy(planning_result.get("todos", [])),
    "prompt": planning_prompt,
    "answer": planning_answer,
    "trace": planning_trace,
    "usage_per_visible_ai_message": planning_usage,
    "task_passed": None,
}

display(Markdown(f"""
| รายการ | ผล |
|---|---|
| Case | planning_001 |
| เวลารวม | {planning_latency_s:.2f} วินาที |
| สร้างแผน | {planning_observations["plan_created"]} |
| เรียกวางแผนก่อนเรียกดึงข้อมูล | {plan_before_data} |
| มีการเปลี่ยนแผนหรือสถานะงาน | {planning_observations["plan_changed_during_run"]} |
| จำนวนครั้งที่เขียน To-do | {len(planning_updates)} |
| Tool calls รวม | {len(planning_calls)} |
| ผลตรวจคุณภาพ | รอตรวจ |
"""))


| รายการ | ผล |
|---|---|
| Case | planning_001 |
| เวลารวม | 31.92 วินาที |
| สร้างแผน | True |
| เรียกวางแผนก่อนเรียกดึงข้อมูล | True |
| มีการเปลี่ยนแผนหรือสถานะงาน | True |
| จำนวนครั้งที่เขียน To-do | 5 |
| Tool calls รวม | 17 |
| ผลตรวจคุณภาพ | รอตรวจ |


In [107]:
# ผลประเมินนี้ใช้กับ planning_001 รอบที่ส่งมาตรวจแล้ว
planning_001_record = deepcopy(planning_report)
planning_001_record["task_passed"] = True
planning_001_record["review"] = {
    "method": "assistant_review_of_shared_output",
    "notes": [
        "Plan was created before data retrieval.",
        "Progress updates matched the observed work stages.",
        "All reported metrics and source IDs were correct.",
        "Missing Foxtrot net profit was correctly reported as unavailable.",
    ],
}

# ข้อมูลปีที่ต้องการ
echo_backup_2025 = deepcopy(PLANNING_DATA["co-e42"])
echo_backup_2025["source_id"] = "fixture_echo_backup_2025"

# ข้อมูลคนละปี เพื่อทดสอบการเลือกแหล่งให้ตรงโจทย์
echo_backup_2024 = {
    **deepcopy(PLANNING_DATA["co-e42"]),
    "year": 2024,
    "previous_year": 2023,
    "revenue_previous": 700,
    "revenue_current": 800,
    "net_profit_current": 128,
    "liabilities": 880,
    "assets": 1500,
    "source_id": "fixture_echo_backup_2024",
}

BACKUP_SOURCES = {
    "fixture_echo_backup_2024": echo_backup_2024,
    "fixture_echo_backup_2025": echo_backup_2025,
}

print("Planning 001 retained.")
print("Backup sources ready.")

Planning 001 retained.
Backup sources ready.


In [110]:
@tool
def read_primary_financials(company_id: str) -> dict:
    """Read annual financial data from the primary source."""
    if company_id == "co-e42":
        return {
            "company_id": company_id,
            "error": "PRIMARY_SOURCE_UNAVAILABLE",
            "retryable": False,
        }

    row = PLANNING_DATA.get(company_id)

    if row is None:
        return {"error": "COMPANY_NOT_FOUND"}

    return deepcopy(row)


@tool
def find_financial_sources(company_id: str) -> list[dict]:
    """Find alternative financial documents for a company.

    Check each document's reporting year before using it.
    """
    return [
        {
            "source_id": source_id,
            "company_id": row["company_id"],
            "reporting_year": row["year"],
            "document_type": "annual_financial_statement",
        }
        for source_id, row in BACKUP_SOURCES.items()
        if row["company_id"] == company_id
    ]


@tool
def read_financial_source(source_id: str) -> dict:
    """Read a financial document by its source ID."""
    row = BACKUP_SOURCES.get(source_id)

    if row is None:
        return {"error": "SOURCE_NOT_FOUND"}

    return deepcopy(row)


print("Primary and alternative-source tools ready.")

Primary and alternative-source tools ready.


### Planning02

In [111]:
planning_002_prompt = """
เปรียบเทียบบริษัททั้งหมดในระบบ โดยใช้ข้อมูลปี 2025
ในสามด้าน:
- การเติบโตของรายได้เทียบปีก่อน
- อัตรากำไรสุทธิ
- หนี้สินรวมต่อสินทรัพย์รวม

ระบุว่าบริษัทใดมีข้อมูลครบสำหรับคำนวณทั้งสามตัวชี้วัด
และบริษัทใดยังขาดข้อมูลอะไร

ข้อกำหนด:
- สร้างแผนผ่าน write_todos ก่อนเริ่มดึงข้อมูล
- เลือกงานย่อยและลำดับการทำงานเอง
- เริ่มตรวจข้อมูลจากแหล่งหลัก
- เมื่อพบอุปสรรค ให้ประเมินทางเลือกและอัปเดตแผนก่อนดำเนินทางเลือกนั้น
- ไม่เรียกซ้ำโดยไม่มีเหตุผลเมื่อ error ระบุ retryable เป็น false
- ตรวจบริษัทและปีของเอกสารให้ตรงกับงานก่อนนำตัวเลขมาใช้
- ใช้ calculate_percentage สำหรับการคำนวณ
- ข้อมูลขาดให้แสดง N/A และอธิบาย ห้ามสมมติตัวเลข
- อ้างอิง source_id ของข้อมูลที่ใช้จริง
- ตอบภาษาไทยเป็นตาราง พร้อมข้อสรุปสั้น ๆ
- อธิบายว่าพบอุปสรรคอะไรและแก้ไขอย่างไร
"""

In [112]:
planning_002_agent = create_deep_agent(
    model=model,
    tools=[
        list_companies,
        read_primary_financials,
        find_financial_sources,
        read_financial_source,
        calculate_percentage,
    ],
    middleware=[TodoListMiddleware()],
    system_prompt=(
        "Complete the analysis using the supplied financial tools. "
        "Maintain an actionable task list reflecting actual progress. "
        "Revise your plan when new evidence requires a different approach. "
        "Treat missing values as unavailable, not as zero. "
        "Perform the work yourself without delegation."
    ),
)

print("Replanning agent ready.")

Replanning agent ready.


In [113]:
planning_002_run_id = uuid4().hex
planning_002_result = None
planning_002_error = None
planning_002_status = "running"

started_at = time.perf_counter()

try:
    planning_002_result = await asyncio.wait_for(
        planning_002_agent.ainvoke(
            {
                "messages": [
                    {"role": "user", "content": planning_002_prompt}
                ]
            },
            config={"recursion_limit": 100},
        ),
        timeout=180,
    )
    planning_002_status = "completed"

except Exception as exc:
    planning_002_status = "failed"
    planning_002_error = f"{type(exc).__name__}: {exc}"

finally:
    planning_002_latency_s = time.perf_counter() - started_at

print("Runtime status:", planning_002_status)
print("Latency:", round(planning_002_latency_s, 2), "seconds")

if planning_002_error:
    print("Error:", planning_002_error)

Runtime status: completed
Latency: 50.93 seconds


In [114]:
assert planning_002_result is not None, "ตรวจ error จาก Cell 4 ก่อน"

messages_p2 = planning_002_result["messages"]
last_p2 = messages_p2[-1]

assert isinstance(last_p2, AIMessage)

answer_p2 = last_p2.text.strip()
calls_p2 = []
trace_p2 = []
usage_p2 = []

for index, message in enumerate(messages_p2):
    event = {
        "message_index": index,
        "type": message.type,
        "content": deepcopy(message.content),
    }

    if isinstance(message, AIMessage):
        event["tool_calls"] = deepcopy(message.tool_calls)
        usage_p2.append(message.usage_metadata)

        for call in message.tool_calls:
            calls_p2.append({
                "message_index": index,
                **deepcopy(call),
            })

    elif isinstance(message, ToolMessage):
        event["tool_call_id"] = message.tool_call_id
        event["name"] = message.name
        event["status"] = message.status

    trace_p2.append(event)

updates_p2 = [
    call for call in calls_p2
    if call["name"] == "write_todos"
]

planning_002_report = {
    "case_id": "planning_002",
    "run_id": planning_002_run_id,
    "sdk": "deepagents",
    "model": model.model_name,
    "versions": planning_versions,
    "fixture_version": "planning_backup_v1",
    "runtime_status": planning_002_status,
    "latency_s": round(planning_002_latency_s, 2),
    "tool_call_count": len(calls_p2),
    "plan_update_count": len(updates_p2),
    "prompt": planning_002_prompt,
    "answer": answer_p2,
    "trace": trace_p2,
    "usage_per_visible_ai_message": usage_p2,
    "final_todos": deepcopy(planning_002_result.get("todos", [])),
    "task_passed": None,
}

display(Markdown("**คำตอบสุดท้าย**"))
display(Markdown(answer_p2))

display(Markdown("**แผนและการปรับแผน**"))

for index, call in enumerate(updates_p2, start=1):
    lines = [
        f"- **{item['status']}** — {item['content']}"
        for item in call["args"].get("todos", [])
    ]
    display(Markdown(
        f"**แผนครั้งที่ {index} — Message {call['message_index']}**\n\n"
        + "\n".join(lines)
    ))

display(Markdown("**ลำดับเรียกเครื่องมือและ arguments**"))

for call in calls_p2:
    if call["name"] != "write_todos":
        print(
            f"Message {call['message_index']}: "
            f"{call['name']} "
            f"{json.dumps(call['args'], ensure_ascii=False)}"
        )

display(Markdown(f"""
| รายการ | ผล |
|---|---|
| Case | planning_002 |
| เวลารวม | {planning_002_latency_s:.2f} วินาที |
| การเขียน To-do | {len(updates_p2)} ครั้ง |
| Tool calls | {len(calls_p2)} |
| ผลตรวจคุณภาพ | รอตรวจ |
"""))

**คำตอบสุดท้าย**

ข้อมูลปี 2025 ของบริษัททั้ง 3 แห่ง โดยคำนวณการเติบโตเทียบปี 2024:

| บริษัท | การเติบโตของรายได้ | อัตรากำไรสุทธิ | หนี้สินรวมต่อสินทรัพย์รวม | ข้อมูลครบทั้ง 3 ตัวชี้วัด | source_id ที่ใช้ |
|---|---:|---:|---:|---|---|
| Delta | 30.0% | 12.0% | 30.0% | **ครบ** | `fixture_delta_v1` |
| Echo | 20.0% | 20.0% | 60.0% | **ครบ** | `fixture_echo_backup_2025` |
| Foxtrot | 20.0% | N/A | 20.0% | **ไม่ครบ** — ไม่มีข้อมูลกำไรสุทธิปี 2025 | `fixture_foxtrot_v1` |

**วิธีคำนวณ:** การเติบโตของรายได้ = (รายได้ปี 2025 − รายได้ปี 2024) ÷ รายได้ปี 2024; อัตรากำไรสุทธิ = กำไรสุทธิปี 2025 ÷ รายได้ปี 2025; หนี้สินรวมต่อสินทรัพย์รวม = หนี้สินรวม ÷ สินทรัพย์รวม

**ข้อมูลที่ขาด:** Foxtrot ไม่มีค่ากำไรสุทธิปี 2025 ในข้อมูลหลัก และการค้นหาเอกสารทางเลือกไม่พบข้อมูลที่เติมช่องว่างนี้ได้ จึงแสดงอัตรากำไรสุทธิเป็น N/A ไม่ได้ถือว่าค่าเป็นศูนย์

**อุปสรรคและการจัดการ:** แหล่งหลักของ Echo แจ้งว่าใช้ไม่ได้และระบุ `retryable=false` จึงไม่เรียกซ้ำ แต่ค้นหาเอกสารทางเลือกแทน พบเอกสารของ Echo ปี 2025 และใช้ `fixture_echo_backup_2025`; เอกสารปี 2024 ที่พบไม่ได้นำมาใช้เพราะไม่ตรงปีที่ต้องการ ส่วน Foxtrot คงตัวชี้วัดกำไรสุทธิเป็น N/A เนื่องจากข้อมูลไม่เพียงพอ

**สรุป:** Delta และ Echo มีข้อมูลครบสำหรับทั้งสามตัวชี้วัด ส่วน Foxtrot ขาดข้อมูลกำไรสุทธิปี 2025 ทำให้คำนวณอัตรากำไรสุทธิไม่ได้

**แผนและการปรับแผน**

**แผนครั้งที่ 1 — Message 1**

- **in_progress** — ดึงรายชื่อบริษัททั้งหมด และอ่านข้อมูลการเงินปี 2025 จากแหล่งหลักของแต่ละบริษัท
- **pending** — ตรวจสอบช่องว่างของข้อมูล แล้วค้นและตรวจเอกสารทางเลือกตามบริษัท/ปีเมื่อจำเป็น
- **pending** — คำนวณตัวชี้วัดที่ทำได้ด้วยเครื่องมือ calculate_percentage และรวบรวม source_id
- **pending** — จัดทำตารางเปรียบเทียบภาษาไทย พร้อมระบุข้อมูลขาด อุปสรรค และข้อสรุป

**แผนครั้งที่ 2 — Message 9**

- **completed** — ดึงรายชื่อบริษัททั้งหมด และอ่านข้อมูลการเงินปี 2025 จากแหล่งหลักของแต่ละบริษัท (เสร็จแล้ว; แหล่งหลัก Echo unavailable แบบ retryable=false)
- **in_progress** — ค้นเอกสารทางเลือกสำหรับ Echo และตรวจเอกสาร/ปีของ Echo กับข้อมูลกำไรสุทธิ Foxtrot ก่อนใช้ตัวเลข
- **pending** — คำนวณตัวชี้วัดที่ทำได้ด้วยเครื่องมือ calculate_percentage และรวบรวม source_id
- **pending** — จัดทำตารางเปรียบเทียบภาษาไทย พร้อมระบุข้อมูลขาด อุปสรรค และข้อสรุป

**แผนครั้งที่ 3 — Message 19**

- **completed** — ดึงรายชื่อบริษัททั้งหมด และอ่านข้อมูลการเงินปี 2025 จากแหล่งหลักของแต่ละบริษัท (เสร็จแล้ว; แหล่งหลัก Echo unavailable แบบ retryable=false)
- **completed** — ค้นเอกสารทางเลือกสำหรับ Echo และตรวจเอกสาร/ปีของ Echo กับข้อมูลกำไรสุทธิ Foxtrot ก่อนใช้ตัวเลข (Echo ใช้เอกสารปี 2025 ได้; ไม่พบแหล่งทางเลือก Foxtrot ที่เติมกำไรสุทธิ)
- **in_progress** — คำนวณตัวชี้วัดที่ทำได้ด้วยเครื่องมือ calculate_percentage และรวบรวม source_id
- **pending** — จัดทำตารางเปรียบเทียบภาษาไทย พร้อมระบุข้อมูลขาด อุปสรรค และข้อสรุป

**แผนครั้งที่ 4 — Message 30**

- **completed** — ดึงรายชื่อบริษัททั้งหมด และอ่านข้อมูลการเงินปี 2025 จากแหล่งหลักของแต่ละบริษัท (เสร็จแล้ว; แหล่งหลัก Echo unavailable แบบ retryable=false)
- **completed** — ค้นเอกสารทางเลือกสำหรับ Echo และตรวจเอกสาร/ปีของ Echo กับข้อมูลกำไรสุทธิ Foxtrot ก่อนใช้ตัวเลข (Echo ใช้เอกสารปี 2025 ได้; ไม่พบแหล่งทางเลือก Foxtrot ที่เติมกำไรสุทธิ)
- **completed** — คำนวณตัวชี้วัดที่ทำได้ด้วยเครื่องมือ calculate_percentage และรวบรวม source_id
- **completed** — จัดทำตารางเปรียบเทียบภาษาไทย พร้อมระบุข้อมูลขาด อุปสรรค และข้อสรุป

**ลำดับเรียกเครื่องมือและ arguments**

Message 3: list_companies {}
Message 5: read_primary_financials {"company_id": "co-d81"}
Message 5: read_primary_financials {"company_id": "co-e42"}
Message 5: read_primary_financials {"company_id": "co-f63"}
Message 11: find_financial_sources {"company_id": "co-e42"}
Message 11: find_financial_sources {"company_id": "co-f63"}
Message 14: read_financial_source {"source_id": "fixture_echo_backup_2025"}
Message 14: find_financial_sources {"company_id": "co-f63"}
Message 17: find_financial_sources {"company_id": "co-f63"}
Message 21: calculate_percentage {"numerator": 1300, "denominator": 1000, "mode": "growth"}
Message 21: calculate_percentage {"numerator": 156, "denominator": 1300, "mode": "ratio"}
Message 21: calculate_percentage {"numerator": 540, "denominator": 1800, "mode": "ratio"}
Message 21: calculate_percentage {"numerator": 960, "denominator": 800, "mode": "growth"}
Message 21: calculate_percentage {"numerator": 192, "denominator": 960, "mode": "ratio"}
Message 21: calculate_pe


| รายการ | ผล |
|---|---|
| Case | planning_002 |
| เวลารวม | 50.93 วินาที |
| การเขียน To-do | 4 ครั้ง |
| Tool calls | 21 |
| ผลตรวจคุณภาพ | รอตรวจ |


In [115]:
from copy import deepcopy

assert planning_001_record["case_id"] == "planning_001"
assert planning_002_report["case_id"] == "planning_002"
assert planning_002_report["runtime_status"] == "completed"

planning_002_record = deepcopy(planning_002_report)

# ผ่านเป้าหมายหลักของโจทย์
# ข้อบกพร่องด้านการอธิบายและประสิทธิภาพบันทึกแยกไว้
planning_002_record["task_passed"] = True

planning_002_record["review"] = {
    "method": "assistant_review_of_shared_output",
    "scope": "final_answer_and_visible_plan_and_tool_calls",
    "checks": {
        "plan_before_data_calls": True,
        "plan_content_adapted_to_obstacles": True,
        "plan_updated_before_alternative_search": True,
        "nonretryable_primary_not_repeated": True,
        "correct_backup_company_and_year": True,
        "calculation_arguments_correct": True,
        "final_metrics_correct": True,
        "missing_data_not_invented": True,
        "source_ids_correct": True,
        "all_plan_items_completed": True,
    },
    "quality_observations": {
        "percentage_formulas_include_times_100": False,
        "repeated_identical_foxtrot_source_search": True,
    },
    "notes": [
        "ผ่านเป้าหมายหลักด้านการปรับแผนและความถูกต้องของผลลัพธ์",
        "สูตรในคำอธิบายขาด × 100 แต่ค่าร้อยละในตารางถูกต้อง",
        "ค้นแหล่งสำรอง Foxtrot ซ้ำที่ Message 11 และ 14",
        "ยังไม่เห็นเหตุผลที่จำเป็นสำหรับการค้นซ้ำจาก trace ที่ส่งมา",
        "โจทย์กำหนดให้วางแผนและปรับแผนอย่างชัดเจน",
    ],
}

planning_records = [
    deepcopy(planning_001_record),
    planning_002_record,
]

for record in planning_records:
    print(
        record["case_id"],
        "| Task passed:",
        record["task_passed"],
    )

planning_001 | Task passed: True
planning_002 | Task passed: True


### บันทึกผลการทดสอบ Planning

In [116]:
from pathlib import Path
from datetime import datetime, timezone
from uuid import uuid4
import json

shared_dir = Path("/workspace/shared")

assert shared_dir.is_dir(), (
    "ไม่พบ /workspace/shared — ตรวจ volume mapping ของ container"
)

# ตำแหน่งมาตรฐานสำหรับผลของ Deep Agents ทุกหมวด
results_dir = shared_dir / "results" / "deepagents"
results_dir.mkdir(parents=True, exist_ok=True)

# ใช้ผลและ run_id เดิม ไม่เรียก agent ใหม่
assert len(planning_records) == 2

saved_at = datetime.now(timezone.utc)

planning_checkpoint = {
    "schema_version": "1.0",
    "saved_at_utc": saved_at.isoformat(),
    "sdk": "deepagents",
    "category": "planning",
    "results": planning_records,
}

serialized = json.dumps(
    planning_checkpoint,
    indent=2,
    ensure_ascii=False,
    allow_nan=False,
)

planning_output_path = results_dir / (
    f"planning_{saved_at:%Y%m%dT%H%M%SZ}_{uuid4().hex[:8]}.json"
)

with planning_output_path.open("x", encoding="utf-8") as file:
    file.write(serialized)

print("Saved:", planning_output_path)
print("Experiments:", len(planning_records))

for record in planning_records:
    print(f"- {record['case_id']}: passed={record['task_passed']}")

Saved: /workspace/shared/results/deepagents/planning_20261006T090506Z_16c0c8c3.json
Experiments: 2
- planning_001: passed=True
- planning_002: passed=True


<hr>

# 6 Long context

ประเมินการจัดการ prompt หรือบริบทที่มีข้อมูลจำนวนมาก โดยตรวจความถูกต้องของข้อเท็จจริงที่ต้องดึงจากเนื้อหา และบันทึกผลเพื่อใช้เปรียบเทียบระหว่างกรณี

In [118]:
import asyncio
import hashlib
import json
import time

from uuid import uuid4
from importlib.metadata import version
from langchain_core.messages import AIMessage
from deepagents import create_deep_agent

### Long context 01 (สรุปเอกสารยาว)

In [119]:
LONG_CASE_ID = "long_context_001"

def make_background(start, stop):
    return "\n\n".join(
        (
            f"[บันทึกประกอบ B{i:03d}]\n"
            f"บริษัท Other-{i:03d} รายงานยอดขายปี 2025 "
            f"จำนวน {2000 + i * 17} ล้านบาท "
            f"และกำไรสุทธิ {100 + i} ล้านบาท "
            "ฝ่ายปฏิบัติการอยู่ระหว่างทบทวนแผนคลังสินค้า "
            "ข้อมูลนี้เป็นของบริษัทอื่น ไม่ใช่งบของบริษัทเป้าหมาย"
        )
        for i in range(start, stop)
    )

long_documents = "\n\n".join([
    """
[เอกสาร N1]
บริษัท Northstar | ปี 2025 | งบรวม | หน่วย ล้านบาท
รายได้ 1200 | กำไรสุทธิ 120

[เอกสาร O1]
บริษัท Orchid | ปี 2025 | งบรวม | หน่วย ล้านบาท
รายได้ 900 | กำไรสุทธิ 90

[เอกสาร P1]
บริษัท Pine | ปี 2025 | งบรวม | หน่วย ล้านบาท
รายได้ 1500 | กำไรสุทธิ 150
""",
    make_background(1, 21),
    """
[ประกาศแก้ไข N2 | revision 2]
แทนที่เอกสาร N1 ทั้งฉบับ
บริษัท Northstar | ปี 2025 | งบรวม | หน่วย ล้านบาท
รายได้ 1250 | กำไรสุทธิ 125

[เอกสาร O-SEPARATE]
บริษัท Orchid | ปี 2025 | งบเฉพาะกิจการ | หน่วย ล้านบาท
รายได้ 700 | กำไรสุทธิ 140
""",
    make_background(21, 41),
    """
[ประกาศแก้ไข P2 | revision 2]
แทนที่เอกสาร P1 ทั้งฉบับ
บริษัท Pine | ปี 2025 | งบรวม | หน่วย ล้านบาท
รายได้ 1500 | กำไรสุทธิ: ไม่เปิดเผย
ห้ามนำกำไรสุทธิจากฉบับที่ถูกแทนที่มาเติม

[ประกาศแก้ไข N3 | revision 3]
แทนที่เอกสาร N2 ทั้งฉบับ
บริษัท Northstar | ปี 2025 | งบรวม | หน่วย ล้านบาท
รายได้ 1300 | กำไรสุทธิ 156
""",
    make_background(41, 61),
    """
[เอกสาร O-2024]
บริษัท Orchid | ปี 2024 | งบรวม | หน่วย ล้านบาท
รายได้ 800 | กำไรสุทธิ 160

[สำเนาประกาศเก่าสำหรับการตรวจสอบย้อนหลัง]
เอกสาร N2 | revision 2
บริษัท Northstar | ปี 2025 | งบรวม | หน่วย ล้านบาท
รายได้ 1250 | กำไรสุทธิ 125
สำเนานี้ไม่ใช่ revision ใหม่
""",
])

print("Dataset ready.")
print("Document characters:", f"{len(long_documents):,}")

Dataset ready.
Document characters: 13,031


In [120]:
long_system_prompt = (
    "Read the supplied documents and follow the user's output schema. "
    "Use only information in the user message. "
    "Do not call tools, browse, delegate, or access files. "
    "Return only the requested JSON object."
)

long_prompt = """
อ่านเอกสารทั้งหมดแล้วสรุปเฉพาะ Northstar, Orchid และ Pine

ข้อกำหนด:
1. ใช้เฉพาะงบรวมปี 2025 หน่วยล้านบาท
2. เรียงบริษัทตามนี้เท่านั้น: Northstar, Orchid, Pine
3. หากมีประกาศแทนที่ ให้เลือก revision สูงสุดของงบที่ตรงเงื่อนไข
   ห้ามเลือกเพียงเพราะข้อความนั้นอยู่ท้ายเอกสาร
4. ประกาศแทนที่มีผลทั้งฉบับ ห้ามผสมข้อมูลกับฉบับเก่า
5. หากกำไรสุทธิไม่เปิดเผย ให้ใช้ null และ missing_fields=["net_profit_million"]
   หากเปิดเผย ให้ missing_fields=[]
6. source_id ต้องเป็นรหัสเอกสารที่ใช้จริง
7. ห้ามใช้ tools ทุกชนิด ใช้ข้อมูลในคำถามนี้เท่านั้น
8. ตอบ JSON object เท่านั้น ไม่ใส่ code fence หรือคำอธิบาย
9. ห้ามเพิ่ม keys นอกโครงสร้างที่กำหนด
10. ตัวเลขต้องเป็น JSON number ไม่ใช่ string

โครงสร้าง:
{
  "year": 2025,
  "unit": "million_thb",
  "companies": [
    {
      "company_name": "ชื่อบริษัท",
      "revenue_million": 0,
      "net_profit_million": null,
      "source_id": "รหัสเอกสาร",
      "missing_fields": []
    }
  ]
}

รายการ companies ต้องมีครบ 3 บริษัทตามลำดับที่กำหนด

เอกสาร:
""" + long_documents

long_agent = create_deep_agent(
    model=model,
    tools=[],
    system_prompt=long_system_prompt,
)

print("Agent ready.")
print("Prompt characters:", f"{len(long_prompt):,}")
print(
    "Prompt SHA256:",
    hashlib.sha256(long_prompt.encode("utf-8")).hexdigest(),
)

Agent ready.
Prompt characters: 14,022
Prompt SHA256: 3e628fe8b42d626292c855a9c91f3e9bd3a910a4e2d44580d482cbf0199021ae


In [121]:
long_run_id = uuid4().hex
long_result = None
long_error = None
long_runtime_status = "running"

long_versions = {
    name: version(name)
    for name in ["deepagents", "langchain", "langchain-openai"]
}

started_at = time.perf_counter()

try:
    long_result = await asyncio.wait_for(
        long_agent.ainvoke(
            {"messages": [{"role": "user", "content": long_prompt}]},
            config={"recursion_limit": 20},
        ),
        timeout=180,
    )
    long_runtime_status = "completed"

except Exception as exc:
    long_runtime_status = "failed"
    long_error = {
        "type": type(exc).__name__,
        "message": str(exc),
    }

finally:
    long_latency_s = time.perf_counter() - started_at

print("Runtime status:", long_runtime_status)
print("Latency:", round(long_latency_s, 2), "seconds")

if long_error:
    print("Error:", long_error)

Runtime status: completed
Latency: 4.02 seconds


In [122]:
import math

long_messages = (
    long_result.get("messages", [])
    if long_result is not None else []
)
long_last = long_messages[-1] if long_messages else None
long_answer = (
    long_last.text.strip()
    if isinstance(long_last, AIMessage) else ""
)

long_ai_messages = [
    msg for msg in long_messages if isinstance(msg, AIMessage)
]
long_tool_calls = [
    call for msg in long_ai_messages for call in msg.tool_calls
]

try:
    long_parsed = json.loads(long_answer)
except json.JSONDecodeError:
    long_parsed = None

expected_long_rows = [
    {
        "company_name": "Northstar",
        "revenue_million": 1300,
        "net_profit_million": 156,
        "source_id": "N3",
        "missing_fields": [],
    },
    {
        "company_name": "Orchid",
        "revenue_million": 900,
        "net_profit_million": 90,
        "source_id": "O1",
        "missing_fields": [],
    },
    {
        "company_name": "Pine",
        "revenue_million": 1500,
        "net_profit_million": None,
        "source_id": "P2",
        "missing_fields": ["net_profit_million"],
    },
]

def same_number(actual, expected):
    if expected is None:
        return actual is None
    return (
        type(actual) in (int, float)
        and math.isfinite(actual)
        and actual == expected
    )

parsed_object = long_parsed if isinstance(long_parsed, dict) else {}
rows = parsed_object.get("companies", [])
rows_valid = (
    isinstance(rows, list)
    and len(rows) == 3
    and all(isinstance(row, dict) for row in rows)
)
safe_rows = rows if rows_valid else []

# ไม่อาศัยลำดับบริษัทในการตรวจตัวเลข
def row_for(name):
    matches = [
        row for row in safe_rows if row.get("company_name") == name
    ]
    return matches[0] if len(matches) == 1 else {}

expected_names = [row["company_name"] for row in expected_long_rows]

extraction_checks = {
    expected["company_name"]: all(
        key in row_for(expected["company_name"])
        and same_number(
            row_for(expected["company_name"])[key],
            expected[key],
        )
        for key in ("revenue_million", "net_profit_million")
    )
    for expected in expected_long_rows
}

revision_checks = {
    expected["company_name"]: (
        row_for(expected["company_name"]).get("source_id")
        == expected["source_id"]
        and row_for(expected["company_name"]).get("missing_fields")
        == expected["missing_fields"]
    )
    for expected in expected_long_rows
}

constraint_checks = {
    "exact_top_level_keys": (
        isinstance(long_parsed, dict)
        and set(long_parsed) == {"year", "unit", "companies"}
    ),
    "correct_year_and_unit": (
        type(parsed_object.get("year")) is int
        and parsed_object.get("year") == 2025
        and parsed_object.get("unit") == "million_thb"
    ),
    "company_count_and_order": (
        rows_valid
        and [row.get("company_name") for row in rows] == expected_names
    ),
    "exact_row_keys": (
        rows_valid
        and all(
            set(row) == set(expected_long_rows[0])
            for row in rows
        )
    ),
    "no_tool_calls": len(long_tool_calls) == 0,
    "final_answer_complete": (
        isinstance(long_last, AIMessage)
        and bool(long_answer)
        and not long_last.tool_calls
    ),
}

long_axes = {
    "information_extraction": all(extraction_checks.values()),
    "instruction_following": all(constraint_checks.values()),
    "revision_and_missing_data": (
        all(revision_checks.values())
        and extraction_checks["Northstar"]
        and extraction_checks["Pine"]
    ),
}

long_report = {
    "case_id": LONG_CASE_ID,
    "run_id": long_run_id,
    "sdk": "deepagents",
    "model": model.model_name,
    "versions": long_versions,
    "fixture_version": "long_context_v1",
    "runtime_status": long_runtime_status,
    "error": long_error,
    "latency_s": round(long_latency_s, 2),
    "prompt_characters": len(long_prompt),
    "prompt_sha256": hashlib.sha256(
        long_prompt.encode("utf-8")
    ).hexdigest(),
    "system_prompt": long_system_prompt,
    "prompt": long_prompt,
    "axes": long_axes,
    "checks": {
        "extraction": extraction_checks,
        "constraints": constraint_checks,
        "revisions": revision_checks,
    },
    "task_passed": (
        long_runtime_status == "completed"
        and all(long_axes.values())
    ),
    "tool_call_count": len(long_tool_calls),
    "raw_answer": long_answer,
    "answer": long_parsed,
    "trace": [
        msg.model_dump(mode="json") for msg in long_messages
    ],
    "usage_per_visible_ai_message": [
        msg.usage_metadata for msg in long_ai_messages
    ],
}

print("ANSWER:")
print(json.dumps(long_parsed, indent=2, ensure_ascii=False))

if long_parsed is None:
    print("Raw answer:", long_answer)

print("\nCHECKS:")
print(json.dumps(long_report["checks"], indent=2, ensure_ascii=False))

print("\nAXES:")
print(json.dumps(long_axes, indent=2, ensure_ascii=False))

print("\nTask passed:", long_report["task_passed"])
print("Latency:", long_report["latency_s"], "seconds")
print("Tool calls:", long_report["tool_call_count"])

ANSWER:
{
  "year": 2025,
  "unit": "million_thb",
  "companies": [
    {
      "company_name": "Northstar",
      "revenue_million": 1300,
      "net_profit_million": 156,
      "source_id": "N3",
      "missing_fields": []
    },
    {
      "company_name": "Orchid",
      "revenue_million": 900,
      "net_profit_million": 90,
      "source_id": "O1",
      "missing_fields": []
    },
    {
      "company_name": "Pine",
      "revenue_million": 1500,
      "net_profit_million": null,
      "source_id": "P2",
      "missing_fields": [
        "net_profit_million"
      ]
    }
  ]
}

CHECKS:
{
  "extraction": {
    "Northstar": true,
    "Orchid": true,
    "Pine": true
  },
  "constraints": {
    "exact_top_level_keys": true,
    "correct_year_and_unit": true,
    "company_count_and_order": true,
    "exact_row_keys": true,
    "no_tool_calls": true,
    "final_answer_complete": true
  },
  "revisions": {
    "Northstar": true,
    "Orchid": true,
    "Pine": true
  }
}

AXES:
{
  "

In [123]:
from copy import deepcopy

# เก็บผลเคสแรกไว้ก่อน
long_001_record = deepcopy(long_report)

assert long_001_record["case_id"] == "long_context_001"

# แยกเอกสารเดิมตรงช่วงข้อมูลรบกวน
# เพื่อรักษาข้อมูลเป้าหมายและประกาศแก้ไขทุกฉบับไว้เหมือนเดิม
remaining = long_documents
document_parts = []

for start, stop in [(1, 21), (21, 41), (41, 61)]:
    old_background = make_background(start, stop)
    before, separator, remaining = remaining.partition(old_background)

    assert separator, f"ไม่พบข้อมูลรบกวนช่วง {start}–{stop - 1}"
    document_parts.append(before)

document_parts.append(remaining)

# เพิ่มจาก 60 เป็น 300 บันทึก โดยใช้หมายเลขไม่ซ้ำ
long_002_documents = (
    document_parts[0]
    + make_background(1, 101)
    + document_parts[1]
    + make_background(101, 201)
    + document_parts[2]
    + make_background(201, 301)
    + document_parts[3]
)

# ใช้คำสั่งเดิม เปลี่ยนเฉพาะเอกสาร
assert long_prompt.endswith(long_documents)

instruction_prefix = long_prompt[:-len(long_documents)]
long_002_prompt = instruction_prefix + long_002_documents

print("Original prompt characters:", f"{len(long_prompt):,}")
print("New prompt characters:", f"{len(long_002_prompt):,}")
print("Length ratio:", round(len(long_002_prompt) / len(long_prompt), 2))
print("Distractor records: 60 → 300")

Original prompt characters: 14,022
New prompt characters: 61,782
Length ratio: 4.41
Distractor records: 60 → 300


### Long context 002

In [124]:
from copy import deepcopy

# เก็บผลเคสแรกไว้ก่อน
long_001_record = deepcopy(long_report)

assert long_001_record["case_id"] == "long_context_001"

# แยกเอกสารเดิมตรงช่วงข้อมูลรบกวน
# เพื่อรักษาข้อมูลเป้าหมายและประกาศแก้ไขทุกฉบับไว้เหมือนเดิม
remaining = long_documents
document_parts = []

for start, stop in [(1, 21), (21, 41), (41, 61)]:
    old_background = make_background(start, stop)
    before, separator, remaining = remaining.partition(old_background)

    assert separator, f"ไม่พบข้อมูลรบกวนช่วง {start}–{stop - 1}"
    document_parts.append(before)

document_parts.append(remaining)

# เพิ่มจาก 60 เป็น 300 บันทึก โดยใช้หมายเลขไม่ซ้ำ
long_002_documents = (
    document_parts[0]
    + make_background(1, 101)
    + document_parts[1]
    + make_background(101, 201)
    + document_parts[2]
    + make_background(201, 301)
    + document_parts[3]
)

# ใช้คำสั่งเดิม เปลี่ยนเฉพาะเอกสาร
assert long_prompt.endswith(long_documents)

instruction_prefix = long_prompt[:-len(long_documents)]
long_002_prompt = instruction_prefix + long_002_documents

print("Original prompt characters:", f"{len(long_prompt):,}")
print("New prompt characters:", f"{len(long_002_prompt):,}")
print("Length ratio:", round(len(long_002_prompt) / len(long_prompt), 2))
print("Distractor records: 60 → 300")

Original prompt characters: 14,022
New prompt characters: 61,782
Length ratio: 4.41
Distractor records: 60 → 300


In [125]:
long_002_agent = create_deep_agent(
    model=model,
    tools=[],
    system_prompt=long_system_prompt,
)

long_002_run_id = uuid4().hex
long_002_result = None
long_002_error = None
long_002_status = "running"

long_002_versions = {
    name: version(name)
    for name in ["deepagents", "langchain", "langchain-openai"]
}

started_at = time.perf_counter()

try:
    long_002_result = await asyncio.wait_for(
        long_002_agent.ainvoke(
            {
                "messages": [
                    {"role": "user", "content": long_002_prompt}
                ]
            },
            config={"recursion_limit": 20},
        ),
        timeout=180,
    )
    long_002_status = "completed"

except Exception as exc:
    long_002_status = "failed"
    long_002_error = {
        "type": type(exc).__name__,
        "message": str(exc),
    }

finally:
    long_002_latency_s = time.perf_counter() - started_at

print("Runtime status:", long_002_status)
print("Latency:", round(long_002_latency_s, 2), "seconds")

if long_002_error:
    print("Error:", long_002_error)

Runtime status: completed
Latency: 4.19 seconds


In [126]:
messages_l2 = (
    long_002_result.get("messages", [])
    if long_002_result is not None else []
)
last_l2 = messages_l2[-1] if messages_l2 else None

answer_l2 = (
    last_l2.text.strip()
    if isinstance(last_l2, AIMessage) else ""
)

ai_messages_l2 = [
    msg for msg in messages_l2 if isinstance(msg, AIMessage)
]
tool_calls_l2 = [
    call for msg in ai_messages_l2 for call in msg.tool_calls
]

try:
    parsed_l2 = json.loads(answer_l2)
except json.JSONDecodeError:
    parsed_l2 = None

object_l2 = parsed_l2 if isinstance(parsed_l2, dict) else {}
rows_l2 = object_l2.get("companies", [])

rows_valid_l2 = (
    isinstance(rows_l2, list)
    and len(rows_l2) == 3
    and all(isinstance(row, dict) for row in rows_l2)
)

def row_for_l2(name):
    matches = [
        row
        for row in (rows_l2 if rows_valid_l2 else [])
        if row.get("company_name") == name
    ]
    return matches[0] if len(matches) == 1 else {}

extraction_l2 = {
    expected["company_name"]: all(
        key in row_for_l2(expected["company_name"])
        and same_number(
            row_for_l2(expected["company_name"])[key],
            expected[key],
        )
        for key in ("revenue_million", "net_profit_million")
    )
    for expected in expected_long_rows
}

revisions_l2 = {
    expected["company_name"]: (
        row_for_l2(expected["company_name"]).get("source_id")
        == expected["source_id"]
        and row_for_l2(expected["company_name"]).get("missing_fields")
        == expected["missing_fields"]
    )
    for expected in expected_long_rows
}

constraints_l2 = {
    "exact_top_level_keys": (
        isinstance(parsed_l2, dict)
        and set(parsed_l2) == {"year", "unit", "companies"}
    ),
    "correct_year_and_unit": (
        type(object_l2.get("year")) is int
        and object_l2.get("year") == 2025
        and object_l2.get("unit") == "million_thb"
    ),
    "company_count_and_order": (
        rows_valid_l2
        and [row.get("company_name") for row in rows_l2]
        == [row["company_name"] for row in expected_long_rows]
    ),
    "exact_row_keys": (
        rows_valid_l2
        and all(
            set(row) == set(expected_long_rows[0])
            for row in rows_l2
        )
    ),
    "no_tool_calls": len(tool_calls_l2) == 0,
    "final_answer_complete": (
        isinstance(last_l2, AIMessage)
        and bool(answer_l2)
        and not last_l2.tool_calls
    ),
}

axes_l2 = {
    "information_extraction": all(extraction_l2.values()),
    "instruction_following": all(constraints_l2.values()),
    "revision_and_missing_data": (
        all(revisions_l2.values())
        and extraction_l2["Northstar"]
        and extraction_l2["Pine"]
    ),
}

long_002_report = {
    "case_id": "long_context_002",
    "run_id": long_002_run_id,
    "sdk": "deepagents",
    "model": model.model_name,
    "versions": long_002_versions,
    "fixture_version": "long_context_v1_distractors_300",
    "runtime_status": long_002_status,
    "error": long_002_error,
    "latency_s": round(long_002_latency_s, 2),
    "prompt_characters": len(long_002_prompt),
    "prompt_sha256": hashlib.sha256(
        long_002_prompt.encode("utf-8")
    ).hexdigest(),
    "configuration": {
        "distractor_records": 300,
        "fresh_conversation": True,
        "recursion_limit": 20,
        "timeout_s": 180,
    },
    "system_prompt": long_system_prompt,
    "prompt": long_002_prompt,
    "axes": axes_l2,
    "checks": {
        "extraction": extraction_l2,
        "constraints": constraints_l2,
        "revisions": revisions_l2,
    },
    "task_passed": (
        long_002_status == "completed"
        and all(axes_l2.values())
    ),
    "tool_call_count": len(tool_calls_l2),
    "raw_answer": answer_l2,
    "answer": parsed_l2,
    "trace": [
        msg.model_dump(mode="json") for msg in messages_l2
    ],
    "usage_per_visible_ai_message": [
        msg.usage_metadata for msg in ai_messages_l2
    ],
}

print("ANSWER:")
print(json.dumps(parsed_l2, indent=2, ensure_ascii=False))

if parsed_l2 is None:
    print("Raw answer:", answer_l2)

print("\nCHECKS:")
print(json.dumps(long_002_report["checks"], indent=2, ensure_ascii=False))

ANSWER:
{
  "year": 2025,
  "unit": "million_thb",
  "companies": [
    {
      "company_name": "Northstar",
      "revenue_million": 1300,
      "net_profit_million": 156,
      "source_id": "N3",
      "missing_fields": []
    },
    {
      "company_name": "Orchid",
      "revenue_million": 900,
      "net_profit_million": 90,
      "source_id": "O1",
      "missing_fields": []
    },
    {
      "company_name": "Pine",
      "revenue_million": 1500,
      "net_profit_million": null,
      "source_id": "P2",
      "missing_fields": [
        "net_profit_million"
      ]
    }
  ]
}

CHECKS:
{
  "extraction": {
    "Northstar": true,
    "Orchid": true,
    "Pine": true
  },
  "constraints": {
    "exact_top_level_keys": true,
    "correct_year_and_unit": true,
    "company_count_and_order": true,
    "exact_row_keys": true,
    "no_tool_calls": true,
    "final_answer_complete": true
  },
  "revisions": {
    "Northstar": true,
    "Orchid": true,
    "Pine": true
  }
}


In [127]:
from IPython.display import Markdown, display

comparison_lines = [
    "| Case | Prompt characters | Latency (s) | Tool calls | Passed |",
    "|---|---:|---:|---:|---|",
]

for report in [long_001_record, long_002_report]:
    comparison_lines.append(
        f"| {report['case_id']} "
        f"| {report['prompt_characters']:,} "
        f"| {report['latency_s']:.2f} "
        f"| {report['tool_call_count']} "
        f"| {report['task_passed']} |"
    )

display(Markdown("\n".join(comparison_lines)))

print("AXES:")
print(json.dumps(axes_l2, indent=2, ensure_ascii=False))

print("\nUSAGE — case 002:")
print(json.dumps(
    long_002_report["usage_per_visible_ai_message"],
    indent=2,
    ensure_ascii=False,
))

| Case | Prompt characters | Latency (s) | Tool calls | Passed |
|---|---:|---:|---:|---|
| long_context_001 | 14,022 | 4.02 | 0 | True |
| long_context_002 | 61,782 | 4.19 | 0 | True |

AXES:
{
  "information_extraction": true,
  "instruction_following": true,
  "revision_and_missing_data": true
}

USAGE — case 002:
[
  {
    "input_tokens": 24100,
    "output_tokens": 238,
    "total_tokens": 24338,
    "input_token_details": {
      "cache_creation": 22096,
      "cache_read": 2001
    },
    "output_token_details": {
      "reasoning": 56
    }
  }
]


### บันทึกผลการทดสอบ Long context

In [128]:
from pathlib import Path
from datetime import datetime, timezone
from uuid import uuid4
from copy import deepcopy
import json

assert long_001_record["task_passed"] is True
assert long_002_report["task_passed"] is True

long_context_records = [
    deepcopy(long_001_record),
    deepcopy(long_002_report),
]

shared_dir = Path("/workspace/shared")
assert shared_dir.is_dir(), "ไม่พบ /workspace/shared"

results_dir = shared_dir / "results" / "deepagents"
results_dir.mkdir(parents=True, exist_ok=True)

saved_at = datetime.now(timezone.utc)

long_context_checkpoint = {
    "schema_version": "1.0",
    "saved_at_utc": saved_at.isoformat(),
    "sdk": "deepagents",
    "category": "long_context",
    "results": long_context_records,
}

output_path = results_dir / (
    f"long_context_{saved_at:%Y%m%dT%H%M%SZ}_{uuid4().hex[:8]}.json"
)

with output_path.open("x", encoding="utf-8") as file:
    json.dump(
        long_context_checkpoint,
        file,
        indent=2,
        ensure_ascii=False,
        allow_nan=False,
    )

print("Saved:", output_path)
print("Experiments:", len(long_context_records))

for record in long_context_records:
    print(
        f"- {record['case_id']}: "
        f"characters={record['prompt_characters']:,}, "
        f"passed={record['task_passed']}"
    )

Saved: /workspace/shared/results/deepagents/long_context_20261006T091058Z_61da0beb.json
Experiments: 2
- long_context_001: characters=14,022, passed=True
- long_context_002: characters=61,782, passed=True


<hr>

# 7 Session history

ทดสอบการเก็บและนำประวัติการสนทนากลับมาใช้ต่อเนื่อง รวมถึงการคงสถานะผ่านการเริ่ม kernel ใหม่ เพื่อดูว่า agent สามารถอ้างอิงบริบทก่อนหน้าได้ตามที่คาดหรือไม่

### history_001 (สร้าง Session)

เริ่มบทสนทนาและเก็บ thread ID กับผลการตอบ เพื่อใช้เป็นบริบทสำหรับการสนทนาต่อเนื่อง


In [130]:
from copy import deepcopy
from uuid import uuid4
from importlib.metadata import version
import asyncio
import hashlib
import json
import math
import time

from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import AIMessage
from deepagents import create_deep_agent

In [131]:
HISTORY_CASE_ID = "history_001"

history_checkpointer = InMemorySaver()

history_system_prompt = (
    "You are a session-memory test agent. "
    "Use only information from the current conversation thread. "
    "When asked for JSON, return JSON only without Markdown."
)

history_agent = create_deep_agent(
    model=model,
    tools=[],
    system_prompt=history_system_prompt,
    checkpointer=history_checkpointer,
)

print("Session-history agent ready.")

Session-history agent ready.


In [132]:
history_thread_id = f"history-{uuid4().hex}"

history_turn_1_prompt = """
จดจำข้อมูลต่อไปนี้ไว้สำหรับคำถามถัดไปใน session นี้:

บริษัท: Aurora
ปีงบประมาณ: 2025
รายได้รวม: 2400 ล้านบาท
กำไรสุทธิ: 288 ล้านบาท
อัตรากำไรสุทธิ: 12%

ตอบเพียงคำว่า CONTEXT_SAVED
"""

history_started_at = time.perf_counter()

history_turn_1_result = await asyncio.wait_for(
    history_agent.ainvoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": history_turn_1_prompt,
                }
            ]
        },
        config={
            "configurable": {
                "thread_id": history_thread_id,
            }
        },
    ),
    timeout=120,
)

history_turn_1_latency_s = time.perf_counter() - history_started_at

history_turn_1_messages = history_turn_1_result["messages"]
history_turn_1_last = history_turn_1_messages[-1]

print("Thread ID:", history_thread_id)
print("Response:", history_turn_1_last.text)
print("Latency:", round(history_turn_1_latency_s, 2), "seconds")

Thread ID: history-29c8c910e570441f862d66bc25f4a806
Response: CONTEXT_SAVED
Latency: 2.78 seconds


In [133]:
history_turn_2_prompt = """
จากข้อมูลที่บันทึกไว้ก่อนหน้านี้ ตอบเป็น JSON object เท่านั้น:

{
  "company": "ชื่อบริษัท",
  "year": 0,
  "revenue_million": 0,
  "net_profit_million": 0,
  "net_margin_pct": 0
}

ห้ามเพิ่ม key อื่น และห้ามคำนวณหรือเดาตัวเลขใหม่
"""

history_started_at = time.perf_counter()

history_turn_2_result = await asyncio.wait_for(
    history_agent.ainvoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": history_turn_2_prompt,
                }
            ]
        },
        config={
            "configurable": {
                "thread_id": history_thread_id,
            }
        },
    ),
    timeout=120,
)

history_turn_2_latency_s = time.perf_counter() - history_started_at

history_turn_2_messages = history_turn_2_result["messages"]
history_turn_2_last = history_turn_2_messages[-1]

history_answer = (
    history_turn_2_last.text.strip()
    if isinstance(history_turn_2_last, AIMessage)
    else ""
)

print("Response:", history_answer)
print("Latency:", round(history_turn_2_latency_s, 2), "seconds")
print("Messages in returned state:", len(history_turn_2_messages))

Response: {
  "company": "Aurora",
  "year": 2025,
  "revenue_million": 2400,
  "net_profit_million": 288,
  "net_margin_pct": 12
}
Latency: 4.17 seconds
Messages in returned state: 4


In [134]:
try:
    history_parsed = json.loads(history_answer)
except json.JSONDecodeError:
    history_parsed = None

expected_history = {
    "company": "Aurora",
    "year": 2025,
    "revenue_million": 2400,
    "net_profit_million": 288,
    "net_margin_pct": 12,
}

history_format_passed = (
    isinstance(history_parsed, dict)
    and set(history_parsed) == set(expected_history)
)

history_values_passed = (
    history_format_passed
    and all(
        history_parsed[key] == value
        for key, value in expected_history.items()
    )
)

history_ai_messages = [
    message
    for message in history_turn_2_messages
    if isinstance(message, AIMessage)
]

history_tool_calls = [
    call
    for message in history_ai_messages
    for call in message.tool_calls
]

history_report = {
    "case_id": HISTORY_CASE_ID,
    "run_id": uuid4().hex,
    "sdk": "deepagents",
    "model": model.model_name,
    "versions": {
        name: version(name)
        for name in [
            "deepagents",
            "langchain",
            "langchain-openai",
        ]
    },
    "fixture_version": "session_history_v1",
    "runtime_status": "completed",
    "thread_id": history_thread_id,
    "checks": {
        "context_saved_turn_1": (
            history_turn_1_last.text.strip()
            == "CONTEXT_SAVED"
        ),
        "same_thread_reused": True,
        "exact_json_structure": history_format_passed,
        "remembered_values_correct": history_values_passed,
        "no_tool_calls": len(history_tool_calls) == 0,
        "final_answer_complete": (
            isinstance(history_turn_2_last, AIMessage)
            and not history_turn_2_last.tool_calls
        ),
    },
    "task_passed": (
        history_turn_1_last.text.strip() == "CONTEXT_SAVED"
        and history_values_passed
        and len(history_tool_calls) == 0
    ),
    "latency_turn_1_s": round(history_turn_1_latency_s, 2),
    "latency_turn_2_s": round(history_turn_2_latency_s, 2),
    "answer": history_parsed,
    "raw_answer": history_answer,
    "turn_1_prompt": history_turn_1_prompt,
    "turn_2_prompt": history_turn_2_prompt,
    "messages": [
        message.model_dump(mode="json")
        for message in history_turn_2_messages
    ],
    "usage_per_visible_ai_message": [
        message.usage_metadata
        for message in history_ai_messages
    ],
}

print(json.dumps(history_report, indent=2, ensure_ascii=False))

{
  "case_id": "history_001",
  "run_id": "51e96da0bc524baea4215967b0368f23",
  "sdk": "deepagents",
  "model": "gpt-6-luna",
  "versions": {
    "deepagents": "0.7.21",
    "langchain": "1.4.3",
    "langchain-openai": "1.6.7"
  },
  "fixture_version": "session_history_v1",
  "runtime_status": "completed",
  "thread_id": "history-29c8c910e570441f862d66bc25f4a806",
  "checks": {
    "context_saved_turn_1": true,
    "same_thread_reused": true,
    "exact_json_structure": true,
    "remembered_values_correct": true,
    "no_tool_calls": true,
    "final_answer_complete": true
  },
  "task_passed": true,
  "latency_turn_1_s": 2.78,
  "latency_turn_2_s": 4.17,
  "answer": {
    "company": "Aurora",
    "year": 2025,
    "revenue_million": 2400,
    "net_profit_million": 288,
    "net_margin_pct": 12
  },
  "raw_answer": "{\n  \"company\": \"Aurora\",\n  \"year\": 2025,\n  \"revenue_million\": 2400,\n  \"net_profit_million\": 288,\n  \"net_margin_pct\": 12\n}",
  "turn_1_prompt": "\nจดจำข้

### History_002 (ดึงข้อมูลจาก Thread เดิม)

ส่งข้อความต่อใน thread เดิมและตรวจว่าคำตอบอาศัยบริบทจาก turn ก่อนหน้าได้ถูกต้อง

In [135]:
from copy import deepcopy

history_001_record = deepcopy(history_report)

assert history_001_record["case_id"] == "history_001"

history_002_run_id = uuid4().hex

# เก็บ ID ข้อความเดิม เพื่อไม่รวม usage ของเคสแรกซ้ำ
history_previous_ids = {
    message.id for message in history_turn_2_messages
}
assert None not in history_previous_ids

history_correction_prompt = """
แก้ไขข้อมูลของบริษัทที่เราคุยกันก่อนหน้านี้:

- กำไรสุทธิที่ถูกต้องคือ 360 ล้านบาท แทนค่าเดิม
- อัตรากำไรสุทธิที่ถูกต้องคือ 15% แทนค่าเดิม
- ชื่อบริษัท ปีงบประมาณ และรายได้รวม คงเดิม

จากนี้ให้ใช้ค่าที่แก้ไขแล้ว
ตอบเพียงคำว่า CONTEXT_UPDATED
"""

history_recall_prompt = """
สรุปข้อมูลล่าสุดของบริษัทที่เราคุยกันไว้
ใช้ค่าที่แก้ไขล่าสุด และคงข้อมูลที่ไม่ได้ถูกแก้ไข

ตอบ JSON object เท่านั้น มี keys ดังนี้:
company, year, revenue_million, net_profit_million, net_margin_pct

ห้ามเพิ่ม key อื่น ห้ามใช้ tools และห้ามเดาข้อมูล
"""

history_002_config = {
    "configurable": {"thread_id": history_thread_id},
    "recursion_limit": 20,
}

print("Thread:", history_thread_id)
print("Correction test ready.")

Thread: history-29c8c910e570441f862d66bc25f4a806
Correction test ready.


In [136]:
history_002_turns = []
history_002_result = None
history_002_error = None
history_002_status = "running"

started_at = time.perf_counter()

try:
    for turn_name, prompt in [
        ("correction", history_correction_prompt),
        ("recall", history_recall_prompt),
    ]:
        turn_started_at = time.perf_counter()

        history_002_result = await asyncio.wait_for(
            history_agent.ainvoke(
                {
                    "messages": [
                        {"role": "user", "content": prompt}
                    ]
                },
                config=history_002_config,
            ),
            timeout=120,
        )

        last_message = history_002_result["messages"][-1]
        answer = (
            last_message.text.strip()
            if isinstance(last_message, AIMessage) else ""
        )

        history_002_turns.append({
            "turn": turn_name,
            "prompt": prompt,
            "answer": answer,
            "latency_s": round(
                time.perf_counter() - turn_started_at, 2
            ),
        })

        print(f"\n{turn_name}:")
        print(answer)

    history_002_status = "completed"

except Exception as exc:
    history_002_status = "failed"
    history_002_error = {
        "type": type(exc).__name__,
        "message": str(exc),
    }

finally:
    history_002_latency_s = time.perf_counter() - started_at

print("\nRuntime status:", history_002_status)
print("Latency:", round(history_002_latency_s, 2), "seconds")

if history_002_error:
    print("Error:", history_002_error)


correction:
CONTEXT_UPDATED

recall:
{
  "company": "Aurora",
  "year": 2025,
  "revenue_million": 2400,
  "net_profit_million": 360,
  "net_margin_pct": 15
}

Runtime status: completed
Latency: 6.18 seconds


In [137]:
messages_h2 = (
    history_002_result["messages"]
    if history_002_result is not None else []
)

new_messages_h2 = [
    message for message in messages_h2
    if message.id not in history_previous_ids
]

new_ai_h2 = [
    message for message in new_messages_h2
    if isinstance(message, AIMessage)
]

calls_h2 = [
    call for message in new_ai_h2
    for call in message.tool_calls
]

recall_turn_h2 = next(
    (
        turn for turn in history_002_turns
        if turn["turn"] == "recall"
    ),
    None,
)

answer_h2 = recall_turn_h2["answer"] if recall_turn_h2 else ""

try:
    parsed_h2 = json.loads(answer_h2)
except json.JSONDecodeError:
    parsed_h2 = None

object_h2 = parsed_h2 if isinstance(parsed_h2, dict) else {}

expected_h2 = {
    "company": "Aurora",
    "year": 2025,
    "revenue_million": 2400,
    "net_profit_million": 360,
    "net_margin_pct": 15,
}

def history_number_matches(key):
    actual = object_h2.get(key)
    return (
        type(actual) in (int, float)
        and math.isfinite(actual)
        and actual == expected_h2[key]
    )

last_h2 = messages_h2[-1] if messages_h2 else None

checks_h2 = {
    "correction_acknowledged": (
        bool(history_002_turns)
        and history_002_turns[0]["answer"] == "CONTEXT_UPDATED"
    ),
    "original_messages_retained": (
        history_previous_ids.issubset(
            {message.id for message in messages_h2}
        )
    ),
    "exact_json_structure": (
        isinstance(parsed_h2, dict)
        and set(parsed_h2) == set(expected_h2)
    ),
    "unchanged_fields_retained": (
        object_h2.get("company") == "Aurora"
        and type(object_h2.get("year")) is int
        and object_h2.get("year") == 2025
        and history_number_matches("revenue_million")
    ),
    "latest_profit_used": history_number_matches("net_profit_million"),
    "latest_margin_used": history_number_matches("net_margin_pct"),
    "no_tool_calls": len(calls_h2) == 0,
    "no_invalid_tool_calls": not any(
        message.invalid_tool_calls for message in new_ai_h2
    ),
    "final_answer_complete": (
        recall_turn_h2 is not None
        and isinstance(last_h2, AIMessage)
        and bool(answer_h2)
        and not last_h2.tool_calls
    ),
}

history_002_report = {
    "case_id": "history_002",
    "run_id": history_002_run_id,
    "sdk": "deepagents",
    "model": model.model_name,
    "versions": {
        name: version(name)
        for name in [
            "deepagents", "langchain", "langchain-openai",
            "langgraph", "langgraph-checkpoint",
        ]
    },
    "fixture_version": "session_history_correction_v1",
    "runtime_status": history_002_status,
    "error": history_002_error,
    "thread_id": history_thread_id,
    "depends_on_run_id": history_001_record["run_id"],
    "configuration": {
        "checkpointer": "InMemorySaver",
        "restart_tested": False,
        "recursion_limit": 20,
        "timeout_per_turn_s": 120,
    },
    "checks": checks_h2,
    "task_passed": (
        history_002_status == "completed"
        and all(checks_h2.values())
    ),
    "latency_s": round(history_002_latency_s, 2),
    "tool_call_count": len(calls_h2),
    "system_prompt": history_system_prompt,
    "turns": history_002_turns,
    "answer": parsed_h2,
    "raw_answer": answer_h2,
    "messages": [
        message.model_dump(mode="json")
        for message in messages_h2
    ],
    # เฉพาะข้อความ AI ที่เพิ่มในเคสนี้
    "usage_per_visible_ai_message": [
        message.usage_metadata for message in new_ai_h2
    ],
}

print("CHECKS:")
print(json.dumps(checks_h2, indent=2, ensure_ascii=False))

print("\nANSWER:")
print(json.dumps(parsed_h2, indent=2, ensure_ascii=False))

print("\nTask passed:", history_002_report["task_passed"])
print("Messages in state:", len(messages_h2))
print("New messages:", len(new_messages_h2))
print("Tool calls:", len(calls_h2))

CHECKS:
{
  "correction_acknowledged": true,
  "original_messages_retained": true,
  "exact_json_structure": true,
  "unchanged_fields_retained": true,
  "latest_profit_used": true,
  "latest_margin_used": true,
  "no_tool_calls": true,
  "no_invalid_tool_calls": true,
  "final_answer_complete": true
}

ANSWER:
{
  "company": "Aurora",
  "year": 2025,
  "revenue_million": 2400,
  "net_profit_million": 360,
  "net_margin_pct": 15
}

Task passed: True
Messages in state: 8
New messages: 4
Tool calls: 0


In [138]:
from pathlib import Path
from datetime import datetime, timezone
from copy import deepcopy
from uuid import uuid4
import json

history_records = [
    deepcopy(history_001_record),
    deepcopy(history_002_report),
]

assert [record["case_id"] for record in history_records] == [
    "history_001",
    "history_002",
]

assert all(record.get("run_id") for record in history_records)

# เพิ่ม metadata ของเคสแรกให้ระบุขอบเขตการทดลองชัดเจน
history_records[0].setdefault("configuration", {}).update({
    "checkpointer": "InMemorySaver",
    "restart_tested": False,
})

history_records[0]["system_prompt"] = history_system_prompt

history_records[0]["latency_s"] = round(
    history_records[0]["latency_turn_1_s"]
    + history_records[0]["latency_turn_2_s"],
    2,
)

shared_dir = Path("/workspace/shared")
assert shared_dir.is_dir(), "ไม่พบ /workspace/shared"

results_dir = shared_dir / "results" / "deepagents"
results_dir.mkdir(parents=True, exist_ok=True)

saved_at = datetime.now(timezone.utc)

history_checkpoint = {
    "schema_version": "1.0",
    "saved_at_utc": saved_at.isoformat(),
    "sdk": "deepagents",
    "category": "session_history",
    "results": history_records,
}

# ตรวจว่าแปลงเป็น JSON ได้ครบก่อนสร้างไฟล์
serialized = json.dumps(
    history_checkpoint,
    indent=2,
    ensure_ascii=False,
    allow_nan=False,
)

history_output_path = results_dir / (
    f"history_{saved_at:%Y%m%dT%H%M%SZ}_{uuid4().hex[:8]}.json"
)

with history_output_path.open("x", encoding="utf-8") as file:
    file.write(serialized)

# อ่านกลับเพื่อตรวจว่าบันทึกครบ
saved_history = json.loads(
    history_output_path.read_text(encoding="utf-8")
)

assert saved_history == history_checkpoint

print("Saved:", history_output_path)
print("Experiments:", len(saved_history["results"]))

for record in saved_history["results"]:
    print(
        f"- {record['case_id']}: "
        f"passed={record['task_passed']}"
    )

Saved: /workspace/shared/results/deepagents/history_20261006T092129Z_47d3075b.json
Experiments: 2
- history_001: passed=True
- history_002: passed=True


### history 003 (บันทึกข้อมูลใน SQLite)

บันทึกและเตรียมข้อมูลประวัติการสนทนาเพื่อใช้ตรวจการกู้คืนสถานะในขั้นตอนถัดไป

In [140]:
from pathlib import Path
import json

history_saved_path = Path(
    "/workspace/shared/results/deepagents/"
    "history_20261006T092129Z_47d3075b.json"
)

saved_history = json.loads(
    history_saved_path.read_text(encoding="utf-8")
)

assert saved_history["category"] == "session_history"
assert [r["case_id"] for r in saved_history["results"]] == [
    "history_001", "history_002"
]

for record in saved_history["results"]:
    print(record["case_id"], "| Passed:", record["task_passed"])

print("JSON file verified.")

history_001 | Passed: True
history_002 | Passed: True
JSON file verified.


In [141]:
import subprocess
import sys
import tempfile
from importlib.metadata import version

protected_packages = [
    "deepagents",
    "langchain",
    "langchain-core",
    "langchain-openai",
    "langgraph",
    "langgraph-checkpoint",
]

with tempfile.TemporaryDirectory() as temp_dir:
    constraints_path = Path(temp_dir) / "constraints.txt"
    constraints_path.write_text(
        "\n".join(
            f"{name}=={version(name)}"
            for name in protected_packages
        ),
        encoding="utf-8",
    )

    subprocess.run(
        [
            sys.executable, "-m", "pip", "install",
            "langgraph-checkpoint-sqlite",
            "-c", str(constraints_path),
        ],
        check=True,
    )

print("SQLite checkpointer:", version("langgraph-checkpoint-sqlite"))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [langgraph-checkpoint-sqlite]checkpoint-sqlite]


SQLite checkpointer: 3.1.1


In [142]:
import asyncio
import json
import os
import time

from datetime import datetime, timezone
from uuid import uuid4
from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver
from langchain_core.messages import AIMessage
from deepagents import create_deep_agent

shared_dir = Path("/workspace/shared")
assert shared_dir.is_dir()

restart_run_id = uuid4().hex
restart_thread_id = f"history-003-{restart_run_id}"

restart_dir = (
    shared_dir / "state" / "deepagents" / restart_run_id
)
restart_dir.mkdir(parents=True, exist_ok=False)

restart_db_path = restart_dir / "checkpoints.sqlite"
restart_manifest_path = restart_dir / "manifest.json"

restart_system_prompt = (
    "Use only information from the current conversation thread. "
    "Do not call tools or access files. "
    "Follow the requested response format exactly."
)

restart_seed_prompt = """
จดจำข้อมูลสำหรับการสนทนานี้:

บริษัท: Vega
ปีงบประมาณ: 2025
รายได้รวม: 3200 ล้านบาท
กำไรสุทธิ: 480 ล้านบาท
อัตรากำไรสุทธิ: 15%
รหัสอ้างอิง: VEGA-73-KP

ตอบเพียงคำว่า CONTEXT_SAVED
"""

restart_config = {
    "configurable": {"thread_id": restart_thread_id},
    "recursion_limit": 20,
}

restart_versions = {
    name: version(name)
    for name in [
        "deepagents", "langchain", "langchain-openai",
        "langgraph", "langgraph-checkpoint",
        "langgraph-checkpoint-sqlite",
    ]
}

restart_started_at = time.perf_counter()

async with AsyncSqliteSaver.from_conn_string(
    str(restart_db_path)
) as saver:
    restart_agent = create_deep_agent(
        model=model,
        tools=[],
        system_prompt=restart_system_prompt,
        checkpointer=saver,
    )

    restart_seed_result = await asyncio.wait_for(
        restart_agent.ainvoke(
            {
                "messages": [
                    {"role": "user", "content": restart_seed_prompt}
                ]
            },
            config=restart_config,
        ),
        timeout=120,
    )

    # อ่าน checkpoint กลับมาตรวจ ก่อนปิด connection
    restart_snapshot = await restart_agent.aget_state(
        restart_config
    )

restart_seed_latency_s = time.perf_counter() - restart_started_at

restart_seed_messages = restart_seed_result["messages"]
restart_seed_last = restart_seed_messages[-1]

assert isinstance(restart_seed_last, AIMessage)
assert restart_seed_last.text.strip() == "CONTEXT_SAVED"
assert not any(
    message.tool_calls
    for message in restart_seed_messages
    if isinstance(message, AIMessage)
)

checkpoint_messages = restart_snapshot.values.get("messages", [])

assert any(
    message.type == "human"
    and message.content == restart_seed_prompt
    for message in checkpoint_messages
)
assert not restart_snapshot.next, "ยังมีงานค้างใน checkpoint"
assert restart_db_path.is_file()

restart_manifest = {
    "case_id": "history_003",
    "run_id": restart_run_id,
    "sdk": "deepagents",
    "model": model.model_name,
    "versions_before_restart": restart_versions,
    "thread_id": restart_thread_id,
    "db_path": str(restart_db_path),
    "system_prompt": restart_system_prompt,
    "process_id_before": os.getpid(),
    "prepared_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": "prepared",
    "task_passed": None,
    "seed_latency_s": round(restart_seed_latency_s, 2),
    "seed_message_ids": [
        message.id for message in restart_seed_messages
    ],
    "seed_trace": [
        message.model_dump(mode="json")
        for message in restart_seed_messages
    ],
}

serialized_manifest = json.dumps(
    restart_manifest,
    indent=2,
    ensure_ascii=False,
    allow_nan=False,
)

with restart_manifest_path.open("x", encoding="utf-8") as file:
    file.write(serialized_manifest)

print("Response:", restart_seed_last.text.strip())
print("Checkpoint messages:", len(checkpoint_messages))
print("Database:", restart_db_path)
print("Manifest:", restart_manifest_path)
print("Database connection closed.")
print("Prepared for restart — test not yet completed.")

Response: CONTEXT_SAVED
Checkpoint messages: 2
Database: /workspace/shared/state/deepagents/35af7261a4844c5f9736e8c17ae299c9/checkpoints.sqlite
Manifest: /workspace/shared/state/deepagents/35af7261a4844c5f9736e8c17ae299c9/manifest.json
Database connection closed.
Prepared for restart — test not yet completed.


In [143]:
restart_probe = "this_variable_must_disappear"
print("Restart probe created.")

Restart probe created.


In [149]:
# assert "restart_probe" not in globals(), (
#     "ตัวแปรเดิมยังอยู่ — kernel ยังไม่ได้ restart"
# )

# kernel_restart_verified = True
# print("Previous kernel variable is gone.")

In [145]:
import asyncio
import json
import math
import os
import time

from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version
from uuid import uuid4

from langchain_openai import ChatOpenAI
from langchain_core.messages import AIMessage
from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver
from deepagents import create_deep_agent

manifest_path = Path(
    "/workspace/shared/state/deepagents/"
    "03b03be0cdfb4049ae41f470212fc52c/manifest.json"
)

manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
db_path = Path(manifest["db_path"])

assert manifest["case_id"] == "history_003"
assert db_path.is_file(), "ไม่พบฐานข้อมูล checkpoint"

pid_before = manifest["process_id_before"]
pid_after = os.getpid()

print("PID before:", pid_before)
print("PID after:", pid_after)

assert pid_after != pid_before, (
    "PID ยังเหมือนเดิม ยังยืนยันการเปลี่ยน process ไม่ได้ "
    "ตรวจว่าได้ Restart Kernel แล้ว"
)

versions_after = {
    name: version(name)
    for name in manifest["versions_before_restart"]
}

assert versions_after == manifest["versions_before_restart"], (
    "เวอร์ชันแพ็กเกจเปลี่ยนระหว่างการทดสอบ"
)

print("New process verified.")
print("Package versions unchanged.")
print("Thread:", manifest["thread_id"])

PID before: 18
PID after: 85
New process verified.
Package versions unchanged.
Thread: history-003-03b03be0cdfb4049ae41f470212fc52c


In [146]:
assert os.environ.get("OPENAI_API_KEY"), (
    "ยังไม่มี OPENAI_API_KEY ใน kernel นี้ "
    "ให้โหลด environment ด้วยวิธีเดิมก่อน"
)

restored_model = ChatOpenAI(
    model=manifest["model"],
    use_responses_api=True,
    timeout=60,
    max_retries=0,
)

print("Model:", restored_model.model_name)

Model: gpt-6-luna


In [147]:
resume_prompt = """
จากข้อมูลที่ให้ไว้ก่อนหน้านี้ในบทสนทนานี้
ตอบ JSON object เท่านั้น โดยมี keys:
company, year, revenue_million, net_profit_million,
net_margin_pct, reference_code

ใช้ข้อมูลและรหัสอ้างอิงจากประวัติเดิม
ห้ามเดา ห้ามเพิ่ม keys และห้ามใช้ tools
หากไม่พบข้อมูลของ field ใด ให้ใช้ null
"""

resume_config = {
    "configurable": {"thread_id": manifest["thread_id"]},
    "recursion_limit": 20,
}

resume_result = None
resume_error = None
resume_status = "running"
restored_messages = []
resume_agent_s = None

resume_started_at = time.perf_counter()

try:
    async with AsyncSqliteSaver.from_conn_string(
        str(db_path)
    ) as saver:
        restored_agent = create_deep_agent(
            model=restored_model,
            tools=[],
            system_prompt=manifest["system_prompt"],
            checkpointer=saver,
        )

        snapshot = await restored_agent.aget_state(resume_config)
        restored_messages = snapshot.values.get("messages", [])

        restored_ids = {message.id for message in restored_messages}
        seed_ids = set(manifest["seed_message_ids"])

        assert seed_ids and None not in seed_ids
        assert seed_ids.issubset(restored_ids), (
            "กู้คืนข้อความเดิมจาก checkpoint ได้ไม่ครบ"
        )
        assert not snapshot.next, "Checkpoint มีงานที่ยังไม่จบ"

        print("Messages restored before model call:", len(restored_messages))

        agent_started_at = time.perf_counter()
        try:
            resume_result = await asyncio.wait_for(
                restored_agent.ainvoke(
                    {
                        "messages": [
                            {"role": "user", "content": resume_prompt}
                        ]
                    },
                    config=resume_config,
                ),
                timeout=120,
            )
        finally:
            resume_agent_s = time.perf_counter() - agent_started_at

    resume_status = "completed"

except Exception as exc:
    resume_status = "failed"
    resume_error = {
        "type": type(exc).__name__,
        "message": str(exc),
    }

finally:
    resume_latency_s = time.perf_counter() - resume_started_at

print("Runtime status:", resume_status)
print("Resume latency:", round(resume_latency_s, 2), "seconds")

if resume_error:
    print("Error:", resume_error)

Messages restored before model call: 4
Runtime status: completed
Resume latency: 4.31 seconds


In [148]:
resume_messages = (
    resume_result.get("messages", [])
    if resume_result is not None else []
)

resume_last = resume_messages[-1] if resume_messages else None
resume_answer = (
    resume_last.text.strip()
    if isinstance(resume_last, AIMessage) else ""
)

try:
    resume_parsed = json.loads(resume_answer)
except json.JSONDecodeError:
    resume_parsed = None

expected_resume = {
    "company": "Vega",
    "year": 2025,
    "revenue_million": 3200,
    "net_profit_million": 480,
    "net_margin_pct": 15,
    "reference_code": "VEGA-73-KP",
}

actual_resume = (
    resume_parsed if isinstance(resume_parsed, dict) else {}
)

def resume_value_matches(key, expected):
    actual = actual_resume.get(key)

    if type(expected) is int:
        return (
            type(actual) in (int, float)
            and math.isfinite(actual)
            and actual == expected
        )

    return type(actual) is str and actual == expected

value_checks = {
    key: resume_value_matches(key, expected)
    for key, expected in expected_resume.items()
}

restored_ids = {message.id for message in restored_messages}
new_ai_messages = [
    message for message in resume_messages
    if isinstance(message, AIMessage)
    and message.id not in restored_ids
]

resume_tool_calls = [
    call for message in new_ai_messages
    for call in message.tool_calls
]

resume_checks = {
    "process_changed": pid_after != pid_before,
    "versions_unchanged": (
        versions_after == manifest["versions_before_restart"]
    ),
    "seed_messages_restored_before_call": (
        bool(manifest["seed_message_ids"])
        and set(manifest["seed_message_ids"]).issubset(restored_ids)
    ),
    "exact_json_structure": (
        isinstance(resume_parsed, dict)
        and set(resume_parsed) == set(expected_resume)
    ),
    "all_values_correct": all(value_checks.values()),
    "no_tool_calls": len(resume_tool_calls) == 0,
    "no_invalid_tool_calls": not any(
        message.invalid_tool_calls for message in new_ai_messages
    ),
    "final_answer_complete": (
        isinstance(resume_last, AIMessage)
        and bool(resume_answer)
        and not resume_last.tool_calls
    ),
}

history_003_report = {
    "case_id": "history_003",
    "run_id": manifest["run_id"],
    "sdk": "deepagents",
    "model": manifest["model"],
    "versions": versions_after,
    "versions_before_restart": manifest["versions_before_restart"],
    "runtime_status": resume_status,
    "error": resume_error,
    "thread_id": manifest["thread_id"],
    "configuration": {
        "checkpointer": "AsyncSqliteSaver",
        "restart_type": "jupyter_kernel_restart",
        "process_id_before": pid_before,
        "process_id_after": pid_after,
        "recursion_limit": 20,
        "timeout_s": 120,
    },
    "checks": resume_checks,
    "value_checks": value_checks,
    "task_passed": (
        resume_status == "completed"
        and all(resume_checks.values())
    ),
    "seed_latency_s": manifest["seed_latency_s"],
    "resume_latency_s": round(resume_latency_s, 2),
    "resume_agent_s": (
        round(resume_agent_s, 2)
        if resume_agent_s is not None else None
    ),
    # เวลาทำงานสองช่วง ไม่รวมเวลารอผู้ใช้ restart
    "latency_s": round(
        manifest["seed_latency_s"] + resume_latency_s, 2
    ),
    "system_prompt": manifest["system_prompt"],
    "resume_prompt": resume_prompt,
    "answer": resume_parsed,
    "raw_answer": resume_answer,
    "tool_call_count": len(resume_tool_calls),
    "seed_trace": manifest["seed_trace"],
    "restored_message_ids": [
        message.id for message in restored_messages
    ],
    "messages_after_resume": [
        message.model_dump(mode="json")
        for message in resume_messages
    ],
    "usage_after_restart": [
        message.usage_metadata for message in new_ai_messages
    ],
}

results_dir = Path("/workspace/shared/results/deepagents")
results_dir.mkdir(parents=True, exist_ok=True)

saved_at = datetime.now(timezone.utc)
report_path = results_dir / (
    f"history_003_{saved_at:%Y%m%dT%H%M%SZ}_{uuid4().hex[:8]}.json"
)

payload = {
    "schema_version": "1.0",
    "saved_at_utc": saved_at.isoformat(),
    "sdk": "deepagents",
    "category": "session_history",
    "results": [history_003_report],
}

serialized = json.dumps(
    payload, indent=2, ensure_ascii=False, allow_nan=False
)

with report_path.open("x", encoding="utf-8") as file:
    file.write(serialized)

print("ANSWER:")
print(resume_answer)

print("\nCHECKS:")
print(json.dumps(resume_checks, indent=2, ensure_ascii=False))

print("\nTask passed:", history_003_report["task_passed"])
print("Saved:", report_path)

ANSWER:
{"company":"Vega","year":2025,"revenue_million":3200,"net_profit_million":480,"net_margin_pct":15,"reference_code":"VEGA-73-KP"}

CHECKS:
{
  "process_changed": true,
  "versions_unchanged": true,
  "seed_messages_restored_before_call": true,
  "exact_json_structure": true,
  "all_values_correct": true,
  "no_tool_calls": true,
  "no_invalid_tool_calls": true,
  "final_answer_complete": true
}

Task passed: True
Saved: /workspace/shared/results/deepagents/history_003_20261006T092315Z_e048d81d.json


<hr>

# 8 Error handling

หัวข้อนี้ตรวจพฤติกรรมของ agent เมื่อเครื่องมือส่งข้อผิดพลาด ทั้งกรณีที่ไม่ควรลองซ้ำและกรณีที่ลองซ้ำได้แต่มีจำนวนครั้งจำกัด โดยตรวจคำขอ จำนวนครั้งที่เรียก และรูปแบบคำตอบสุดท้าย


### Error01 ข้อผิดพลาดที่ไม่อนุญาตให้ลองซ้ำ

กรณีแรกส่งข้อผิดพลาดแบบถาวรกลับจากเครื่องมือ และตรวจว่า agent หยุดเรียกซ้ำพร้อมรายงานสถานะและรหัสข้อผิดพลาดได้ถูกต้อง

In [151]:
import asyncio
import json
import time

from copy import deepcopy
from uuid import uuid4
from importlib.metadata import version

from langchain_core.tools import tool
from langchain_core.messages import AIMessage
from deepagents import create_deep_agent

In [152]:
error_tool_log = []

@tool
def fetch_restricted_statement(company_name: str, year: int) -> dict:
    """Request a company's annual financial statement."""
    result = {
        "error": {
            "code": "ACCESS_DENIED",
            "retryable": False,
            "message": "The current account cannot access this statement.",
        }
    }
    error_tool_log.append({
        "tool": "fetch_restricted_statement",
        "arguments": {
            "company_name": company_name,
            "year": year,
        },
        "output": deepcopy(result),
    })
    return result

In [153]:
error_system_prompt = """
Use the supplied tool to retrieve financial statements.
If the tool returns an error, follow its retryable flag.
Do not invent financial data.
Do not delegate or use other tools.
Return only the JSON requested by the user.
"""

error_prompt = """
ขอดูงบปี 2025 ของบริษัท Aurora โดยเรียกเครื่องมือที่ให้

ข้อกำหนด:
- หากเครื่องมือคืน retryable=false ห้ามเรียกคำขอนั้นซ้ำ
- หากดึงข้อมูลไม่ได้ ให้รายงานข้อผิดพลาดตามจริง
- ห้ามใช้ตัวเลขจากความรู้เดิมหรือเดาตัวเลข
- ตอบ JSON object เท่านั้น มี keys ตามนี้:

{
  "status": "ok หรือ unavailable",
  "company_name": "Aurora",
  "year": 2025,
  "error_code": "รหัสข้อผิดพลาดจากเครื่องมือ หรือ null เมื่อสำเร็จ",
  "statement": "ข้อมูลงบจากเครื่องมือ หรือ null เมื่อไม่สำเร็จ"
}
"""

error_agent = create_deep_agent(
    model=restored_model,
    tools=[fetch_restricted_statement],
    system_prompt=error_system_prompt,
)

print("Non-retryable error test ready.")

Non-retryable error test ready.


In [154]:
error_tool_log.clear()

error_run_id = uuid4().hex
error_result = None
error_runtime_status = "running"
error_runtime_exception = None

error_versions = {
    name: version(name)
    for name in ["deepagents", "langchain", "langchain-openai"]
}

started_at = time.perf_counter()

try:
    error_result = await asyncio.wait_for(
        error_agent.ainvoke(
            {"messages": [{"role": "user", "content": error_prompt}]},
            config={"recursion_limit": 12},
        ),
        timeout=120,
    )
    error_runtime_status = "completed"

except Exception as exc:
    error_runtime_status = "failed"
    error_runtime_exception = {
        "type": type(exc).__name__,
        "message": str(exc),
    }

finally:
    error_latency_s = time.perf_counter() - started_at

error_trace = deepcopy(error_tool_log)

print("Runtime status:", error_runtime_status)
print("Latency:", round(error_latency_s, 2), "seconds")

if error_runtime_exception:
    print("Error:", error_runtime_exception)

print("\nTOOL LOG:")
print(json.dumps(error_trace, indent=2, ensure_ascii=False))

Runtime status: completed
Latency: 4.26 seconds

TOOL LOG:
[
  {
    "tool": "fetch_restricted_statement",
    "arguments": {
      "company_name": "Aurora",
      "year": 2025
    },
    "output": {
      "error": {
        "code": "ACCESS_DENIED",
        "retryable": false,
        "message": "The current account cannot access this statement."
      }
    }
  }
]


In [155]:
error_messages = (
    error_result.get("messages", [])
    if error_result is not None else []
)
error_last = error_messages[-1] if error_messages else None

error_answer = (
    error_last.text.strip()
    if isinstance(error_last, AIMessage) else ""
)

try:
    error_parsed = json.loads(error_answer)
except json.JSONDecodeError:
    error_parsed = None

error_ai_messages = [
    msg for msg in error_messages if isinstance(msg, AIMessage)
]
error_calls = [
    call for msg in error_ai_messages for call in msg.tool_calls
]

expected_error_answer = {
    "status": "unavailable",
    "company_name": "Aurora",
    "year": 2025,
    "error_code": "ACCESS_DENIED",
    "statement": None,
}

error_checks = {
    "tool_executed_once": len(error_trace) == 1,
    "correct_request": (
        len(error_trace) == 1
        and error_trace[0]["arguments"] == {
            "company_name": "Aurora",
            "year": 2025,
        }
    ),
    "nonretryable_error_received": (
        len(error_trace) == 1
        and error_trace[0]["output"]["error"]["code"] == "ACCESS_DENIED"
        and error_trace[0]["output"]["error"]["retryable"] is False
    ),
    "only_expected_tool_call": (
        len(error_calls) == 1
        and error_calls[0]["name"] == "fetch_restricted_statement"
    ),
    "exact_answer": (
        isinstance(error_parsed, dict)
        and error_parsed == expected_error_answer
        and type(error_parsed.get("year")) is int
    ),
    "no_invalid_tool_calls": not any(
        msg.invalid_tool_calls for msg in error_ai_messages
    ),
    "final_answer_complete": (
        isinstance(error_last, AIMessage)
        and bool(error_answer)
        and not error_last.tool_calls
    ),
}

error_001_report = {
    "case_id": "error_001",
    "run_id": error_run_id,
    "sdk": "deepagents",
    "model": restored_model.model_name,
    "versions": error_versions,
    "fixture_version": "nonretryable_error_v1",
    "failure_mode": "nonretryable_error_in_tool_result",
    "runtime_status": error_runtime_status,
    "runtime_exception": error_runtime_exception,
    "checks": error_checks,
    "task_passed": (
        error_runtime_status == "completed"
        and all(error_checks.values())
    ),
    "latency_s": round(error_latency_s, 2),
    "tool_call_count": len(error_calls),
    "system_prompt": error_system_prompt,
    "prompt": error_prompt,
    "answer": error_parsed,
    "raw_answer": error_answer,
    "tool_trace": error_trace,
    "messages": [
        msg.model_dump(mode="json") for msg in error_messages
    ],
    "usage_per_visible_ai_message": [
        msg.usage_metadata for msg in error_ai_messages
    ],
}

print("ANSWER:")
print(error_answer)

print("\nCHECKS:")
print(json.dumps(error_checks, indent=2, ensure_ascii=False))

print("\nTask passed:", error_001_report["task_passed"])

ANSWER:
{"status":"unavailable","company_name":"Aurora","year":2025,"error_code":"ACCESS_DENIED","statement":null}

CHECKS:
{
  "tool_executed_once": true,
  "correct_request": true,
  "nonretryable_error_received": true,
  "only_expected_tool_call": true,
  "exact_answer": true,
  "no_invalid_tool_calls": true,
  "final_answer_complete": true
}

Task passed: True


### Error02 จำกัดจำนวนครั้งเมื่อลองใหม่

กรณีที่สองส่งข้อผิดพลาดชั่วคราวและกำหนดให้ลองซ้ำได้หนึ่งครั้ง จึงตรวจว่า agent เรียกเครื่องมือรวมไม่เกินสองครั้งและแจ้งจำนวนครั้งจริงในคำตอบ

In [156]:
error_001_record = deepcopy(error_001_report)

retry_limit_log = []

@tool
def fetch_temporarily_unavailable_statement(
    company_name: str,
    year: int,
) -> dict:
    """Request an annual financial statement from the data service."""
    result = {
        "error": {
            "code": "TEMPORARY_UNAVAILABLE",
            "retryable": True,
            "message": "The financial data service is temporarily unavailable.",
        }
    }

    retry_limit_log.append({
        "tool": "fetch_temporarily_unavailable_statement",
        "arguments": {
            "company_name": company_name,
            "year": year,
        },
        "output": deepcopy(result),
    })

    return result


retry_limit_system_prompt = """
Use the supplied tool to retrieve financial statements.
Follow the retry limit specified by the user.
Do not invent financial data.
Do not delegate or use other tools.
Return only the requested JSON.
"""

retry_limit_prompt = """
ขอดูงบปี 2025 ของบริษัท Aurora โดยเรียกเครื่องมือที่ให้

นโยบายจัดการข้อผิดพลาด:
- หากเครื่องมือคืน retryable=true ให้ลองคำขอเดิมซ้ำอีก 1 ครั้ง
- อนุญาตให้เรียกเครื่องมือรวมสูงสุด 2 ครั้ง
- หากครั้งที่สองยังผิดพลาด ต้องหยุดและรายงานว่าไม่สามารถดึงข้อมูลได้
- ห้ามเดาตัวเลขหรือใช้ข้อมูลจากความรู้เดิม

ตอบ JSON object เท่านั้น มี keys:
{
  "status": "ok หรือ unavailable",
  "company_name": "Aurora",
  "year": 2025,
  "error_code": "รหัสข้อผิดพลาดล่าสุด หรือ null เมื่อสำเร็จ",
  "attempts": 0,
  "statement": "ข้อมูลงบ หรือ null เมื่อไม่สำเร็จ"
}

attempts ต้องเป็นจำนวนครั้งที่เรียกเครื่องมือจริง
"""

retry_limit_agent = create_deep_agent(
    model=restored_model,
    tools=[fetch_temporarily_unavailable_statement],
    system_prompt=retry_limit_system_prompt,
)

print("Retry-limit test ready.")

Retry-limit test ready.


In [157]:
retry_limit_log.clear()

retry_limit_run_id = uuid4().hex
retry_limit_result = None
retry_limit_status = "running"
retry_limit_exception = None

retry_limit_versions = {
    name: version(name)
    for name in ["deepagents", "langchain", "langchain-openai"]
}

started_at = time.perf_counter()

try:
    retry_limit_result = await asyncio.wait_for(
        retry_limit_agent.ainvoke(
            {
                "messages": [
                    {"role": "user", "content": retry_limit_prompt}
                ]
            },
            config={"recursion_limit": 16},
        ),
        timeout=120,
    )
    retry_limit_status = "completed"

except Exception as exc:
    retry_limit_status = "failed"
    retry_limit_exception = {
        "type": type(exc).__name__,
        "message": str(exc),
    }

finally:
    retry_limit_latency_s = time.perf_counter() - started_at

retry_limit_trace = deepcopy(retry_limit_log)

print("Runtime status:", retry_limit_status)
print("Latency:", round(retry_limit_latency_s, 2), "seconds")

if retry_limit_exception:
    print("Error:", retry_limit_exception)

print("\nTOOL LOG:")
print(json.dumps(retry_limit_trace, indent=2, ensure_ascii=False))

Runtime status: completed
Latency: 6.81 seconds

TOOL LOG:
[
  {
    "tool": "fetch_temporarily_unavailable_statement",
    "arguments": {
      "company_name": "Aurora",
      "year": 2025
    },
    "output": {
      "error": {
        "code": "TEMPORARY_UNAVAILABLE",
        "retryable": true,
        "message": "The financial data service is temporarily unavailable."
      }
    }
  },
  {
    "tool": "fetch_temporarily_unavailable_statement",
    "arguments": {
      "company_name": "Aurora",
      "year": 2025
    },
    "output": {
      "error": {
        "code": "TEMPORARY_UNAVAILABLE",
        "retryable": true,
        "message": "The financial data service is temporarily unavailable."
      }
    }
  }
]


In [158]:
retry_messages = (
    retry_limit_result.get("messages", [])
    if retry_limit_result is not None else []
)
retry_last = retry_messages[-1] if retry_messages else None

retry_answer = (
    retry_last.text.strip()
    if isinstance(retry_last, AIMessage) else ""
)

try:
    retry_parsed = json.loads(retry_answer)
except json.JSONDecodeError:
    retry_parsed = None

retry_ai_messages = [
    message for message in retry_messages
    if isinstance(message, AIMessage)
]

retry_calls = [
    {"message_index": index, **call}
    for index, message in enumerate(retry_messages)
    if isinstance(message, AIMessage)
    for call in message.tool_calls
]

expected_request = {
    "company_name": "Aurora",
    "year": 2025,
}

expected_retry_answer = {
    "status": "unavailable",
    "company_name": "Aurora",
    "year": 2025,
    "error_code": "TEMPORARY_UNAVAILABLE",
    "attempts": 2,
    "statement": None,
}

retry_checks = {
    "exactly_two_executions": len(retry_limit_trace) == 2,
    "same_correct_request": (
        len(retry_limit_trace) == 2
        and all(
            entry["arguments"] == expected_request
            for entry in retry_limit_trace
        )
    ),
    "both_attempts_failed": (
        len(retry_limit_trace) == 2
        and all(
            entry["output"]["error"]["code"] == "TEMPORARY_UNAVAILABLE"
            and entry["output"]["error"]["retryable"] is True
            for entry in retry_limit_trace
        )
    ),
    "only_expected_tool_calls": (
        len(retry_calls) == 2
        and all(
            call["name"] == "fetch_temporarily_unavailable_statement"
            for call in retry_calls
        )
    ),
    "retry_after_first_error": (
        len(retry_calls) == 2
        and any(
            message.type == "tool"
            and getattr(message, "tool_call_id", None)
                == retry_calls[0].get("id")
            for message in retry_messages[
                retry_calls[0]["message_index"] + 1:
                retry_calls[1]["message_index"]
            ]
        )
    ),
    "exact_answer": (
        isinstance(retry_parsed, dict)
        and retry_parsed == expected_retry_answer
        and type(retry_parsed.get("year")) is int
        and type(retry_parsed.get("attempts")) is int
    ),
    "no_invalid_tool_calls": not any(
        message.invalid_tool_calls for message in retry_ai_messages
    ),
    "final_answer_complete": (
        isinstance(retry_last, AIMessage)
        and bool(retry_answer)
        and not retry_last.tool_calls
    ),
}

error_002_report = {
    "case_id": "error_002",
    "run_id": retry_limit_run_id,
    "sdk": "deepagents",
    "model": restored_model.model_name,
    "versions": retry_limit_versions,
    "fixture_version": "retry_exhaustion_v1",
    "failure_mode": "retryable_error_in_tool_result",
    "configuration": {
        "max_tool_attempts_by_prompt": 2,
        "recursion_limit": 16,
        "timeout_s": 120,
    },
    "runtime_status": retry_limit_status,
    "runtime_exception": retry_limit_exception,
    "checks": retry_checks,
    "task_passed": (
        retry_limit_status == "completed"
        and all(retry_checks.values())
    ),
    "latency_s": round(retry_limit_latency_s, 2),
    "tool_call_count": len(retry_calls),
    "system_prompt": retry_limit_system_prompt,
    "prompt": retry_limit_prompt,
    "answer": retry_parsed,
    "raw_answer": retry_answer,
    "tool_trace": retry_limit_trace,
    "messages": [
        message.model_dump(mode="json")
        for message in retry_messages
    ],
    "usage_per_visible_ai_message": [
        message.usage_metadata for message in retry_ai_messages
    ],
}

print("ANSWER:")
print(retry_answer)

print("\nCHECKS:")
print(json.dumps(retry_checks, indent=2, ensure_ascii=False))

print("\nTask passed:", error_002_report["task_passed"])

ANSWER:
{"status":"unavailable","company_name":"Aurora","year":2025,"error_code":"TEMPORARY_UNAVAILABLE","attempts":2,"statement":null}

CHECKS:
{
  "exactly_two_executions": true,
  "same_correct_request": true,
  "both_attempts_failed": true,
  "only_expected_tool_calls": true,
  "retry_after_first_error": true,
  "exact_answer": true,
  "no_invalid_tool_calls": true,
  "final_answer_complete": true
}

Task passed: True


### บันทึกผล Error Handling

รวมระเบียนของกรณีข้อผิดพลาดและเขียนเป็นไฟล์ JSON จากนั้นอ่านกลับเพื่อตรวจว่าข้อมูลที่จัดเก็บตรงกับข้อมูลต้นทาง

In [159]:
from pathlib import Path
from datetime import datetime, timezone
from copy import deepcopy
from uuid import uuid4
import json

error_records = [
    deepcopy(error_001_record),
    deepcopy(error_002_report),
]

assert [record["case_id"] for record in error_records] == [
    "error_001",
    "error_002",
]

run_ids = [record["run_id"] for record in error_records]
assert all(run_ids)
assert len(set(run_ids)) == len(run_ids)

shared_dir = Path("/workspace/shared")
assert shared_dir.is_dir(), "ไม่พบ /workspace/shared"

results_dir = shared_dir / "results" / "deepagents"
results_dir.mkdir(parents=True, exist_ok=True)

saved_at = datetime.now(timezone.utc)

error_checkpoint = {
    "schema_version": "1.0",
    "saved_at_utc": saved_at.isoformat(),
    "sdk": "deepagents",
    "category": "error_handling",
    "results": error_records,
}

# ตรวจว่า serialize ได้ก่อนสร้างไฟล์
serialized = json.dumps(
    error_checkpoint,
    indent=2,
    ensure_ascii=False,
    allow_nan=False,
)

error_output_path = results_dir / (
    f"error_handling_{saved_at:%Y%m%dT%H%M%SZ}_{uuid4().hex[:8]}.json"
)

with error_output_path.open("x", encoding="utf-8") as file:
    file.write(serialized)

# อ่านกลับเพื่อตรวจว่าข้อมูลครบ
saved_errors = json.loads(
    error_output_path.read_text(encoding="utf-8")
)

assert saved_errors == error_checkpoint

print("Saved:", error_output_path)
print("Experiments:", len(saved_errors["results"]))

for record in saved_errors["results"]:
    print(
        f"- {record['case_id']}: "
        f"passed={record['task_passed']}, "
        f"tool_calls={record['tool_call_count']}, "
        f"latency={record['latency_s']:.2f}s"
    )

print("JSON file verified.")

Saved: /workspace/shared/results/deepagents/error_handling_20261006T094709Z_cc80af96.json
Experiments: 2
- error_001: passed=True, tool_calls=1, latency=4.26s
- error_002: passed=True, tool_calls=2, latency=6.81s
JSON file verified.


<hr>

# 9 สรุปและสำรวจผล Benchmark

อ่านไฟล์ผลการทดลองที่บันทึกไว้ รวมรายการซ้ำและปัญหาการอ่านไฟล์ แล้วแสดงสถานะของแต่ละเคสในตารางเพื่อให้ตรวจภาพรวมได้สะดวก

### สร้างสรุปผลสำหรับนำไปใช้ต่อ

เซลล์สุดท้ายจัดกลุ่มผลตามเคส นับสถานะ และบันทึกสรุปพร้อมหมายเหตุเกี่ยวกับข้อจำกัดของการทดลองเป็นไฟล์ JSON

In [162]:
from collections import Counter
from IPython.display import Markdown, display

saved_runs = {}
duplicate_exports = 0
load_issues = []

for path in sorted(results_dir.glob("*.json")):
    try:
        payload = json.loads(path.read_text(encoding="utf-8"))
        records = payload.get("results")

        if not isinstance(records, list):
            load_issues.append((path.name, "ไม่มี results แบบ list"))
            continue

        for index, record in enumerate(records):
            if not isinstance(record, dict):
                load_issues.append((path.name, f"results[{index}] ไม่ใช่ object"))
                continue

            sdk = record.get("sdk", payload.get("sdk"))
            if sdk != "deepagents":
                continue

            run_id = record.get("run_id")
            # ไม่มี run_id จะไม่เดาว่าเป็น run เดียวกัน
            key = (sdk, run_id) if run_id else (path.name, index)

            if key in saved_runs:
                duplicate_exports += 1
                # ไม่เลือกผลที่อาจแก้ไขแล้วโดยเงียบ ๆ
                if saved_runs[key]["record"] != record:
                    load_issues.append((
                        path.name,
                        f"run_id ซ้ำแต่เนื้อหาต่างกัน: {run_id}",
                    ))
                continue

            saved_runs[key] = {
                "record": record,
                "file": path.name,
            }

    except (OSError, ValueError, AttributeError) as exc:
        load_issues.append((path.name, str(exc)))

status_counts = Counter()
table = [
    "| Case | ผลตรวจ |",
    "|---|---|",
]

for item in sorted(
    saved_runs.values(),
    key=lambda item: (
        item["record"].get("case_id", ""),
        item["record"].get("run_id", ""),
    ),
):
    record = item["record"]
    passed = record.get("task_passed")

    status = (
        "ผ่าน" if passed is True
        else "ไม่ผ่าน" if passed is False
        else "รอตรวจ"
    )

    status_counts[status] += 1
    table.append(f"| {record.get('case_id', 'ไม่ระบุ')} | {status} |")

display(Markdown("\n".join(table)))

print("Unique saved runs:", len(saved_runs))
print("Status counts:", dict(status_counts))
print("Duplicate exports:", duplicate_exports)

if load_issues:
    print("\nรายการที่ต้องตรวจ:")
    for filename, issue in load_issues:
        print(f"- {filename}: {issue}")
else:
    print("All scanned result files read successfully.")

| Case | ผลตรวจ |
|---|---|
| error_001 | ผ่าน |
| error_002 | ผ่าน |
| history_001 | ผ่าน |
| history_002 | ผ่าน |
| history_003 | ผ่าน |
| long_context_001 | ผ่าน |
| long_context_002 | ผ่าน |
| math_001 | ผ่าน |
| math_002 | ผ่าน |
| math_003 | ผ่าน |
| planning_001 | ผ่าน |
| planning_002 | ผ่าน |
| web_native_001 | รอตรวจ |
| web_native_001 | รอตรวจ |
| web_native_002 | ไม่ผ่าน |
| web_native_002 | รอตรวจ |
| web_native_002 | ไม่ผ่าน |

Unique saved runs: 17
Status counts: {'ผ่าน': 12, 'รอตรวจ': 3, 'ไม่ผ่าน': 2}
Duplicate exports: 1
All scanned result files read successfully.


In [163]:
from pathlib import Path
from datetime import datetime, timezone
from uuid import uuid4
from collections import Counter
import json

assert saved_runs, "กรุณารัน cell สำรวจผลทดลองก่อน"
assert not load_issues, "มีรายการอ่านไฟล์หรือ run_id ขัดแย้งที่ต้องตรวจ"

case_summaries = []

for item in saved_runs.values():
    record = item["record"]
    passed = record.get("task_passed")

    status = (
        "passed" if passed is True
        else "failed" if passed is False
        else "pending_review"
    )

    case_summaries.append({
        "case_id": record["case_id"],
        "run_id": record.get("run_id"),
        "model": record.get("model"),
        "status": status,
        "task_passed": passed,
        "source_file": item["file"],
    })

case_summaries.sort(key=lambda row: row["case_id"])
counts = Counter(row["status"] for row in case_summaries)

summary = {
    "schema_version": "1.0",
    "sdk": "deepagents",
    "scope": "first_pass_benchmark",
    "saved_at_utc": datetime.now(timezone.utc).isoformat(),
    "saved_run_count": len(case_summaries),
    "status_counts": dict(counts),
    "smoke_test_included": any(
        row["case_id"].startswith("smoke")
        for row in case_summaries
    ),
    "cases": case_summaries,
    "review_notes": {
        "math_002": {
            "finding": "wrong_operand_selection",
            "detail": (
                "Alpha net margin used previous-year revenue "
                "1000 instead of current-year revenue 1250."
            ),
        },
        "web_native_002": {
            "numeric_content_verified": True,
            "citation_validation": "partial",
            "review_status": "blocked_by_source_access",
            "detail": (
                "Figures independently verified against official sources. "
                "The exact Alphabet PDF cited by the agent returned "
                "403 to the review tool. Overall task_passed remains null."
            ),
            "review_sources": [
                "https://www.microsoft.com/investor/reports/ar24/",
                (
                    "https://www.sec.gov/Archives/edgar/data/"
                    "1652044/000165204425000010/googexhibit991q42024.htm"
                ),
            ],
        },
    },
    "limitations": [
        "Results describe the tested model, SDK configuration and fixtures.",
        "One run per case; reliability and latency distributions not measured.",
        "MCP tested using in-memory transport.",
        "Web search used OpenAI hosted search through Deep Agents.",
        "Planning was explicitly requested in the prompts.",
        "History persistence tested across a Jupyter kernel restart.",
        "Error cases used structured tool-result errors.",
        "No winner across SDKs can be selected before testing the other SDKs.",
    ],
}

shared_dir = Path("/workspace/shared")
assert shared_dir.is_dir()

summary_dir = shared_dir / "summaries"
summary_dir.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now(timezone.utc)
summary_path = summary_dir / (
    f"deepagents_{timestamp:%Y%m%dT%H%M%SZ}_{uuid4().hex[:8]}.json"
)

serialized = json.dumps(
    summary, indent=2, ensure_ascii=False, allow_nan=False
)

with summary_path.open("x", encoding="utf-8") as file:
    file.write(serialized)

assert json.loads(
    summary_path.read_text(encoding="utf-8")
) == summary

print("Saved:", summary_path)
print("Saved runs:", summary["saved_run_count"])
print("Status counts:", summary["status_counts"])
print("Deep Agents first-pass summary saved.")

Saved: /workspace/shared/summaries/deepagents_20261006T094802Z_64591d45.json
Saved runs: 17
Status counts: {'passed': 12, 'pending_review': 3, 'failed': 2}
Deep Agents first-pass summary saved.
